# BoneAge_LOCAL4050_V1 — Clean Local/Kaggle YOLO11s + Anatomy-QC Foundation

**Research / Clinical Decision-Support only — not a definitive clinical diagnosis.**

This notebook is a clean rebuild. It is **not** a patch of V1edit/V1edit2.

## Locked design decisions

- Primary hardware: **NVIDIA RTX 4050 Laptop GPU (6 GB)**.
- Secondary runtime: **Kaggle GPU**.
- `YOLO11s` is retained for paper continuity.
- YOLO training starts only from generic `yolo11s.pt`.
- Candidate A/B/C are archival failures and are **not implemented**.
- Legacy Val260/Test260 are historically exposed and are quarantined.
- A new V1 Validation260 and V1 FinalTest260 are deterministically created **only from the old Train12068 pool**.
- The new FinalTest260 is inaccessible to development cells until explicitly unlocked after the model/rules are frozen.
- Detector training keeps only **traceable exact-9/9** labeled images and excludes all protected bone-age IDs.
- Detector confidence is a proposal signal, not anatomical truth.
- Automatic proposal confidence floor is 0.10; no `<0.10` rescue exists.
- Final ROI crops always come from the original radiograph.
- No center-crop fallback.
- No legacy ROI patch is accepted as final anatomy.
- L/R/text/marker contamination is treated as an anatomical-QC risk.
- Windows DataLoader/training workers default to `0` to avoid Jupyter multiprocessing cleanup errors seen in the legacy run.
- Training checkpoints and artifacts are hashed and resumable.

## Scope of this notebook

This V1 notebook establishes the **clean dataset/test policy + strict YOLO11s detector + ROI proposal/QC/human-review foundation**.
Bone-age regression experiments should start only after the verified ROI gate is complete.

The notebook deliberately stops rather than fabricating data when a critical prerequisite is unavailable.

In [1]:
# Cell 1 — Dependency bootstrap (does NOT replace your CUDA-enabled PyTorch build)

import sys, subprocess, importlib.util

AUTO_INSTALL = True

# Torch is intentionally NOT auto-installed because replacing an existing
# CUDA-enabled PyTorch build is a common source of local-GPU breakage.
required = {
    "ultralytics": "ultralytics==8.4.118",
    "yaml": "pyyaml>=6.0",
    "cv2": "opencv-python>=4.8",
    "PIL": "pillow>=10.0",
    "matplotlib": "matplotlib>=3.7",
    "pandas": "pandas>=2.0",
    "sklearn": "scikit-learn>=1.3",
    "tqdm": "tqdm>=4.66",
}

missing = [pip_name for mod, pip_name in required.items()
           if importlib.util.find_spec(mod) is None]

if missing:
    print("Missing:", missing)
    if not AUTO_INSTALL:
        raise RuntimeError(
            "Dependencies are missing. Set AUTO_INSTALL=True or install them manually."
        )
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )

if importlib.util.find_spec("torch") is None:
    raise RuntimeError(
        "PyTorch is missing. Install the CUDA build appropriate for your NVIDIA driver "
        "before running this notebook. PyTorch is intentionally not auto-installed."
    )

print("Dependency bootstrap PASS")


Dependency bootstrap PASS


In [2]:
# Cell 2 — Imports + environment detector

from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict
from typing import Any, Iterable, Optional
import os, sys, re, json, math, time, random, hashlib, shutil, platform, subprocess
import warnings

import numpy as np
import pandas as pd
import yaml
import cv2
from PIL import Image, ImageOps, ImageDraw, ImageFont
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
from ultralytics import YOLO
import ultralytics

IS_WINDOWS = os.name == "nt"
IS_KAGGLE = Path("/kaggle/working").is_dir()
IS_COLAB = "COLAB_RELEASE_TAG" in os.environ
ENVIRONMENT = (
    "windows_local" if IS_WINDOWS
    else "kaggle" if IS_KAGGLE
    else "colab" if IS_COLAB
    else "linux"
)

if IS_WINDOWS:
    default_source_root = Path(r"D:\workspace\Bone_age")
    default_project_root = default_source_root / "BoneAge_LOCAL4050_V1"
elif IS_KAGGLE:
    default_source_root = Path("/kaggle/input")
    default_project_root = Path("/kaggle/working/BoneAge_LOCAL4050_V1")
else:
    default_source_root = Path(os.environ.get("BONEAGE_SOURCE_ROOT", "/workspace/Bone_age"))
    default_project_root = Path(os.environ.get(
        "BONEAGE_PROJECT_ROOT",
        str(default_source_root / "BoneAge_LOCAL4050_V1")
    ))

print("Environment :", ENVIRONMENT)
print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Ultralytics :", ultralytics.__version__)
print("CUDA        :", torch.cuda.is_available())

if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    print("GPU         :", torch.cuda.get_device_name(0))
    print("VRAM GB     :", round(prop.total_memory / 1024**3, 2))


Environment : windows_local
Python      : 3.11.8
PyTorch     : 2.11.0+cu128
Ultralytics : 8.4.118
CUDA        : True
GPU         : NVIDIA GeForce RTX 4050 Laptop GPU
VRAM GB     : 6.0


In [3]:

# Cell 3 — Central configuration (single source of truth)
#
# ROOT-CAUSE R2 changes:
# - auto_accept_conf raised from 0.20 -> 0.60 as a conservative provisional gate.
#   This is an engineering safety threshold, NOT a clinically calibrated probability.
# - 1280 rescue enabled for small-hand / large-background radiographs.
# - hand-mask overlap is no longer allowed to auto-REJECT anatomy.
# - pilot freeze gates added so a Boolean approval alone cannot override gross detector failure.

@dataclass(frozen=True)
class ProjectConfig:
    version: str = "BoneAge_LOCAL4050_V1_ROOTCAUSE_R3_20260817"
    seed: int = 42

    # Source / project roots can also be overridden by environment variables.
    source_root: str = str(default_source_root)
    project_root: str = str(default_project_root)

    # Known source filenames.
    source_csv_name: str = "boneage_training_years_upto20.csv"
    legacy_manifest_name: str = "patient_roi_manifest.csv"

    # Reviewer-safe test policy.
    legacy_val_expected: int = 260
    legacy_test_expected: int = 260
    v1_validation_n: int = 260
    v1_final_test_n: int = 260

    # Nine ROI classes.
    roi_order: tuple = (
        "Carpal", "DP3", "MC1", "MP3", "MP5",
        "PP1", "PP3", "Radius", "Ulna"
    )

    # Detector dataset controls.
    detector_dataset_name_hint: str = "SMI_Fishman.v4i.yolov11"
    detector_min_train_images: int = 1000
    detector_min_val_images: int = 100
    detector_min_test_images: int = 50

    # YOLO11s baseline.
    detector_model: str = "yolo11s.pt"
    detector_imgsz: int = 640
    detector_epochs: int = 100
    detector_patience: int = 20
    detector_nbs: int = 64

    detector_optimizer: str = "AdamW"
    detector_lr0: float = 7.5e-4
    detector_lrf: float = 0.01
    detector_momentum: float = 0.90
    detector_weight_decay: float = 5e-4
    detector_warmup_epochs: float = 3.0
    detector_save_period: int = 5

    # Conservative training augmentation.
    # Do NOT increase rotation here merely to hide deployment-domain problems.
    hsv_h: float = 0.0
    hsv_s: float = 0.0
    hsv_v: float = 0.08
    degrees: float = 3.0
    translate: float = 0.03
    scale: float = 0.10
    fliplr: float = 0.0
    flipud: float = 0.0
    mosaic: float = 0.0
    mixup: float = 0.0
    shear: float = 0.0
    perspective: float = 0.0

    detector_min_val_map50: float = 0.95
    detector_min_val_map5095: float = 0.70

    # Development must tune against detector VAL, not repeatedly inspect detector TEST.
    # Historical test metrics remain in the audit trail, but routine reruns keep it locked.
    detector_run_test_during_development: bool = False

    # ROI proposal/QC policy.
    auto_conf_floor: float = 0.10

    # Rescue passes may look below the normal proposal floor so an immature/small
    # ROI is not reported as MISSING merely because confidence is 0.05–0.10.
    # Low-confidence rescue proposals are still NEEDS_REVIEW, never auto-accepted.
    rescue_conf_floor: float = 0.05

    # IMPORTANT:
    # 0.20 previously allowed 100/1031 pilot ACCEPTs below confidence 0.60,
    # including 49 below 0.40. Until human calibration exists, require >=0.60
    # for automatic acceptance. Lower-confidence detections are preserved as
    # NEEDS_REVIEW rather than discarded.
    auto_accept_conf: float = 0.60

    primary_inference_imgsz: int = 640
    rescue_inference_imgsz: int = 960
    enable_1280_rescue: bool = True

    # Deployment-domain rescue angles.
    # Rotation is inference-only and boxes are mapped back to the original image.
    # No horizontal/vertical mirroring is used.
    enable_rotation_rescue: bool = True
    rotation_rescue_degrees: tuple = (-90, -45, 45, 90, 180)

    # Candidate audit/consensus metadata. Consensus is evidence for review only;
    # it does not magically turn a low-confidence detection into an ACCEPT.
    candidate_consensus_iou: float = 0.30

    # Geometry / foreground heuristics are advisory only.
    accept_geometry_z_max: float = 3.5
    reject_geometry_z_min: float = 6.0  # retained for reporting; no anatomy auto-reject
    accept_hand_overlap_min: float = 0.25
    reject_hand_overlap_below: float = 0.10  # retained for reporting; no anatomy auto-reject

    # Pilot visual-QC controls.
    roi_pilot_per_age: int = 8
    roi_pilot_accept_audit_per_roi: int = 8

    # Full DEV/VAL accepted-ROI audit.
    devval_accept_audit_per_stratum: int = 1

    # Structural pilot gates.
    # IMPORTANT: a single age-bin rate is estimated from only 8 pilot patients
    # (72 ROI slots), so R3 uses WARNING vs HARD-BLOCK tiers instead of treating
    # 30.00% and 30.56% as meaningfully different.
    pilot_max_missing_rate: float = 0.10
    pilot_max_any_roi_missing_rate: float = 0.25
    pilot_agebin_warning_missing_rate: float = 0.20
    pilot_agebin_hard_missing_rate: float = 0.40
    pilot_nonaccept_warning_rate: float = 0.15

    # Geometry priors should have meaningful support even though they are advisory.
    geometry_prior_min_samples: int = 100

    # Human approval remains false until CURRENT-HASH complete sheets are reviewed.
    roi_rule_manual_approval: bool = False

    # If R3 reports known quality warnings (e.g. young-age domain gap / weak MC1),
    # this separate acknowledgement prevents a single Boolean from hiding them.
    roi_rule_accept_known_warnings: bool = False

CFG = ProjectConfig(
    source_root=os.environ.get("BONEAGE_SOURCE_ROOT", str(default_source_root)),
    project_root=os.environ.get("BONEAGE_PROJECT_ROOT", str(default_project_root)),
)

SOURCE_ROOT = Path(CFG.source_root)
PROJECT_ROOT = Path(CFG.project_root)

print(json.dumps(asdict(CFG), indent=2, ensure_ascii=False, default=str))


{
  "version": "BoneAge_LOCAL4050_V1_ROOTCAUSE_R3_20260817",
  "seed": 42,
  "source_root": "D:\\workspace\\Bone_age",
  "project_root": "D:\\workspace\\Bone_age\\BoneAge_LOCAL4050_V1",
  "source_csv_name": "boneage_training_years_upto20.csv",
  "legacy_manifest_name": "patient_roi_manifest.csv",
  "legacy_val_expected": 260,
  "legacy_test_expected": 260,
  "v1_validation_n": 260,
  "v1_final_test_n": 260,
  "roi_order": [
    "Carpal",
    "DP3",
    "MC1",
    "MP3",
    "MP5",
    "PP1",
    "PP3",
    "Radius",
    "Ulna"
  ],
  "detector_dataset_name_hint": "SMI_Fishman.v4i.yolov11",
  "detector_min_train_images": 1000,
  "detector_min_val_images": 100,
  "detector_min_test_images": 50,
  "detector_model": "yolo11s.pt",
  "detector_imgsz": 640,
  "detector_epochs": 100,
  "detector_patience": 20,
  "detector_nbs": 64,
  "detector_optimizer": "AdamW",
  "detector_lr0": 0.00075,
  "detector_lrf": 0.01,
  "detector_momentum": 0.9,
  "detector_weight_decay": 0.0005,
  "detector_warmu

In [4]:
# Cell 4 — Directory manager + artifact registry

DIR = {
    "root": PROJECT_ROOT,
    "configs": PROJECT_ROOT / "configs",
    "registry": PROJECT_ROOT / "data" / "registry",
    "splits": PROJECT_ROOT / "data" / "splits",
    "detector_data": PROJECT_ROOT / "data" / "detector_strict",
    "qc": PROJECT_ROOT / "qc",
    "qc_reports": PROJECT_ROOT / "qc" / "reports",
    "qc_review": PROJECT_ROOT / "qc" / "needs_review",
    "roi": PROJECT_ROOT / "roi",
    "manual": PROJECT_ROOT / "roi" / "manual_overrides",
    "models": PROJECT_ROOT / "checkpoints",
    "detector_models": PROJECT_ROOT / "checkpoints" / "detector_yolo11s",
    "experiments": PROJECT_ROOT / "experiments",
    "logs": PROJECT_ROOT / "logs",
    "outputs": PROJECT_ROOT / "outputs",
}

for p in DIR.values():
    p.mkdir(parents=True, exist_ok=True)

CONFIG_FILE = DIR["configs"] / "project_config.yaml"
CONFIG_FILE.write_text(
    yaml.safe_dump(asdict(CFG), sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

ARTIFACT_REGISTRY = DIR["registry"] / "artifacts.jsonl"

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def append_jsonl(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

def register_artifact(stage: str, path: Path, role: str, metadata: Optional[dict] = None) -> None:
    p = Path(path)
    record = {
        "time": pd.Timestamp.now(tz="UTC").isoformat(),
        "project_version": CFG.version,
        "stage": stage,
        "role": role,
        "path": str(p.resolve()) if p.exists() else str(p),
        "exists": p.exists(),
        "sha256": sha256_file(p) if p.is_file() else None,
        "metadata": metadata or {},
    }
    append_jsonl(ARTIFACT_REGISTRY, record)

register_artifact("STAGE_0", CONFIG_FILE, "central_config")
print("Project root:", PROJECT_ROOT)
print("Config      :", CONFIG_FILE)
print("Artifact log:", ARTIFACT_REGISTRY)


Project root: D:\workspace\Bone_age\BoneAge_LOCAL4050_V1
Config      : D:\workspace\Bone_age\BoneAge_LOCAL4050_V1\configs\project_config.yaml
Artifact log: D:\workspace\Bone_age\BoneAge_LOCAL4050_V1\data\registry\artifacts.jsonl


In [5]:
# Cell 5 — Reproducibility + hardware metadata + stage gates

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)

# Deterministic mode can reduce speed; it is retained for auditability.
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e:
    warnings.warn(f"Could not enable deterministic algorithms: {e}")

torch.backends.cudnn.benchmark = False

def gpu_vram_gb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.get_device_properties(0).total_memory / 1024**3

def recommended_detector_batch() -> int:
    override = os.environ.get("BONEAGE_DETECTOR_BATCH", "").strip()

    if override:
        b = int(override)
        if b <= 0:
            raise ValueError(
                "BONEAGE_DETECTOR_BATCH must be a positive integer."
            )
        return b

    vram = gpu_vram_gb()

    if vram <= 0:
        return 1

    if vram <= 7.5:      # RTX 4050 Laptop 6 GB
        return 8

    if vram <= 12.5:     # 8–12 GB
        return 16

    if vram <= 24.5:     # RTX 4090 / similar
        return 32

    return 32


DETECTOR_BATCH = recommended_detector_batch()
DETECTOR_WORKERS = 0 if IS_WINDOWS else 2

DETECTOR_NBS = CFG.detector_nbs

# Target accumulation after warmup.
DETECTOR_TARGET_ACCUMULATE = max(
    round(DETECTOR_NBS / DETECTOR_BATCH),
    1
)

DETECTOR_EFFECTIVE_BATCH = (
    DETECTOR_BATCH * DETECTOR_TARGET_ACCUMULATE
)



ENV_META = {
    "project_version": CFG.version,
    "environment": ENVIRONMENT,
    "platform": platform.platform(),
    "python": sys.version,
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "vram_gb": gpu_vram_gb(),

    "detector_batch": DETECTOR_BATCH,
    "detector_nbs": DETECTOR_NBS,
    "detector_target_accumulate": DETECTOR_TARGET_ACCUMULATE,
    "detector_effective_batch": DETECTOR_EFFECTIVE_BATCH,

    "detector_workers": DETECTOR_WORKERS,
}
ENV_FILE = DIR["qc_reports"] / "00_environment.json"
ENV_FILE.write_text(json.dumps(ENV_META, indent=2, ensure_ascii=False), encoding="utf-8")
register_artifact("STAGE_0", ENV_FILE, "environment_metadata")

STAGE_STATE = {}

def stage_result(stage: str, status: str, message: str, details: Optional[dict] = None) -> None:
    # BLOCKED = expected quality/procedure hold. It stops downstream work by gate
    # checks but does not masquerade as a Python/runtime failure.
    allowed = {"PASS", "WARNING", "FAIL", "NOT_VERIFIED", "BLOCKED"}
    if status not in allowed:
        raise ValueError(status)
    payload = {
        "project_version": CFG.version,
        "stage": stage,
        "status": status,
        "message": message,
        "details": details or {},
        "time": pd.Timestamp.now(tz="UTC").isoformat(),
    }
    STAGE_STATE[stage] = payload
    p = DIR["qc_reports"] / f"{stage}.json"
    p.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    print(f"[{stage}] {status}: {message}")
    if status == "FAIL":
        raise RuntimeError(f"{stage} FAIL — STOP PIPELINE: {message}")


def get_stage_state(stage: str) -> Optional[dict]:
    """Read current-runtime state, then a persisted state from the SAME project version."""
    if stage in STAGE_STATE:
        return STAGE_STATE[stage]

    p = DIR["qc_reports"] / f"{stage}.json"
    if not p.is_file():
        return None

    try:
        payload = json.loads(p.read_text(encoding="utf-8"))
    except Exception as e:
        warnings.warn(f"Could not read persisted stage {stage}: {type(e).__name__}: {e}")
        return None

    if payload.get("project_version") != CFG.version:
        return None

    return payload


stage_result(
    "STAGE_0_ENVIRONMENT",
    "PASS" if torch.cuda.is_available() else "WARNING",
    "Environment captured; GPU required only when detector training begins.",
    ENV_META,
)


C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\triton\windows_utils.py:405: UserWarning: Failed to find CUDA.
  warnings.warn("Failed to find CUDA.")


[STAGE_0_ENVIRONMENT] PASS: Environment captured; GPU required only when detector training begins.


## Test policy lock

The historical `Val260/Test260` were viewed in earlier development and therefore are **not treated as pristine V1 hold-outs**.

V1 policy:

1. Recover the historical split from the legacy canonical manifest.
2. Quarantine all legacy Val260/Test260 IDs.
3. Use only the historical Train12068 pool to create:
   - `V1_DEV`
   - `V1_VALIDATION` = 260
   - `V1_FINAL_TEST` = 260
4. Allocation is deterministic, bone-age-year-bin stratified, and sex-aware within each bin.
5. Once the split lock exists, this notebook refuses to silently regenerate it.
6. The V1 final test cannot be read by development ROI cells unless explicitly unlocked later.

In [6]:
# Cell 6 — Source discovery (CSV, legacy split manifest, full-hand roots)

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def find_named_files(roots: Iterable[Path], filename: str, max_hits: int = 20) -> list[Path]:
    hits = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue

        direct = root / filename
        if direct.is_file():
            hits.append(direct.resolve())

        # Targeted recursive search for a small number of metadata files.
        try:
            for p in root.rglob(filename):
                if p.is_file():
                    rp = p.resolve()
                    if rp not in hits:
                        hits.append(rp)
                    if len(hits) >= max_hits:
                        return hits
        except PermissionError:
            continue
    return hits

search_roots = [SOURCE_ROOT]
if IS_KAGGLE:
    search_roots += [Path("/kaggle/input")]
if IS_WINDOWS:
    search_roots += [Path(r"D:\workspace\Bone_age"), Path(r"D:\Bone_age")]

source_hits = find_named_files(search_roots, CFG.source_csv_name)
if not source_hits:
    stage_result(
        "STAGE_1_SOURCE_DISCOVERY",
        "FAIL",
        f"{CFG.source_csv_name} not found. Set BONEAGE_SOURCE_ROOT to the folder containing the project data."
    )

SOURCE_CSV = source_hits[0]

legacy_candidates = []
for name in [CFG.legacy_manifest_name, "patient_roi_manifest_1213.csv"]:
    legacy_candidates.extend(find_named_files(search_roots, name))

# Prefer the pipeline_output canonical manifest.
legacy_candidates = list(dict.fromkeys(legacy_candidates))
legacy_candidates.sort(
    key=lambda p: (
        0 if p.name == "patient_roi_manifest.csv" else 1,
        0 if "pipeline_output" in str(p).lower() else 1,
        len(str(p))
    )
)

if not legacy_candidates:
    stage_result(
        "STAGE_1_SOURCE_DISCOVERY",
        "FAIL",
        "Legacy split manifest not found. It is required to quarantine historically exposed Val260/Test260 IDs."
    )

LEGACY_MANIFEST = legacy_candidates[0]

stage_result(
    "STAGE_1_SOURCE_DISCOVERY",
    "PASS",
    "Source CSV and legacy split manifest found.",
    {"source_csv": str(SOURCE_CSV), "legacy_manifest": str(LEGACY_MANIFEST)}
)
print("SOURCE_CSV     :", SOURCE_CSV)
print("LEGACY_MANIFEST:", LEGACY_MANIFEST)


[STAGE_1_SOURCE_DISCOVERY] PASS: Source CSV and legacy split manifest found.
SOURCE_CSV     : D:\workspace\Bone_age\boneage_training_years_upto20.csv
LEGACY_MANIFEST: D:\Bone_age\pipeline_output\patient_roi_manifest.csv


In [7]:
# Cell 7 — Canonical 12,588 registry + full-hand discovery + correct label semantics

src = pd.read_csv(SOURCE_CSV)
legacy = pd.read_csv(LEGACY_MANIFEST, dtype={"patient_id": str})

# Resolve source columns.
id_col = "patient_id" if "patient_id" in src.columns else "id" if "id" in src.columns else None
if id_col is None:
    stage_result("STAGE_2_CANONICAL", "FAIL", "Source CSV has no patient ID column.")

bone_col = "boneage" if "boneage" in src.columns else "bone_age_months" if "bone_age_months" in src.columns else None
if bone_col is None:
    stage_result("STAGE_2_CANONICAL", "FAIL", "Source CSV has no bone-age-month label column.")

sex_col = "male" if "male" in src.columns else None
if sex_col is None:
    stage_result("STAGE_2_CANONICAL", "FAIL", "Source CSV has no binary 'male' column.")

canonical = src.copy()
canonical["patient_id"] = canonical[id_col].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
canonical["bone_age_months"] = pd.to_numeric(canonical[bone_col], errors="coerce")
canonical["male"] = pd.to_numeric(canonical[sex_col], errors="coerce")

if canonical["bone_age_months"].isna().any():
    stage_result("STAGE_2_CANONICAL", "FAIL", "Invalid bone-age labels found.")
if not set(canonical["male"].dropna().astype(int).unique()).issubset({0, 1}):
    stage_result("STAGE_2_CANONICAL", "FAIL", "Sex encoding must be binary male=0/1.")

canonical["male"] = canonical["male"].astype(int)
canonical["bone_age_year_bin"] = np.rint(canonical["bone_age_months"] / 12.0).astype(int)

# Legacy policy filtered to year bins 1..18; preserve this cohort definition,
# but name the variable correctly: this is a bone-age bin, NOT chronological age.
canonical = canonical[
    canonical["bone_age_year_bin"].between(1, 18)
].copy().reset_index(drop=True)

if len(canonical) != 12588:
    stage_result(
        "STAGE_2_CANONICAL",
        "FAIL",
        f"Expected 12,588 cases after the legacy 1..18 bone-age-bin filter; found {len(canonical)}."
    )

if canonical["patient_id"].duplicated().any():
    stage_result("STAGE_2_CANONICAL", "FAIL", "Duplicate patient_id in canonical source.")

# Discover original full-hand images. Legacy ROI files are deliberately ignored.
legacy_path_map = {}
if "fullhand_image_path" in legacy.columns:
    for _, r in legacy.iterrows():
        pid = str(r["patient_id"])
        p = Path(str(r["fullhand_image_path"]))
        if p.is_file():
            legacy_path_map[pid] = p.resolve()

candidate_image_roots = []
for root in [SOURCE_ROOT, SOURCE_CSV.parent, SOURCE_ROOT / "pipeline_output", SOURCE_CSV.parent / "pipeline_output"]:
    if Path(root).exists():
        candidate_image_roots.append(Path(root))

# Build filename index only for likely male/female folders; avoids a full project scan where possible.
image_index = {}
for base in candidate_image_roots:
    for sex_dir in ["male", "female"]:
        d = base / sex_dir
        if d.is_dir():
            for p in d.iterdir():
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                    image_index[p.stem] = p.resolve()

def resolve_fullhand(pid: str, male: int) -> str:
    if pid in legacy_path_map:
        return str(legacy_path_map[pid])
    if pid in image_index:
        return str(image_index[pid])

    sex_dir = "male" if int(male) == 1 else "female"
    for base in candidate_image_roots:
        d = base / sex_dir
        if not d.is_dir():
            continue
        for ext in [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]:
            p = d / f"{pid}{ext}"
            if p.is_file():
                return str(p.resolve())
    return ""

canonical["fullhand_image_path"] = [
    resolve_fullhand(pid, male)
    for pid, male in zip(canonical["patient_id"], canonical["male"])
]
canonical["fullhand_found"] = canonical["fullhand_image_path"].map(lambda x: bool(x) and Path(x).is_file())

if not canonical["fullhand_found"].all():
    bad = canonical.loc[~canonical["fullhand_found"], ["patient_id", "male"]]
    bad.to_csv(DIR["qc_reports"] / "02_missing_fullhand.csv", index=False)
    stage_result(
        "STAGE_2_CANONICAL",
        "FAIL",
        f"{len(bad)} original full-hand images are missing. No ROI fallback is permitted."
    )

CANONICAL_REGISTRY = DIR["registry"] / "canonical_12588.csv"
canonical.to_csv(CANONICAL_REGISTRY, index=False, encoding="utf-8-sig")
register_artifact("STAGE_2", CANONICAL_REGISTRY, "canonical_registry")

stage_result(
    "STAGE_2_CANONICAL",
    "PASS",
    "Canonical 12,588 registry built; bone_age_year_bin semantics corrected; all original X-rays resolved.",
    {
        "n": len(canonical),
        "male": int(canonical["male"].sum()),
        "female": int((1-canonical["male"]).sum()),
        "bone_age_months_min": float(canonical["bone_age_months"].min()),
        "bone_age_months_max": float(canonical["bone_age_months"].max()),
    }
)


[STAGE_2_CANONICAL] PASS: Canonical 12,588 registry built; bone_age_year_bin semantics corrected; all original X-rays resolved.


In [8]:
# Cell 8 — FIXED VERIFIED split policy
# Self-healing: re-discovers and verifies the correct legacy 12588 split
# instead of trusting a stale/wrong `legacy` object from an earlier cell.

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 8 — VERIFIED LEGACY SPLIT + NEW V1 HOLDOUT")
print("=" * 100)

# ---------------------------------------------------------------------
# A. Find candidate legacy manifests
# ---------------------------------------------------------------------

search_roots = []

for p in [
    SOURCE_ROOT if "SOURCE_ROOT" in globals() else None,
    SOURCE_CSV.parent if "SOURCE_CSV" in globals() else None,
    Path(r"D:\workspace\Bone_age"),
    Path(r"D:\workspace\Bone_age\pipeline_output"),
    Path(r"D:\Bone_age"),
    Path(r"D:\Bone_age\pipeline_output"),
]:
    if p is not None:
        p = Path(p)
        if p.exists() and p not in search_roots:
            search_roots.append(p)

candidate_paths = []

# Existing variable first, but it still has to pass validation.
if "LEGACY_MANIFEST" in globals():
    p = Path(LEGACY_MANIFEST)
    if p.is_file():
        candidate_paths.append(p.resolve())

for root in search_roots:
    try:
        for p in root.rglob("patient_roi_manifest.csv"):
            if p.is_file():
                rp = p.resolve()
                if rp not in candidate_paths:
                    candidate_paths.append(rp)
    except (PermissionError, OSError) as e:
        print(f"[WARN] Search skipped: {root} -> {e}")


# ---------------------------------------------------------------------
# B. Inspect CONTENT, not filename
# ---------------------------------------------------------------------

def inspect_manifest(path: Path) -> dict:
    result = {
        "path": str(path),
        "valid": False,
        "rows": None,
        "unique_ids": None,
        "train": None,
        "val": None,
        "test": None,
        "reason": "",
    }

    try:
        df = pd.read_csv(
            path,
            dtype={"patient_id": str},
            low_memory=False
        )
    except Exception as e:
        result["reason"] = f"READ_FAIL: {e!r}"
        return result

    result["rows"] = len(df)

    if "patient_id" not in df.columns:
        result["reason"] = "NO patient_id COLUMN"
        return result

    if "split" not in df.columns:
        result["reason"] = "NO split COLUMN"
        return result

    df["patient_id"] = (
        df["patient_id"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

    df["split"] = (
        df["split"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    counts = df["split"].value_counts().to_dict()

    result["unique_ids"] = df["patient_id"].nunique()
    result["train"] = int(counts.get("train", 0))
    result["val"] = int(counts.get("val", 0))
    result["test"] = int(counts.get("test", 0))

    valid = (
        len(df) == 12588
        and df["patient_id"].nunique() == 12588
        and result["train"] == 12068
        and result["val"] == 260
        and result["test"] == 260
    )

    result["valid"] = bool(valid)

    if valid:
        result["reason"] = "VERIFIED 12588/12068/260/260"
    else:
        result["reason"] = "COUNT/SPLIT MISMATCH"

    return result


audit_rows = [
    inspect_manifest(p)
    for p in candidate_paths
]

audit_df = pd.DataFrame(audit_rows)

print("\nManifest candidates:")
if len(audit_df):
    display(
        audit_df[
            [
                "path",
                "rows",
                "unique_ids",
                "train",
                "val",
                "test",
                "valid",
                "reason",
            ]
        ]
    )
else:
    print("No patient_roi_manifest.csv found.")


# ---------------------------------------------------------------------
# C. Select ONLY a verified manifest
# ---------------------------------------------------------------------

valid_rows = audit_df[
    audit_df["valid"].eq(True)
].copy() if len(audit_df) else pd.DataFrame()

if len(valid_rows) == 0:
    report_path = (
        DIR["qc_reports"]
        / "08_legacy_manifest_verification_FAILED.csv"
    )

    audit_df.to_csv(
        report_path,
        index=False,
        encoding="utf-8-sig"
    )

    raise RuntimeError(
        "\nFAIL — STOP PIPELINE\n"
        "ไม่พบ patient_roi_manifest.csv ที่มีโครงสร้าง\n"
        "12588 total / 12068 train / 260 val / 260 test.\n\n"
        "ไฟล์ 1213 rows ไม่ใช่ canonical legacy split และจะไม่ถูกใช้.\n"
        f"Audit report: {report_path}"
    )


# Prefer D:\workspace\Bone_age over historical D:\Bone_age.
valid_rows["_priority"] = valid_rows["path"].map(
    lambda x:
        0 if str(x).lower().startswith(
            str(Path(r"D:\workspace\Bone_age")).lower()
        )
        else 1
)

valid_rows["_length"] = valid_rows["path"].str.len()

valid_rows = valid_rows.sort_values(
    ["_priority", "_length"]
)

LEGACY_MANIFEST = Path(
    valid_rows.iloc[0]["path"]
)

print("\n✅ VERIFIED LEGACY_MANIFEST:")
print(LEGACY_MANIFEST)


# ---------------------------------------------------------------------
# D. Reload `legacy` from VERIFIED manifest
# ---------------------------------------------------------------------

legacy = pd.read_csv(
    LEGACY_MANIFEST,
    dtype={"patient_id": str},
    low_memory=False
)

legacy["patient_id"] = (
    legacy["patient_id"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

legacy["split"] = (
    legacy["split"]
    .astype(str)
    .str.lower()
    .str.strip()
)

LEGACY_VAL_IDS = set(
    legacy.loc[
        legacy["split"].eq("val"),
        "patient_id"
    ]
)

LEGACY_TEST_IDS = set(
    legacy.loc[
        legacy["split"].eq("test"),
        "patient_id"
    ]
)

LEGACY_TRAIN_IDS = set(
    legacy.loc[
        legacy["split"].eq("train"),
        "patient_id"
    ]
)

print("\nVerified counts:")
print("Train:", len(LEGACY_TRAIN_IDS))
print("Val  :", len(LEGACY_VAL_IDS))
print("Test :", len(LEGACY_TEST_IDS))

assert len(legacy) == 12588, (
    f"Expected 12588 rows, got {len(legacy)}"
)

assert legacy["patient_id"].nunique() == 12588, (
    "patient_id must be unique 12588"
)

assert len(LEGACY_TRAIN_IDS) == 12068, (
    f"Expected Train12068, got {len(LEGACY_TRAIN_IDS)}"
)

assert len(LEGACY_VAL_IDS) == 260, (
    f"Expected Val260, got {len(LEGACY_VAL_IDS)}"
)

assert len(LEGACY_TEST_IDS) == 260, (
    f"Expected Test260, got {len(LEGACY_TEST_IDS)}"
)

assert not (
    LEGACY_TRAIN_IDS & LEGACY_VAL_IDS
), "Train/Val overlap"

assert not (
    LEGACY_TRAIN_IDS & LEGACY_TEST_IDS
), "Train/Test overlap"

assert not (
    LEGACY_VAL_IDS & LEGACY_TEST_IDS
), "Val/Test overlap"


# ---------------------------------------------------------------------
# E. Verify against canonical 12,588
# ---------------------------------------------------------------------

canonical_ids = set(
    canonical["patient_id"].astype(str)
)

legacy_all_ids = (
    LEGACY_TRAIN_IDS
    | LEGACY_VAL_IDS
    | LEGACY_TEST_IDS
)

missing_from_canonical = (
    legacy_all_ids - canonical_ids
)

if missing_from_canonical:
    raise RuntimeError(
        "Legacy manifest contains patient IDs "
        "not found in canonical registry. "
        f"Example: {list(sorted(missing_from_canonical))[:10]}"
    )

if len(legacy_all_ids) != 12588:
    raise RuntimeError(
        f"Legacy union expected 12588 unique IDs, "
        f"got {len(legacy_all_ids)}"
    )

base_dev = canonical[
    canonical["patient_id"].isin(
        LEGACY_TRAIN_IDS
    )
].copy()

assert len(base_dev) == 12068


# ---------------------------------------------------------------------
# F. Deterministic stratified selection
# ---------------------------------------------------------------------

def stable_rank(
    pid: str,
    salt: str
) -> str:

    return hashlib.sha256(
        f"{CFG.seed}|{salt}|{pid}".encode(
            "utf-8"
        )
    ).hexdigest()


def largest_remainder_targets(
    counts: pd.Series,
    total_target: int
) -> dict[int, int]:

    counts = counts.astype(int)

    if counts.sum() <= 0:
        raise RuntimeError(
            "Cannot allocate from empty counts."
        )

    raw = (
        counts
        / counts.sum()
        * total_target
    )

    out = np.floor(
        raw
    ).astype(int)

    remainder = (
        total_target
        - int(out.sum())
    )

    order = sorted(
        counts.index,
        key=lambda k: (
            -(raw.loc[k] - out.loc[k]),
            int(k)
        )
    )

    for k in order[:remainder]:
        out.loc[k] += 1

    result = {
        int(k): int(v)
        for k, v in out.items()
    }

    if sum(result.values()) != total_target:
        raise RuntimeError(
            "Largest-remainder allocation failed."
        )

    return result


def choose_stratified(
    pool: pd.DataFrame,
    total_n: int,
    salt: str
) -> pd.DataFrame:

    age_counts = (
        pool
        .groupby("bone_age_year_bin")
        .size()
        .reindex(
            range(1, 19),
            fill_value=0
        )
    )

    age_targets = (
        largest_remainder_targets(
            age_counts,
            total_n
        )
    )

    selected = []

    for age in range(1, 19):

        g = pool[
            pool["bone_age_year_bin"]
            .eq(age)
        ].copy()

        n = age_targets[age]

        if len(g) < n:
            raise RuntimeError(
                f"Age bin {age}: "
                f"{len(g)} available < {n} requested"
            )

        sex_counts = (
            g.groupby("male")
            .size()
            .reindex(
                [0, 1],
                fill_value=0
            )
        )

        sex_targets = (
            largest_remainder_targets(
                sex_counts,
                n
            )
        )

        chosen_ids = []

        for sex in [0, 1]:

            sg = g[
                g["male"].eq(sex)
            ].copy()

            sg["_rank"] = (
                sg["patient_id"]
                .map(
                    lambda pid:
                    stable_rank(
                        pid,
                        f"{salt}|A{age}|S{sex}"
                    )
                )
            )

            sg = sg.sort_values(
                "_rank"
            )

            take = min(
                int(
                    sex_targets[sex]
                ),
                len(sg)
            )

            chosen_ids.extend(
                sg.head(take)[
                    "patient_id"
                ].tolist()
            )

        # deterministic same-age fill
        if len(chosen_ids) < n:

            rem = g[
                ~g["patient_id"]
                .isin(chosen_ids)
            ].copy()

            rem["_rank"] = (
                rem["patient_id"]
                .map(
                    lambda pid:
                    stable_rank(
                        pid,
                        f"{salt}|A{age}|FILL"
                    )
                )
            )

            chosen_ids.extend(
                rem.sort_values(
                    "_rank"
                )
                .head(
                    n - len(chosen_ids)
                )["patient_id"]
                .tolist()
            )

        if len(chosen_ids) != n:
            raise RuntimeError(
                f"Age {age}: "
                f"{len(chosen_ids)} selected != {n}"
            )

        selected.extend(
            chosen_ids
        )

    out = pool[
        pool["patient_id"]
        .isin(selected)
    ].copy()

    if (
        len(out) != total_n
        or out["patient_id"]
        .nunique() != total_n
    ):
        raise RuntimeError(
            "Stratified selection cardinality mismatch."
        )

    return out


# ---------------------------------------------------------------------
# G. Split-lock files
# ---------------------------------------------------------------------

SPLIT_CSV = (
    DIR["splits"]
    / "v1_split_lock.csv"
)

SPLIT_JSON = (
    DIR["splits"]
    / "v1_split_lock.json"
)

base_hash = sha256_text(
    "\n".join(
        sorted(
            base_dev["patient_id"]
            .astype(str)
        )
    )
)


# ---------------------------------------------------------------------
# H. Reuse existing lock ONLY if valid
# ---------------------------------------------------------------------

EXPECTED_ROLE_COUNTS = {
    "V1_DEV": 11548,
    "V1_VALIDATION": 260,
    "V1_FINAL_TEST": 260,
    "LEGACY_VAL_QUARANTINE": 260,
    "LEGACY_TEST_QUARANTINE": 260,
}

def validate_split_lock_membership(candidate: pd.DataFrame) -> tuple[bool, list[str]]:
    """Validate counts AND exact cohort membership, not counts alone."""
    errors = []

    required = {"patient_id", "v1_role"}
    missing_cols = required - set(candidate.columns)
    if missing_cols:
        return False, [f"missing columns: {sorted(missing_cols)}"]

    t = candidate.copy()
    t["patient_id"] = (
        t["patient_id"].astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )
    t["v1_role"] = t["v1_role"].astype(str).str.strip()

    if len(t) != 12588:
        errors.append(f"row count={len(t)} != 12588")

    if t["patient_id"].duplicated().any():
        errors.append("duplicate patient_id rows")

    if set(t["patient_id"]) != canonical_ids:
        errors.append("patient ID union differs from canonical 12,588")

    counts_now = t["v1_role"].value_counts().to_dict()
    if counts_now != EXPECTED_ROLE_COUNTS:
        errors.append(
            f"role counts mismatch: got={counts_now}, expected={EXPECTED_ROLE_COUNTS}"
        )

    roles_now = {
        role: set(t.loc[t["v1_role"].eq(role), "patient_id"])
        for role in EXPECTED_ROLE_COUNTS
    }

    if roles_now["LEGACY_VAL_QUARANTINE"] != LEGACY_VAL_IDS:
        errors.append("LEGACY_VAL_QUARANTINE membership != verified legacy Val260")

    if roles_now["LEGACY_TEST_QUARANTINE"] != LEGACY_TEST_IDS:
        errors.append("LEGACY_TEST_QUARANTINE membership != verified legacy Test260")

    v1_train_family = (
        roles_now["V1_DEV"]
        | roles_now["V1_VALIDATION"]
        | roles_now["V1_FINAL_TEST"]
    )
    if v1_train_family != LEGACY_TRAIN_IDS:
        errors.append("V1 DEV/VAL/FinalTest union != verified legacy Train12068")

    role_names = list(roles_now)
    for i, a in enumerate(role_names):
        for b in role_names[i+1:]:
            overlap = roles_now[a] & roles_now[b]
            if overlap:
                errors.append(f"{a}/{b} overlap n={len(overlap)}")

    return len(errors) == 0, errors


reuse_lock = False

if (
    SPLIT_CSV.is_file()
    and SPLIT_JSON.is_file()
):

    lock_meta = json.loads(
        SPLIT_JSON.read_text(
            encoding="utf-8"
        )
    )

    old_hash = (
        lock_meta.get(
            "base_legacy_train_ids_sha256"
        )
    )

    if old_hash == base_hash:

        candidate_split = pd.read_csv(
            SPLIT_CSV,
            dtype={"patient_id": str}
        )

        candidate_counts = (
            candidate_split[
                "v1_role"
            ]
            .value_counts()
            .to_dict()
        )

        lock_ok, lock_errors = validate_split_lock_membership(candidate_split)

        if lock_ok:
            split_df = candidate_split
            reuse_lock = True
            print("[LOCK] Existing V1 split reused after exact membership validation.")
        else:
            print("[LOCK] Existing split rejected; it will NOT be reused.")
            for err in lock_errors:
                print("  -", err)

    else:

        print(
            "[LOCK] Existing split belongs "
            "to another base cohort; "
            "not reused."
        )


# ---------------------------------------------------------------------
# I. Create a new split if necessary
# ---------------------------------------------------------------------

if not reuse_lock:

    v1_test = choose_stratified(
        base_dev,
        CFG.v1_final_test_n,
        "V1_FINAL_TEST"
    )

    remaining = base_dev[
        ~base_dev["patient_id"]
        .isin(
            v1_test["patient_id"]
        )
    ].copy()

    v1_val = choose_stratified(
        remaining,
        CFG.v1_validation_n,
        "V1_VALIDATION"
    )

    v1_dev = remaining[
        ~remaining["patient_id"]
        .isin(
            v1_val["patient_id"]
        )
    ].copy()

    assert len(v1_test) == 260
    assert len(v1_val) == 260
    assert len(v1_dev) == 11548

    assert not (
        set(v1_test["patient_id"])
        & set(v1_val["patient_id"])
    )

    assert not (
        set(v1_test["patient_id"])
        & set(v1_dev["patient_id"])
    )

    assert not (
        set(v1_val["patient_id"])
        & set(v1_dev["patient_id"])
    )

    rows = []

    for role, frame in [
        ("V1_DEV", v1_dev),
        ("V1_VALIDATION", v1_val),
        ("V1_FINAL_TEST", v1_test),

        (
            "LEGACY_VAL_QUARANTINE",
            canonical[
                canonical["patient_id"]
                .isin(LEGACY_VAL_IDS)
            ]
        ),

        (
            "LEGACY_TEST_QUARANTINE",
            canonical[
                canonical["patient_id"]
                .isin(LEGACY_TEST_IDS)
            ]
        ),
    ]:

        t = frame[
            [
                "patient_id",
                "bone_age_year_bin",
                "male"
            ]
        ].copy()

        t["v1_role"] = role

        rows.append(t)

    split_df = pd.concat(
        rows,
        ignore_index=True
    )

    split_df.to_csv(
        SPLIT_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    lock_meta = {

        "policy":
        "NEW_V1_HOLDOUT_FROM_VERIFIED_LEGACY_TRAIN_ONLY",

        "seed":
        CFG.seed,

        "verified_legacy_manifest":
        str(LEGACY_MANIFEST),

        "verified_legacy_manifest_sha256":
        sha256_file(
            LEGACY_MANIFEST
        ),

        "base_legacy_train_ids_sha256":
        base_hash,

        "v1_dev_n":
        int(
            (
                split_df["v1_role"]
                == "V1_DEV"
            ).sum()
        ),

        "v1_validation_n":
        int(
            (
                split_df["v1_role"]
                == "V1_VALIDATION"
            ).sum()
        ),

        "v1_final_test_n":
        int(
            (
                split_df["v1_role"]
                == "V1_FINAL_TEST"
            ).sum()
        ),

        "legacy_val_quarantine_n":
        int(
            (
                split_df["v1_role"]
                == "LEGACY_VAL_QUARANTINE"
            ).sum()
        ),

        "legacy_test_quarantine_n":
        int(
            (
                split_df["v1_role"]
                == "LEGACY_TEST_QUARANTINE"
            ).sum()
        ),

        "split_csv_sha256":
        sha256_file(
            SPLIT_CSV
        ),

        "note":
        (
            "Legacy Val/Test are historical "
            "quarantine sets. "
            "This split still requires the "
            "separate historical-exposure audit "
            "before calling V1 FinalTest pristine."
        ),
    }

    SPLIT_JSON.write_text(
        json.dumps(
            lock_meta,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )


# ---------------------------------------------------------------------
# J. Final hard validation
# ---------------------------------------------------------------------

counts = (
    split_df[
        "v1_role"
    ]
    .value_counts()
    .to_dict()
)

expected = EXPECTED_ROLE_COUNTS

print("\nFinal V1 role counts:")
print(counts)

final_lock_ok, final_lock_errors = validate_split_lock_membership(split_df)
if not final_lock_ok:
    raise RuntimeError(
        "V1 split lock failed exact membership validation:\n- "
        + "\n- ".join(final_lock_errors)
    )


# ---------------------------------------------------------------------
# K. Register artifacts
# ---------------------------------------------------------------------

register_artifact(
    "STAGE_3",
    SPLIT_CSV,
    "v1_split_lock"
)

register_artifact(
    "STAGE_3",
    SPLIT_JSON,
    "v1_split_lock_metadata"
)

register_artifact(
    "STAGE_3",
    LEGACY_MANIFEST,
    "verified_legacy_manifest"
)

stage_result(
    "STAGE_3_SPLIT_LOCK",
    "PASS",
    (
        "Verified legacy 12588/12068/260/260 "
        "manifest loaded and V1 split locked."
    ),
    {
        **counts,
        "legacy_manifest":
        str(LEGACY_MANIFEST)
    }
)

print("\n" + "=" * 100)
print("CELL 8 PASSED")
print("=" * 100)
print("LEGACY MANIFEST :", LEGACY_MANIFEST)
print("Legacy Train    :", len(LEGACY_TRAIN_IDS))
print("Legacy Val      :", len(LEGACY_VAL_IDS))
print("Legacy Test     :", len(LEGACY_TEST_IDS))
print("V1 DEV          :", counts["V1_DEV"])
print("V1 Validation   :", counts["V1_VALIDATION"])
print("V1 FinalTest    :", counts["V1_FINAL_TEST"])
print("=" * 100)

display(
    split_df
    .groupby(
        [
            "v1_role",
            "bone_age_year_bin"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)


CELL 8 — VERIFIED LEGACY SPLIT + NEW V1 HOLDOUT

Manifest candidates:


,path,rows,unique_ids,train,val,test,valid,reason
0,D:\Bone_age\pipeline_output\patient_roi_manife...,1733,1733,1213,260,260,False,COUNT/SPLIT MISMATCH
1,D:\workspace\Bone_age\pipeline_output\patient_...,12588,12588,12068,260,260,True,VERIFIED 12588/12068/260/260
2,D:\Bone_age\patient_roi_manifest.csv,1733,1733,1213,260,260,False,COUNT/SPLIT MISMATCH



✅ VERIFIED LEGACY_MANIFEST:
D:\workspace\Bone_age\pipeline_output\patient_roi_manifest.csv

Verified counts:
Train: 12068
Val  : 260
Test : 260
[LOCK] Existing V1 split reused after exact membership validation.

Final V1 role counts:
{'V1_DEV': 11548, 'V1_VALIDATION': 260, 'V1_FINAL_TEST': 260, 'LEGACY_VAL_QUARANTINE': 260, 'LEGACY_TEST_QUARANTINE': 260}
[STAGE_3_SPLIT_LOCK] PASS: Verified legacy 12588/12068/260/260 manifest loaded and V1 split locked.

CELL 8 PASSED
LEGACY MANIFEST : D:\workspace\Bone_age\pipeline_output\patient_roi_manifest.csv
Legacy Train    : 12068
Legacy Val      : 260
Legacy Test     : 260
V1 DEV          : 11548
V1 Validation   : 260
V1 FinalTest    : 260


bone_age_year_bin,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
v1_role,,,,,,,,,,,,,,,,,,
LEGACY_TEST_QUARANTINE,7,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,13
LEGACY_VAL_QUARANTINE,6,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,14
V1_DEV,29,142,130,329,286,536,661,886,790,1192,1053,1779,1159,1650,384,317,165,60
V1_FINAL_TEST,1,3,3,7,6,12,15,20,18,27,24,40,26,37,9,7,4,1
V1_VALIDATION,1,3,3,7,6,12,15,20,18,27,24,40,26,37,9,7,4,1


In [9]:
# Cell 9 — Split/leakage audit + guarded access API

role_sets = {
    role: set(split_df.loc[split_df["v1_role"].eq(role), "patient_id"].astype(str))
    for role in split_df["v1_role"].unique()
}

roles = list(role_sets)
overlaps = []
for i, a in enumerate(roles):
    for b in roles[i+1:]:
        n = len(role_sets[a] & role_sets[b])
        if n:
            overlaps.append((a,b,n))

if overlaps:
    stage_result("STAGE_4_SPLIT_AUDIT", "FAIL", f"Patient overlap detected: {overlaps}")

if set.union(*role_sets.values()) != set(canonical["patient_id"]):
    stage_result("STAGE_4_SPLIT_AUDIT", "FAIL", "Split lock does not cover canonical 12,588 exactly once.")

V1_DEV_IDS = role_sets["V1_DEV"]
V1_VAL_IDS = role_sets["V1_VALIDATION"]
V1_FINAL_TEST_IDS = role_sets["V1_FINAL_TEST"]
LEGACY_QUARANTINE_IDS = role_sets["LEGACY_VAL_QUARANTINE"] | role_sets["LEGACY_TEST_QUARANTINE"]


# Defense-in-depth: verify role identity, not only zero-overlap/counts.
if role_sets["LEGACY_VAL_QUARANTINE"] != LEGACY_VAL_IDS:
    stage_result(
        "STAGE_4_SPLIT_AUDIT", "FAIL",
        "LEGACY_VAL_QUARANTINE does not exactly match the verified legacy Val260 IDs."
    )
if role_sets["LEGACY_TEST_QUARANTINE"] != LEGACY_TEST_IDS:
    stage_result(
        "STAGE_4_SPLIT_AUDIT", "FAIL",
        "LEGACY_TEST_QUARANTINE does not exactly match the verified legacy Test260 IDs."
    )
if (V1_DEV_IDS | V1_VAL_IDS | V1_FINAL_TEST_IDS) != LEGACY_TRAIN_IDS:
    stage_result(
        "STAGE_4_SPLIT_AUDIT", "FAIL",
        "V1_DEV/V1_VALIDATION/V1_FINAL_TEST union does not exactly equal verified legacy Train12068."
    )

FINAL_TEST_UNLOCKED = False

def get_v1_role(role: str, *, allow_final_test: bool = False) -> pd.DataFrame:
    if role == "V1_FINAL_TEST" and not (allow_final_test and FINAL_TEST_UNLOCKED):
        raise PermissionError(
            "V1_FINAL_TEST is locked. Development code must not access it. "
            "Unlock only after all model/rule selection is frozen."
        )
    ids = role_sets[role]
    return canonical[canonical["patient_id"].isin(ids)].copy()

SPLIT_AUDIT = DIR["qc_reports"] / "04_split_audit.json"
split_audit_payload = {
    "patient_overlap_rows": overlaps,
    "all_12588_covered_exactly_once": True,
    "v1_dev_hash": sha256_text("\n".join(sorted(V1_DEV_IDS))),
    "v1_val_hash": sha256_text("\n".join(sorted(V1_VAL_IDS))),
    "v1_final_test_hash": sha256_text("\n".join(sorted(V1_FINAL_TEST_IDS))),
    "legacy_quarantine_hash": sha256_text("\n".join(sorted(LEGACY_QUARANTINE_IDS))),
}
SPLIT_AUDIT.write_text(json.dumps(split_audit_payload, indent=2), encoding="utf-8")
register_artifact("STAGE_4", SPLIT_AUDIT, "split_leakage_audit")

stage_result("STAGE_4_SPLIT_AUDIT", "PASS", "Patient-level V1 split leakage = 0; final test access guard enabled.")


[STAGE_4_SPLIT_AUDIT] PASS: Patient-level V1 split leakage = 0; final test access guard enabled.


## Detector-data availability policy

The detector dataset is **not downloaded automatically** because its provenance/license must remain auditable.

This notebook searches for a YOLO-format dataset whose class map resolves exactly to the nine required ROIs.  
If only a `data.yaml` metadata file exists but images/labels are missing, the stage is a critical **FAIL**.

Set one of these before running the next cell if discovery is ambiguous:

- Environment variable `BONEAGE_DETECTOR_DATA_ROOT`
- Or place the dataset under a folder such as  
  `D:\workspace\Bone_age\pipeline_output\SMI_Fishman.v4i.yolov11`

Expected classes resolve to:

`Carpal, DP3, MC1, MP3, MP5, PP1, PP3, Radius, Ulna`

In [10]:
# Cell 10 — Detector dataset discovery + availability lock

ROI_ORDER = list(CFG.roi_order)

def norm_roi_class(name: str) -> Optional[str]:
    token = re.sub(r"[^a-z0-9]+", "", str(name).lower())
    aliases = {
        "carpal": "Carpal",
        "dp3": "DP3",
        "mc1": "MC1",
        "mp3": "MP3",
        "mp5": "MP5",
        "pp1": "PP1",
        "pp3": "PP3",
        "pp3mc": "PP3",
        "pp3metacarpal": "PP3",
        "radius": "Radius",
        "ulna": "Ulna",
    }
    return aliases.get(token)

def parse_names(cfg: dict) -> dict[int, str]:
    raw = cfg.get("names", {})
    if isinstance(raw, list):
        return {i: str(v) for i, v in enumerate(raw)}
    if isinstance(raw, dict):
        return {int(k): str(v) for k, v in raw.items()}
    raise ValueError("Unsupported YAML names format.")

def detector_yaml_score(yaml_path: Path) -> tuple[int, dict]:
    try:
        cfg = yaml.safe_load(yaml_path.read_text(encoding="utf-8", errors="replace"))
        if not isinstance(cfg, dict):
            return -999, {}
        names = parse_names(cfg)
        mapped = [norm_roi_class(v) for v in names.values()]
        score = 0
        if len(names) == 9:
            score += 10
        if set(mapped) == set(ROI_ORDER) and None not in mapped:
            score += 100
        if CFG.detector_dataset_name_hint.lower() in str(yaml_path).lower():
            score += 20
        return score, cfg
    except Exception as e:
        warnings.warn(
            f"Detector YAML rejected: {yaml_path} -> {type(e).__name__}: {e}"
        )
        return -999, {}

override = os.environ.get("BONEAGE_DETECTOR_DATA_ROOT", "").strip()

yaml_candidates = []
if override:
    ov = Path(override)
    if ov.is_file() and ov.suffix.lower() in {".yaml",".yml"}:
        yaml_candidates.append(ov.resolve())
    elif ov.is_dir():
        yaml_candidates += list(ov.rglob("data.yaml"))

preferred_roots = [
    SOURCE_ROOT / "pipeline_output" / CFG.detector_dataset_name_hint,
    SOURCE_ROOT / CFG.detector_dataset_name_hint,
    SOURCE_CSV.parent / "pipeline_output" / CFG.detector_dataset_name_hint,
]
for r in preferred_roots:
    if r.is_dir():
        yaml_candidates += list(r.rglob("data.yaml"))

for root in [SOURCE_ROOT, SOURCE_CSV.parent]:
    if root.exists():
        try:
            yaml_candidates += list(root.rglob("data.yaml"))[:100]
        except Exception as e:
            warnings.warn(
                f"Detector YAML scan skipped: {root} -> {type(e).__name__}: {e}"
            )

if IS_KAGGLE:
    try:
        yaml_candidates += list(Path("/kaggle/input").rglob("data.yaml"))[:100]
    except Exception as e:
        warnings.warn(
            f"Kaggle detector YAML scan skipped: {type(e).__name__}: {e}"
        )

yaml_candidates = list(dict.fromkeys(Path(p).resolve() for p in yaml_candidates if Path(p).is_file()))
ranked = sorted(
    [(detector_yaml_score(p)[0], p, detector_yaml_score(p)[1]) for p in yaml_candidates],
    key=lambda x: x[0],
    reverse=True
)

if not ranked or ranked[0][0] < 100:
    report = DIR["qc_reports"] / "10_detector_dataset_not_found.json"
    report.write_text(
        json.dumps(
            {"searched_root": str(SOURCE_ROOT), "yaml_candidates": [str(x) for x in yaml_candidates]},
            indent=2
        ),
        encoding="utf-8"
    )
    stage_result(
        "STAGE_5_DETECTOR_AVAILABILITY",
        "FAIL",
        "No available YOLO detector dataset with the required nine-class map was found. "
        "Provide BONEAGE_DETECTOR_DATA_ROOT; this notebook will not fabricate/download an unverified dataset."
    )

DETECTOR_DATA_YAML = ranked[0][1]
DET_CFG_RAW = ranked[0][2]
DET_NAMES_RAW = parse_names(DET_CFG_RAW)
DET_NAMES_CANON = {cid: norm_roi_class(name) for cid, name in DET_NAMES_RAW.items()}

if set(DET_NAMES_CANON.values()) != set(ROI_ORDER):
    stage_result("STAGE_5_DETECTOR_AVAILABILITY", "FAIL", "Detector class map mismatch.")

DETECTOR_DATA_ROOT = DETECTOR_DATA_YAML.parent.resolve()

def resolve_split_dir(yaml_path: Path, cfg: dict, key: str) -> Optional[Path]:
    aliases = [key]
    if key == "val":
        aliases += ["valid", "validation"]

    entry = None
    for k in aliases:
        if k in cfg:
            entry = cfg[k]
            break

    candidates = []
    if isinstance(entry, str):
        p = Path(entry)
        candidates += [
            p if p.is_absolute() else (yaml_path.parent / p),
        ]
        if cfg.get("path"):
            base = Path(cfg["path"])
            if not base.is_absolute():
                base = yaml_path.parent / base
            candidates.append(base / p)

    conventional = {
        "train": ["train/images", "images/train"],
        "val": ["valid/images", "val/images", "images/val", "images/valid"],
        "test": ["test/images", "images/test"],
    }[key]
    candidates += [yaml_path.parent / rel for rel in conventional]

    for p in candidates:
        try:
            rp = p.resolve()
        except Exception:
            rp = p
        if rp.is_dir():
            return rp
    return None

DET_SPLIT_DIRS = {k: resolve_split_dir(DETECTOR_DATA_YAML, DET_CFG_RAW, k) for k in ["train","val","test"]}

if not all(DET_SPLIT_DIRS.values()):
    stage_result(
        "STAGE_5_DETECTOR_AVAILABILITY",
        "FAIL",
        f"Detector metadata exists but actual split folders are incomplete: {DET_SPLIT_DIRS}"
    )

availability = {
    "data_yaml": str(DETECTOR_DATA_YAML),
    "dataset_root": str(DETECTOR_DATA_ROOT),
    "original_names": DET_NAMES_RAW,
    "canonical_names": DET_NAMES_CANON,
    "split_dirs": {k: str(v) for k,v in DET_SPLIT_DIRS.items()},
    "readme_dataset_exists": (DETECTOR_DATA_ROOT / "README.dataset.txt").is_file(),
    "readme_roboflow_exists": (DETECTOR_DATA_ROOT / "README.roboflow.txt").is_file(),
    "license_status": "NOT_VERIFIED_UNLESS_DOCUMENTED_IN_DATASET_METADATA",
}
AVAIL_FILE = DIR["qc_reports"] / "05_detector_data_availability.json"
AVAIL_FILE.write_text(json.dumps(availability, indent=2, ensure_ascii=False), encoding="utf-8")
register_artifact("STAGE_5", AVAIL_FILE, "detector_data_availability")

detector_docs_present = (
    availability["readme_dataset_exists"]
    or availability["readme_roboflow_exists"]
)

stage_result(
    "STAGE_5_DETECTOR_AVAILABILITY",
    "PASS" if detector_docs_present else "WARNING",
    (
        "Actual detector image/label split directories are available. "
        + (
            "Dataset metadata documentation was found."
            if detector_docs_present
            else "Dataset provenance/license documentation was not found beside the dataset; "
                 "keep license/provenance as NOT_VERIFIED."
        )
    ),
    availability
)


[STAGE_5_DETECTOR_AVAILABILITY] PASS: Actual detector image/label split directories are available. Dataset metadata documentation was found.


In [11]:
# Cell 11 — Full YOLO label audit: exact 9 objects / 9 unique classes / valid normalized boxes

def list_images(path: Path) -> list[Path]:
    return sorted(
        p.resolve()
        for p in Path(path).rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

def infer_label_path(image_path: Path) -> Path:
    p = Path(image_path)
    parts = list(p.parts)
    lower = [str(x).lower() for x in parts]
    if "images" in lower:
        i = len(lower) - 1 - lower[::-1].index("images")
        parts[i] = "labels"
        return Path(*parts).with_suffix(".txt")
    sibling = p.parent.parent / "labels" / p.name
    return sibling.with_suffix(".txt")

split_images = {sp: list_images(DET_SPLIT_DIRS[sp]) for sp in ["train","val","test"]}

valid_rows = []
invalid_rows = []
image_rows = []

for sp, images in split_images.items():
    for image_path in tqdm(images, desc=f"audit detector {sp}"):
        label_path = infer_label_path(image_path)
        objs = []

        if not label_path.is_file():
            image_rows.append({
                "split": sp, "image": str(image_path), "label": str(label_path),
                "label_exists": False, "valid_objects": 0,
                "unique_roi_classes": 0, "complete_9of9": False
            })
            continue

        for line_no, raw in enumerate(label_path.read_text(encoding="utf-8", errors="replace").splitlines(), 1):
            raw = raw.strip()
            if not raw:
                continue
            parts = raw.split()
            if len(parts) != 5:
                invalid_rows.append({
                    "split": sp, "image": str(image_path), "label": str(label_path),
                    "line": line_no, "reason": "NOT_5_COLUMNS", "raw": raw
                })
                continue

            try:
                cls_raw = float(parts[0])
                xc, yc, bw, bh = map(float, parts[1:])
            except Exception:
                invalid_rows.append({
                    "split": sp, "image": str(image_path), "label": str(label_path),
                    "line": line_no, "reason": "PARSE_FAIL", "raw": raw
                })
                continue

            if not np.isfinite(cls_raw) or not float(cls_raw).is_integer():
                invalid_rows.append({
                    "split": sp, "image": str(image_path), "label": str(label_path),
                    "line": line_no, "reason": "NON_INTEGER_CLASS_ID", "raw": raw
                })
                continue

            cls = int(cls_raw)

            if not np.isfinite([xc, yc, bw, bh]).all():
                invalid_rows.append({
                    "split": sp, "image": str(image_path), "label": str(label_path),
                    "line": line_no, "reason": "NON_FINITE_BOX", "raw": raw
                })
                continue

            if cls not in DET_NAMES_CANON:
                invalid_rows.append({
                    "split": sp, "image": str(image_path), "label": str(label_path),
                    "line": line_no, "reason": "BAD_CLASS_ID", "raw": raw
                })
                continue

            if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < bw <= 1 and 0 < bh <= 1):
                invalid_rows.append({
                    "split": sp, "image": str(image_path), "label": str(label_path),
                    "line": line_no, "reason": "BAD_NORMALIZED_BOX", "raw": raw
                })
                continue

            row = {
                "split": sp, "image": str(image_path), "label": str(label_path),
                "class_id": cls, "roi": DET_NAMES_CANON[cls],
                "xc": xc, "yc": yc, "w": bw, "h": bh
            }
            valid_rows.append(row)
            objs.append(row)

        rois = [x["roi"] for x in objs]
        complete = (len(objs) == 9 and len(set(rois)) == 9 and set(rois) == set(ROI_ORDER))
        image_rows.append({
            "split": sp, "image": str(image_path), "label": str(label_path),
            "label_exists": True, "valid_objects": len(objs),
            "unique_roi_classes": len(set(rois)), "complete_9of9": complete
        })

label_audit = pd.DataFrame(valid_rows)
invalid_labels = pd.DataFrame(invalid_rows)
image_annotation_audit = pd.DataFrame(image_rows)

label_audit.to_csv(DIR["qc_reports"] / "11_detector_valid_boxes.csv", index=False)
invalid_labels.to_csv(DIR["qc_reports"] / "11_detector_invalid_labels.csv", index=False)
image_annotation_audit.to_csv(DIR["qc_reports"] / "11_detector_image_annotations.csv", index=False)

summary = (
    image_annotation_audit.groupby("split")
    .agg(
        images=("image","size"),
        label_files=("label_exists","sum"),
        exact_9of9=("complete_9of9","sum")
    )
)
print(summary)
print("\nObjects by ROI:")
print(label_audit["roi"].value_counts().reindex(ROI_ORDER, fill_value=0))

if len(invalid_labels):
    stage_result("STAGE_6_DETECTOR_LABEL_AUDIT", "FAIL", f"{len(invalid_labels)} invalid YOLO label rows found.")
if set(label_audit["roi"].unique()) != set(ROI_ORDER):
    stage_result("STAGE_6_DETECTOR_LABEL_AUDIT", "FAIL", "Not all nine ROI classes are present.")

stage_result(
    "STAGE_6_DETECTOR_LABEL_AUDIT",
    "PASS",
    "YOLO labels parsed; exact-9/9 image status established.",
    summary.reset_index().to_dict(orient="records")
)


audit detector test: 100%|██████████| 380/380 [00:00<00:00, 4846.15it/s]


       images  label_files  exact_9of9
split                                 
test      380          380         380
train    6435         6435        6421
val      1173         1173        1173

Objects by ROI:
roi
Carpal    7988
DP3       7986
MC1       7988
MP3       7988
MP5       7988
PP1       7988
PP3       7988
Radius    7980
Ulna      7982
Name: count, dtype: int64
[STAGE_6_DETECTOR_LABEL_AUDIT] PASS: YOLO labels parsed; exact-9/9 image status established.


In [12]:
# Cell 12 — Traceability + protected-ID exclusion + strict detector split build

SOURCE_IDS = set(canonical["patient_id"].astype(str))

# Detector training is not allowed to see either:
# - new V1 validation/final test
# - historically exposed legacy Val/Test quarantine
PROTECTED_BONEAGE_IDS = V1_VAL_IDS | V1_FINAL_TEST_IDS | LEGACY_QUARANTINE_IDS

def detector_source_pid(path: Path) -> str:
    # Roboflow copies often use "<source>.rf.<hash>.*".
    prefix = Path(path).stem.split(".rf.")[0]
    hits = []
    for token in re.findall(r"\d+", prefix):
        pid = str(int(token)) if token.isdigit() else token
        if pid in SOURCE_IDS:
            hits.append(pid)
    hits = list(dict.fromkeys(hits))
    return hits[0] if len(hits) == 1 else ""

complete_paths = set(
    image_annotation_audit.loc[image_annotation_audit["complete_9of9"], "image"].map(lambda x: str(Path(x).resolve()))
)

rows = []
for sp in ["train","val","test"]:
    for image_path in split_images[sp]:
        rp = image_path.resolve()
        pid = detector_source_pid(rp)
        rows.append({
            "detector_split": sp,
            "image": str(rp),
            "source_patient_id": pid,
            "mapped": bool(pid),
            "complete_label_9of9": str(rp) in complete_paths,
            "protected_boneage_id": bool(pid and pid in PROTECTED_BONEAGE_IDS),
        })

det_map = pd.DataFrame(rows)
mapping_rate = float(det_map["mapped"].mean()) if len(det_map) else 0.0

mapped_group = det_map[det_map["mapped"]].groupby("source_patient_id")["detector_split"].agg(lambda x: sorted(set(x)))
cross_split_pids = {pid for pid, splits in mapped_group.items() if len(splits) > 1}
priority = {"train":0, "val":1, "test":2}
owner = {pid: sorted(splits, key=lambda s: priority[s])[0] for pid, splits in mapped_group.items()}

strict_images = {"train": [], "val": [], "test": []}
audit_rows = []

for _, r in det_map.iterrows():
    sp = str(r["detector_split"])
    pid = str(r["source_patient_id"])
    keep = True
    reason = "KEEP"

    if not bool(r["complete_label_9of9"]):
        keep, reason = False, "EXCLUDE_LABEL_NOT_EXACT_9OF9"
    elif not bool(r["mapped"]):
        keep, reason = False, "EXCLUDE_UNTRACEABLE_SOURCE_ID"
    elif bool(r["protected_boneage_id"]):
        keep, reason = False, "EXCLUDE_PROTECTED_BONEAGE_ID"
    elif pid in cross_split_pids and sp != owner[pid]:
        keep, reason = False, f"EXCLUDE_CROSS_SPLIT_OWNER_{owner[pid]}"

    if keep:
        strict_images[sp].append(Path(r["image"]))

    audit_rows.append({
        **r.to_dict(),
        "cross_split_patient": bool(pid and pid in cross_split_pids),
        "owner_split": owner.get(pid, "") if pid else "",
        "retained": keep,
        "reason": reason,
    })

STRICT_AUDIT = pd.DataFrame(audit_rows)
STRICT_AUDIT_FILE = DIR["qc_reports"] / "12_strict_detector_split_audit.csv"
STRICT_AUDIT.to_csv(STRICT_AUDIT_FILE, index=False)

retained = STRICT_AUDIT[STRICT_AUDIT["retained"]].copy()
retained_sets = {
    sp: set(retained.loc[retained["detector_split"].eq(sp), "source_patient_id"].astype(str))
    for sp in ["train","val","test"]
}

if retained_sets["train"] & PROTECTED_BONEAGE_IDS:
    stage_result("STAGE_7_STRICT_DETECTOR", "FAIL", "Protected bone-age ID survived detector TRAIN filter.")
if retained_sets["train"] & retained_sets["val"] or retained_sets["train"] & retained_sets["test"] or retained_sets["val"] & retained_sets["test"]:
    stage_result("STAGE_7_STRICT_DETECTOR", "FAIL", "Patient overlap remains across strict detector splits.")

sizes = {sp: len(strict_images[sp]) for sp in strict_images}
if sizes["train"] < CFG.detector_min_train_images or sizes["val"] < CFG.detector_min_val_images or sizes["test"] < CFG.detector_min_test_images:
    stage_result("STAGE_7_STRICT_DETECTOR", "FAIL", f"Strict detector split too small: {sizes}")

strict_txt = {}
for sp in ["train","val","test"]:
    p = DIR["detector_data"] / f"{sp}.txt"
    p.write_text("\n".join(str(x.resolve()) for x in strict_images[sp]) + "\n", encoding="utf-8")
    strict_txt[sp] = p.resolve()

STRICT_YOLO_YAML = DIR["detector_data"] / "data_strict_yolo11s_local4050_v1.yaml"
strict_cfg = {
    "train": str(strict_txt["train"]),
    "val": str(strict_txt["val"]),
    "test": str(strict_txt["test"]),
    "names": {int(cid): str(DET_NAMES_CANON[cid]) for cid in sorted(DET_NAMES_CANON)},
}
STRICT_YOLO_YAML.write_text(yaml.safe_dump(strict_cfg, sort_keys=False), encoding="utf-8")

strict_summary = {
    "policy": "TRACEABLE_EXACT_9OF9_PROTECTED_IDS_EXCLUDED",
    "mapping_rate_all_detector_images": mapping_rate,
    "cross_split_mapped_patient_ids": len(cross_split_pids),
    "protected_boneage_ids_n": len(PROTECTED_BONEAGE_IDS),
    "retained": sizes,
    "generic_pretrained_start": CFG.detector_model,
    "legacy_custom_weights_used_to_initialize": False,
}
STRICT_SUMMARY_FILE = DIR["qc_reports"] / "12_strict_detector_summary.json"
STRICT_SUMMARY_FILE.write_text(json.dumps(strict_summary, indent=2), encoding="utf-8")

register_artifact("STAGE_7", STRICT_YOLO_YAML, "strict_detector_yaml")
register_artifact("STAGE_7", STRICT_AUDIT_FILE, "strict_detector_audit")
register_artifact("STAGE_7", STRICT_SUMMARY_FILE, "strict_detector_summary")

status = "WARNING" if mapping_rate < 0.80 else "PASS"
stage_result(
    "STAGE_7_STRICT_DETECTOR",
    status,
    f"Strict traceable exact-9/9 detector dataset ready; mapping rate={mapping_rate:.1%}. "
    "Untraceable images are excluded, not guessed.",
    strict_summary
)
print("Strict sizes:", sizes)
print("Strict YAML :", STRICT_YOLO_YAML)


[STAGE_7_STRICT_DETECTOR] WARNING: Strict traceable exact-9/9 detector dataset ready; mapping rate=77.9%. Untraceable images are excluded, not guessed.
Strict sizes: {'train': 4612, 'val': 814, 'test': 270}
Strict YAML : D:\workspace\Bone_age\BoneAge_LOCAL4050_V1\data\detector_strict\data_strict_yolo11s_local4050_v1.yaml


In [13]:
# Cell 13 — Geometry priors from strict TRAIN annotations (QC support, not anatomical truth)

strict_train_set = {str(p.resolve()) for p in strict_images["train"]}
geom = label_audit[
    label_audit["split"].eq("train")
    & label_audit["image"].map(lambda x: str(Path(x).resolve())).isin(strict_train_set)
].copy()

if len(geom) == 0:
    stage_result("STAGE_8_GEOMETRY_PRIORS", "FAIL", "No strict TRAIN labels available for geometry priors.")

def robust_center_scale(series: pd.Series) -> tuple[float,float]:
    x = series.astype(float).to_numpy()
    med = float(np.median(x))
    mad = float(np.median(np.abs(x-med)))
    scale = max(1.4826 * mad, 1e-4)
    return med, scale

prior_rows = []
for roi in ROI_ORDER:
    g = geom[geom["roi"].eq(roi)]
    if len(g) < CFG.geometry_prior_min_samples:
        stage_result(
            "STAGE_8_GEOMETRY_PRIORS",
            "FAIL",
            (
                f"Too few strict geometry samples for {roi}: {len(g)} "
                f"< required {CFG.geometry_prior_min_samples}"
            )
        )
    row = {"roi": roi, "n": len(g)}
    for col in ["xc","yc","w","h"]:
        med, sc = robust_center_scale(g[col])
        row[f"{col}_median"] = med
        row[f"{col}_scale"] = sc
    prior_rows.append(row)

GEOMETRY_PRIORS = pd.DataFrame(prior_rows).set_index("roi")
GEOMETRY_PRIOR_FILE = DIR["qc_reports"] / "13_detector_geometry_priors.csv"

# Write with stable precision/order, then fingerprint the exact artifact used by QC.
geometry_export = (
    GEOMETRY_PRIORS
    .reset_index()
    .loc[:, [
        "roi", "n",
        "xc_median", "xc_scale",
        "yc_median", "yc_scale",
        "w_median", "w_scale",
        "h_median", "h_scale",
    ]]
)
geometry_export.to_csv(
    GEOMETRY_PRIOR_FILE,
    index=False,
    encoding="utf-8-sig",
    float_format="%.10g",
)

# CRITICAL FIX:
# Cell 18/19/20 use this to invalidate stale ROI cache/rules whenever the
# geometry-prior artifact changes.
GEOMETRY_PRIORS_SHA256 = sha256_file(GEOMETRY_PRIOR_FILE)

if not isinstance(GEOMETRY_PRIORS_SHA256, str) or len(GEOMETRY_PRIORS_SHA256) != 64:
    stage_result(
        "STAGE_8_GEOMETRY_PRIORS",
        "FAIL",
        f"Invalid geometry-prior SHA256: {GEOMETRY_PRIORS_SHA256!r}"
    )

register_artifact(
    "STAGE_8",
    GEOMETRY_PRIOR_FILE,
    "geometry_priors",
    {
        "sha256": GEOMETRY_PRIORS_SHA256,
        "strict_train_rows": int(len(geom)),
        "min_roi_samples": int(min(GEOMETRY_PRIORS["n"])),
    }
)

stage_result(
    "STAGE_8_GEOMETRY_PRIORS",
    "PASS",
    (
        "Robust normalized-box priors built from strict TRAIN only. "
        "Priors are QC support, not anatomical ground truth. "
        f"SHA256={GEOMETRY_PRIORS_SHA256[:12]}..."
    ),
    {
        "geometry_prior_sha256": GEOMETRY_PRIORS_SHA256,
        "per_roi_n": {
            str(k): int(v)
            for k, v in GEOMETRY_PRIORS["n"].to_dict().items()
        },
    }
)

print("Geometry prior file  :", GEOMETRY_PRIOR_FILE)
print("Geometry prior SHA256:", GEOMETRY_PRIORS_SHA256)
display(GEOMETRY_PRIORS)


[STAGE_8_GEOMETRY_PRIORS] PASS: Robust normalized-box priors built from strict TRAIN only. Priors are QC support, not anatomical ground truth. SHA256=a711179f94c4...
Geometry prior file  : D:\workspace\Bone_age\BoneAge_LOCAL4050_V1\qc\reports\13_detector_geometry_priors.csv
Geometry prior SHA256: a711179f94c459407464c6c6aafaa1af4e459a4d527d760ca62bad26535b9975


,n,xc_median,xc_scale,yc_median,yc_scale,w_median,w_scale,h_median,h_scale
roi,,,,,,,,,
Carpal,4612,0.500000,0.109462,0.598437,0.133202,0.268177,0.095485,0.232135,0.133357
DP3,4612,0.495312,0.158607,0.267046,0.210202,0.067731,0.013548,0.066644,0.013264
MC1,4612,0.590625,0.123529,0.622656,0.098454,0.096811,0.021304,0.092078,0.027576
MP3,4612,0.496094,0.126313,0.346875,0.159843,0.082610,0.017074,0.082228,0.015933
MP5,4612,0.369262,0.204617,0.389563,0.122962,0.068877,0.012715,0.066834,0.017786
PP1,4612,0.595596,0.194222,0.553125,0.150577,0.099549,0.020360,0.097656,0.023166
PP3,4612,0.495642,0.057266,0.461719,0.062547,0.108501,0.032724,0.113281,0.022411
Radius,4612,0.505014,0.161580,0.675495,0.197920,0.143177,0.034440,0.134207,0.052449
Ulna,4612,0.411719,0.213124,0.696357,0.173266,0.094688,0.019142,0.092188,0.025148


In [14]:

# Cell 14 — Select the already-trained VERIFIED YOLO detector (NO automatic retraining)
#
# Root-cause fix:
# The original detector already completed training and passed strict validation.
# Do not rebuild its identity from a newly generated config/YAML hash and then
# silently start another 100-epoch run when that fingerprint differs.
#
# Proven project checkpoint from the executed legacy notebook + QC reports:
#   run: yolo11s_strict_e35e0d410b
#   best/frozen SHA256:
#   c06a87c6ba80764b48ecbe87538971b64f6d322c0578f308ceaee6b1d9ae3736
#   Val  mAP50=0.992563, mAP50-95=0.774761
#   Test mAP50=0.988528, mAP50-95=0.770941
#
# Policy: if this exact checkpoint cannot be found and verified, FAIL.
# Never fall back to yolo11s.pt automatically.

if not torch.cuda.is_available():
    stage_result(
        'STAGE_9_DETECTOR_PREFLIGHT', 'FAIL',
        'CUDA GPU is required for detector validation/inference. No training fallback is allowed.'
    )

LEGACY_VERIFIED_RUN_HASH = 'e35e0d410b0d93dbc0a13907abcdc6fa24265d143bd1d474ed4417c989c8971f'
LEGACY_VERIFIED_RUN_SHORT = LEGACY_VERIFIED_RUN_HASH[:10]
LEGACY_VERIFIED_BEST_SHA256 = 'c06a87c6ba80764b48ecbe87538971b64f6d322c0578f308ceaee6b1d9ae3736'
ALLOW_DETECTOR_RETRAIN = False
TRAIN_REUSE_MODE = 'VERIFIED_EXISTING_CHECKPOINT_ONLY'

# Keep training params for provenance only. Cell 15 will not train.
train_params = {
    'model': CFG.detector_model,
    'data': str(STRICT_YOLO_YAML),
    'epochs': CFG.detector_epochs,
    'imgsz': CFG.detector_imgsz,
    'batch': DETECTOR_BATCH,
    'nbs': CFG.detector_nbs,
    'optimizer': CFG.detector_optimizer,
    'lr0': CFG.detector_lr0,
    'lrf': CFG.detector_lrf,
    'momentum': CFG.detector_momentum,
    'weight_decay': CFG.detector_weight_decay,
    'warmup_epochs': CFG.detector_warmup_epochs,
    'patience': CFG.detector_patience,
    'workers': DETECTOR_WORKERS,
    'cache': False,
    'amp': True,
    'seed': CFG.seed,
    'deterministic': True,
    'save': True,
    'save_period': CFG.detector_save_period,
    'val': True,
    'plots': True,
    'verbose': True,
    'device': 0,
    'hsv_h': CFG.hsv_h,
    'hsv_s': CFG.hsv_s,
    'hsv_v': CFG.hsv_v,
    'degrees': CFG.degrees,
    'translate': CFG.translate,
    'scale': CFG.scale,
    'fliplr': CFG.fliplr,
    'flipud': CFG.flipud,
    'mosaic': CFG.mosaic,
    'mixup': CFG.mixup,
    'shear': CFG.shear,
    'perspective': CFG.perspective,
    'close_mosaic': 0,
    'rect': False,
    'multi_scale': 0.0,
    'cos_lr': False,
    'pretrained': True,
}


def _read_json(path: Path):
    if not path.is_file():
        return None
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception as e:
        warnings.warn(f'Cannot read {path}: {type(e).__name__}: {e}')
        return None


# Collect trusted hash/path evidence from the old project reports BEFORE Cell 16
# has a chance to overwrite them during this new run.
trusted_hashes = {LEGACY_VERIFIED_BEST_SHA256}
reported_paths = []
reported_train_hash = None

metrics_old = _read_json(DIR['qc_reports'] / '16_detector_metrics.json')
if isinstance(metrics_old, dict):
    sha = metrics_old.get('best_sha256')
    p = metrics_old.get('best_pt')
    if (
        metrics_old.get('architecture') == 'YOLO11s'
        and metrics_old.get('boneage_v1_final_test_used') is False
        and float(metrics_old.get('strict_val_map50', -1)) >= CFG.detector_min_val_map50
        and float(metrics_old.get('strict_val_map50_95', -1)) >= CFG.detector_min_val_map5095
        and isinstance(sha, str) and len(sha) == 64
    ):
        trusted_hashes.add(sha)
        if p:
            reported_paths.append(Path(p))
        reported_train_hash = metrics_old.get('train_config_hash') or reported_train_hash

stage11_old = _read_json(DIR['qc_reports'] / 'STAGE_11_DETECTOR_VALIDATE.json')
if isinstance(stage11_old, dict) and stage11_old.get('status') == 'PASS':
    d = stage11_old.get('details', {}) or {}
    sha = d.get('best_sha256')
    p = d.get('best_pt')
    if (
        d.get('boneage_v1_final_test_used') is False
        and float(d.get('strict_val_map50', -1)) >= CFG.detector_min_val_map50
        and float(d.get('strict_val_map50_95', -1)) >= CFG.detector_min_val_map5095
        and isinstance(sha, str) and len(sha) == 64
    ):
        trusted_hashes.add(sha)
        if p:
            reported_paths.append(Path(p))
        reported_train_hash = d.get('train_config_hash') or reported_train_hash

stage10_old = _read_json(DIR['qc_reports'] / 'STAGE_10_DETECTOR_TRAIN.json')
if isinstance(stage10_old, dict) and stage10_old.get('status') == 'PASS':
    d = stage10_old.get('details', {}) or {}
    sha = d.get('sha256')
    p = d.get('best_pt')
    if isinstance(sha, str) and len(sha) == 64:
        trusted_hashes.add(sha)
        if p:
            reported_paths.append(Path(p))
        reported_train_hash = d.get('train_config_hash') or reported_train_hash

legacy_run_dir = DIR['detector_models'] / f'yolo11s_strict_{LEGACY_VERIFIED_RUN_SHORT}'
explicit_paths = [
    legacy_run_dir / 'weights' / 'best.pt',
    DIR['detector_models'] / f'yolo11s_strict_frozen_{LEGACY_VERIFIED_RUN_SHORT}.pt',
]

# Search candidates. Filename alone never proves identity; SHA256 must match a trusted hash.
candidates = []
seen = set()
for p in reported_paths + explicit_paths:
    key = str(p)
    if key not in seen:
        candidates.append(p)
        seen.add(key)
for pattern in ['yolo11s_strict_*/weights/best.pt', 'yolo11s_strict_frozen_*.pt']:
    for p in sorted(DIR['detector_models'].glob(pattern)):
        key = str(p)
        if key not in seen:
            candidates.append(p)
            seen.add(key)

candidate_audit = []
verified = []
for p in candidates:
    rec = {'path': str(p), 'exists': p.is_file(), 'sha256': None, 'trusted_sha': False}
    if p.is_file():
        try:
            sha = sha256_file(p)
            rec['sha256'] = sha
            rec['trusted_sha'] = sha in trusted_hashes
            if rec['trusted_sha']:
                verified.append((p, sha))
        except Exception as e:
            rec['error'] = f'{type(e).__name__}: {e}'
    candidate_audit.append(rec)

# Prefer the original e35 best.pt, then any best.pt, then an exact-hash frozen copy.
def _priority(item):
    p, sha = item
    s = str(p)
    return (
        int(LEGACY_VERIFIED_RUN_SHORT in s and p.name.lower() == 'best.pt'),
        int(p.name.lower() == 'best.pt'),
        int(LEGACY_VERIFIED_RUN_SHORT in s),
    )
verified = sorted(verified, key=_priority, reverse=True)

SELECTION_FILE = DIR['qc_reports'] / '14_verified_detector_selection.json'

if not verified:
    fail_payload = {
        'status': 'FAIL_NO_TRUSTED_CHECKPOINT',
        'automatic_retraining_allowed': False,
        'expected_run': f'yolo11s_strict_{LEGACY_VERIFIED_RUN_SHORT}',
        'expected_sha256': LEGACY_VERIFIED_BEST_SHA256,
        'trusted_hashes': sorted(trusted_hashes),
        'candidates': candidate_audit,
        'time': pd.Timestamp.now(tz='UTC').isoformat(),
    }
    SELECTION_FILE.write_text(json.dumps(fail_payload, indent=2, ensure_ascii=False), encoding='utf-8')
    stage_result(
        'STAGE_9_DETECTOR_PREFLIGHT', 'FAIL',
        'Previously validated detector could not be found with a trusted SHA256. '
        'Automatic retraining is disabled. Restore the old e35e0d410b best.pt/frozen .pt.',
        fail_payload,
    )

BEST_PT, BEST_HASH = verified[0]
TRAIN_CONFIG_HASH = reported_train_hash or LEGACY_VERIFIED_RUN_HASH
if not isinstance(TRAIN_CONFIG_HASH, str) or not TRAIN_CONFIG_HASH:
    TRAIN_CONFIG_HASH = LEGACY_VERIFIED_RUN_HASH

if BEST_PT.parent.name.lower() == 'weights':
    RUN_DIR = BEST_PT.parent.parent
else:
    RUN_DIR = legacy_run_dir if legacy_run_dir.is_dir() else DIR['detector_models']

RUN_NAME = RUN_DIR.name
LAST_PT = RUN_DIR / 'weights' / 'last.pt'
TRAIN_COMPLETE_MARKER = RUN_DIR / 'TRAIN_COMPLETE.json'

selection_payload = {
    'status': 'PASS_VERIFIED_EXISTING_CHECKPOINT',
    'automatic_retraining_allowed': False,
    'selected_checkpoint': str(BEST_PT.resolve()),
    'selected_sha256': BEST_HASH,
    'run_name': RUN_NAME,
    'run_dir': str(RUN_DIR.resolve()),
    'train_config_hash': TRAIN_CONFIG_HASH,
    'known_legacy_run_hash': LEGACY_VERIFIED_RUN_HASH,
    'known_legacy_sha256': LEGACY_VERIFIED_BEST_SHA256,
    'candidates': candidate_audit,
    'time': pd.Timestamp.now(tz='UTC').isoformat(),
}
SELECTION_FILE.write_text(json.dumps(selection_payload, indent=2, ensure_ascii=False), encoding='utf-8')

print('=' * 92)
print('[REUSE VERIFIED] Existing trained detector selected.')
print('Cell 15 will NOT train YOLO.')
print('Checkpoint :', BEST_PT)
print('SHA256     :', BEST_HASH)
print('Run dir    :', RUN_DIR)
print('Train hash :', TRAIN_CONFIG_HASH)
print('=' * 92)

stage_result(
    'STAGE_9_DETECTOR_PREFLIGHT', 'PASS',
    'Existing validated YOLO11s selected by project provenance + exact SHA256. Automatic retraining disabled.',
    {
        'checkpoint': str(BEST_PT),
        'sha256': BEST_HASH,
        'run_name': RUN_NAME,
        'train_config_hash': TRAIN_CONFIG_HASH,
        'automatic_retraining_allowed': False,
    }
)


[REUSE VERIFIED] Existing trained detector selected.
Cell 15 will NOT train YOLO.
Checkpoint : D:\workspace\Bone_age\BoneAge_LOCAL4050_V1\checkpoints\detector_yolo11s\yolo11s_strict_e35e0d410b\weights\best.pt
SHA256     : c06a87c6ba80764b48ecbe87538971b64f6d322c0578f308ceaee6b1d9ae3736
Run dir    : D:\workspace\Bone_age\BoneAge_LOCAL4050_V1\checkpoints\detector_yolo11s\yolo11s_strict_e35e0d410b
Train hash : e35e0d410b0d93dbc0a13907abcdc6fa24265d143bd1d474ed4417c989c8971f
[STAGE_9_DETECTOR_PREFLIGHT] PASS: Existing validated YOLO11s selected by project provenance + exact SHA256. Automatic retraining disabled.


In [15]:

# Cell 15 — REUSE existing YOLO11s checkpoint; NO TRAINING CODE
#
# This cell deliberately does not invoke Ultralytics training.
# Missing or modified checkpoints must FAIL rather than trigger a fresh 100-epoch run.

if TRAIN_REUSE_MODE != 'VERIFIED_EXISTING_CHECKPOINT_ONLY':
    raise RuntimeError('Cell 14 did not select a verified existing detector.')
if ALLOW_DETECTOR_RETRAIN:
    raise RuntimeError('Safety invariant failed: detector retraining must remain disabled here.')
if not BEST_PT.is_file():
    raise FileNotFoundError(f'Verified detector checkpoint is missing: {BEST_PT}')

actual_hash = sha256_file(BEST_PT)
if actual_hash != BEST_HASH:
    raise RuntimeError(
        f'Detector SHA256 changed after Cell 14: expected={BEST_HASH}, actual={actual_hash}'
    )

results_file = RUN_DIR / 'results.csv'
results_rows = 0
if results_file.is_file() and results_file.stat().st_size > 0:
    try:
        results_rows = int(len(pd.read_csv(results_file)))
    except Exception as e:
        warnings.warn(f'Could not parse historical results.csv: {type(e).__name__}: {e}')

marker = None
if TRAIN_COMPLETE_MARKER.is_file():
    try:
        marker = json.loads(TRAIN_COMPLETE_MARKER.read_text(encoding='utf-8'))
    except Exception as e:
        warnings.warn(f'Could not parse historical TRAIN_COMPLETE.json: {type(e).__name__}: {e}')

reuse_receipt = {
    'status': 'REUSED_EXISTING_DETECTOR_NO_TRAINING',
    'checkpoint': str(BEST_PT.resolve()),
    'sha256': actual_hash,
    'run_name': RUN_NAME,
    'run_dir': str(RUN_DIR.resolve()),
    'train_config_hash': TRAIN_CONFIG_HASH,
    'results_rows': results_rows,
    'train_complete_marker_present': TRAIN_COMPLETE_MARKER.is_file(),
    'train_complete_marker': marker,
    'training_executed': False,
    'automatic_retraining_allowed': False,
    'time': pd.Timestamp.now(tz='UTC').isoformat(),
}
REUSE_RECEIPT = DIR['qc_reports'] / '15_detector_reuse_no_training.json'
REUSE_RECEIPT.write_text(json.dumps(reuse_receipt, indent=2, ensure_ascii=False, default=str), encoding='utf-8')

print('[REUSE] Detector training SKIPPED.')
print('Checkpoint      :', BEST_PT)
print('SHA256          :', actual_hash)
print('Historical rows :', results_rows)
print('Training now    : NO')

register_artifact('STAGE_10', BEST_PT, 'verified_existing_yolo11s_checkpoint', {'sha256': actual_hash, 'reused': True})
register_artifact('STAGE_10', REUSE_RECEIPT, 'detector_reuse_receipt')
stage_result(
    'STAGE_10_DETECTOR_TRAIN', 'PASS',
    'Existing validated YOLO11s checkpoint reused. No detector training was executed.',
    {
        'best_pt': str(BEST_PT),
        'sha256': actual_hash,
        'train_config_hash': TRAIN_CONFIG_HASH,
        'training_executed': False,
    }
)


[REUSE] Detector training SKIPPED.
Checkpoint      : D:\workspace\Bone_age\BoneAge_LOCAL4050_V1\checkpoints\detector_yolo11s\yolo11s_strict_e35e0d410b\weights\best.pt
SHA256          : c06a87c6ba80764b48ecbe87538971b64f6d322c0578f308ceaee6b1d9ae3736
Historical rows : 57
Training now    : NO
[STAGE_10_DETECTOR_TRAIN] PASS: Existing validated YOLO11s checkpoint reused. No detector training was executed.


In [16]:

# Cell 16 — LIVE revalidation of selected existing detector
#
# R3 test policy:
# - strict detector VAL is used for routine development/revalidation
# - detector TEST is NOT repeatedly inspected during development by default
# - historical detector-test metrics remain provenance only
# - V1_FINAL_TEST remains completely separate and locked

if not BEST_PT.is_file():
    stage_result(
        "STAGE_11_DETECTOR_VALIDATE",
        "FAIL",
        f"Selected existing best.pt missing: {BEST_PT}"
    )

BEST_HASH = sha256_file(BEST_PT)

args_yaml = RUN_DIR / "args.yaml"
saved_args = {}

if args_yaml.is_file():
    saved_args = yaml.safe_load(args_yaml.read_text(encoding="utf-8")) or {}

    critical_expected = {
        "imgsz": CFG.detector_imgsz,
        "batch": DETECTOR_BATCH,
        "nbs": CFG.detector_nbs,
        "optimizer": CFG.detector_optimizer,
    }

    arg_mismatches = {}
    for k, expected in critical_expected.items():
        if k not in saved_args:
            arg_mismatches[k] = {"expected": expected, "actual": "MISSING"}
            continue

        actual = saved_args[k]
        if str(actual).lower() != str(expected).lower():
            arg_mismatches[k] = {"expected": expected, "actual": actual}

    if arg_mismatches:
        stage_result(
            "STAGE_11_DETECTOR_VALIDATE",
            "FAIL",
            (
                "Existing checkpoint args.yaml conflicts with current detector-training "
                f"assumptions: {arg_mismatches}"
            ),
            arg_mismatches,
        )
else:
    warnings.warn(
        f"Original args.yaml not found at {args_yaml}. Proceeding because "
        "checkpoint provenance/SHA256 is verified and strict VAL is rerun now."
    )

model = YOLO(str(BEST_PT))

val_metrics = model.val(
    data=str(STRICT_YOLO_YAML),
    split="val",
    imgsz=CFG.detector_imgsz,
    batch=DETECTOR_BATCH,
    workers=DETECTOR_WORKERS,
    device=0,
    plots=True,
    verbose=False,
)

val_map50 = float(val_metrics.box.map50)
val_map5095 = float(val_metrics.box.map)

test_map50 = None
test_map5095 = None
detector_test_status = "LOCKED_DURING_DEVELOPMENT"

if CFG.detector_run_test_during_development:
    warnings.warn(
        "Detector TEST was explicitly unlocked during development. "
        "Do not use repeated TEST inspection for model/hyperparameter selection."
    )
    test_metrics = model.val(
        data=str(STRICT_YOLO_YAML),
        split="test",
        imgsz=CFG.detector_imgsz,
        batch=DETECTOR_BATCH,
        workers=DETECTOR_WORKERS,
        device=0,
        plots=True,
        verbose=False,
    )
    test_map50 = float(test_metrics.box.map50)
    test_map5095 = float(test_metrics.box.map)
    detector_test_status = "EXPLICITLY_RUN"

per_class = []
try:
    maps = list(val_metrics.box.maps)
    for i, value in enumerate(maps):
        per_class.append({
            "class_id": i,
            "roi": model.names[i],
            "val_map50_95": float(value),
        })
except Exception as e:
    warnings.warn(
        f"Per-class detector metrics could not be extracted: "
        f"{type(e).__name__}: {e}"
    )

per_class_df = pd.DataFrame(per_class)
per_class_file = DIR["qc_reports"] / "16_detector_val_per_class.csv"
per_class_df.to_csv(per_class_file, index=False, encoding="utf-8-sig")

metrics_payload = {
    "architecture": "YOLO11s",
    "reused_existing_checkpoint": True,
    "training_executed_this_run": False,
    "best_pt": str(BEST_PT.resolve()),
    "best_sha256": BEST_HASH,
    "strict_val_map50": val_map50,
    "strict_val_map50_95": val_map5095,
    "strict_detector_test_map50": test_map50,
    "strict_detector_test_map50_95": test_map5095,
    "detector_test_status": detector_test_status,
    "boneage_v1_final_test_used": False,
    "train_config_hash": TRAIN_CONFIG_HASH,
}

METRICS_FILE = DIR["qc_reports"] / "16_detector_metrics.json"
METRICS_FILE.write_text(
    json.dumps(metrics_payload, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

if (
    val_map50 < CFG.detector_min_val_map50
    or val_map5095 < CFG.detector_min_val_map5095
):
    stage_result(
        "STAGE_11_DETECTOR_VALIDATE",
        "FAIL",
        (
            "Reused detector failed CURRENT strict validation floors: "
            f"mAP50={val_map50:.4f}, mAP50-95={val_map5095:.4f}."
        ),
        metrics_payload,
    )

matching_frozen = None
for p in sorted(DIR["detector_models"].glob("yolo11s_strict_frozen_*.pt")):
    try:
        if sha256_file(p) == BEST_HASH:
            matching_frozen = p
            break
    except Exception as e:
        warnings.warn(
            f"Could not hash frozen candidate {p}: {type(e).__name__}: {e}"
        )

if matching_frozen is not None:
    FROZEN_DETECTOR = matching_frozen
else:
    FROZEN_DETECTOR = (
        DIR["detector_models"]
        / f"yolo11s_strict_frozen_{str(TRAIN_CONFIG_HASH)[:10]}.pt"
    )
    if FROZEN_DETECTOR.is_file():
        existing_hash = sha256_file(FROZEN_DETECTOR)
        if existing_hash != BEST_HASH:
            raise RuntimeError(
                "Frozen target exists but has a different SHA256: "
                f"{FROZEN_DETECTOR}"
            )
    else:
        shutil.copy2(BEST_PT, FROZEN_DETECTOR)

FROZEN_HASH = sha256_file(FROZEN_DETECTOR)

if FROZEN_HASH != BEST_HASH:
    stage_result(
        "STAGE_11_DETECTOR_VALIDATE",
        "FAIL",
        "Frozen detector SHA256 does not match selected validated checkpoint."
    )

register_artifact(
    "STAGE_11",
    FROZEN_DETECTOR,
    "frozen_yolo11s_detector",
    {"sha256": FROZEN_HASH, "reused_existing_checkpoint": True},
)
register_artifact("STAGE_11", METRICS_FILE, "detector_metrics")
register_artifact("STAGE_11", per_class_file, "detector_val_per_class")

stage_result(
    "STAGE_11_DETECTOR_VALIDATE",
    "PASS",
    (
        f"Existing YOLO11s revalidated without retraining. "
        f"strict VAL mAP50={val_map50:.4f}, mAP50-95={val_map5095:.4f}. "
        f"Detector TEST status={detector_test_status}."
    ),
    metrics_payload,
)

print("Reused best.pt  :", BEST_PT)
print("Frozen detector :", FROZEN_DETECTOR)
print("SHA256          :", FROZEN_HASH)
print("Training now    : NO")
print("Detector TEST   :", detector_test_status)
display(per_class_df)


Ultralytics 8.4.118  Python-3.11.8 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO11s summary (fused): 101 layers, 9,416,283 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 349.3102.8 MB/s, size: 23.9 KB)
val: Scanning D:\workspace\Bone_age\pipeline_output\SMI_Fishman.v4i.yolov11\valid\labels.cache... 814 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 814/814  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 102/102 8.0it/s 12.7s0.1s
                   all        814       7326      0.994      0.994      0.993      0.775
Speed: 0.3ms preprocess, 7.4ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to C:\Users\ASUS\Downloads\runs\detect\val-5
[STAGE_11_DETECTOR_VALIDATE] PASS: Existing YOLO11s revalidated without retraining. strict VAL mAP50=0.9926, mAP50-95=0.7748. Detector TEST status=LOCKED_DURING_DEVELOPMENT.
Reused best.pt  : D:\wo

,class_id,roi,val_map50_95
0,0,Carpal,0.896803
1,1,DP3,0.830135
2,2,MC1,0.433649
3,3,MP3,0.849552
4,4,MP5,0.767566
5,5,PP1,0.784414
6,6,PP3,0.829119
7,7,Radius,0.804836
8,8,Ulna,0.776774


## ROI proposal and anatomy-QC policy

The frozen detector is a **proposal generator**.

A proposal is not automatically accepted merely because:

- a file exists,
- YOLO found nine boxes,
- confidence is high,
- the crop is sharp.

Every proposal receives:

- detector confidence,
- robust geometry-deviation score from strict detector TRAIN,
- hand-foreground overlap,
- marker/text risk flag,
- technical crop checks,
- status = `ACCEPT / NEEDS_REVIEW / REJECT`.

The QC thresholds below are conservative engineering heuristics and **must be validated on the DEV pilot contact sheets**.
They are not claimed to be clinically validated anatomical classifiers.

Final crop pixels always come from the **original radiograph**.

In [17]:

# Cell 17 — ROI QC utilities: robust X-ray read + advisory anatomy-risk heuristics
#
# ROOT-CAUSE R2:
# The previous policy used hand-mask overlap <0.10 as automatic REJECT with
# "OUTSIDE_HAND_OR_MARKER_RISK". Visual audit of the uploaded pilot REJECT sheets
# showed that this heuristic rejected many obvious bone crops.
#
# New policy:
# - invalid/out-of-bounds/tiny crops can be REJECTED automatically
# - detector confidence, geometry and foreground-mask overlap can trigger NEEDS_REVIEW
# - foreground-mask overlap is NOT a marker detector and MUST NOT auto-reject anatomy
# - marker/text risk remains a review signal until a dedicated validated validator exists

XRAY_READER_VERSION = "20260817_gray_safe_v2"
ROI_QC_CODE_VERSION = "20260817_roi_qc_rootcause_r2"

def _ensure_gray_2d(img: np.ndarray, path: Path) -> np.ndarray:
    arr = np.asarray(img)

    if arr.ndim == 2:
        gray = arr
    elif arr.ndim == 3:
        channels = int(arr.shape[2])
        if channels == 1:
            gray = arr[..., 0]
        elif channels == 3:
            gray = cv2.cvtColor(arr, cv2.COLOR_BGR2GRAY)
        elif channels == 4:
            gray = cv2.cvtColor(arr, cv2.COLOR_BGRA2GRAY)
        else:
            raise RuntimeError(
                f"Unsupported X-ray channel count={channels}: {path} "
                f"shape={arr.shape} dtype={arr.dtype}"
            )
    else:
        raise RuntimeError(
            f"Unsupported X-ray dimensions: {path} "
            f"shape={arr.shape} dtype={arr.dtype}"
        )

    if gray.ndim != 2 or gray.size == 0:
        raise RuntimeError(
            f"X-ray did not resolve to a valid 2-D grayscale image: {path} "
            f"shape={gray.shape} dtype={gray.dtype}"
        )

    return np.ascontiguousarray(gray)


def read_xray_u8(path: Path) -> np.ndarray:
    p = Path(path)

    if not p.is_file():
        raise FileNotFoundError(f"X-ray file does not exist: {p}")

    img = None
    read_errors = []

    try:
        img = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    except Exception as e:
        read_errors.append(f"cv2.imread: {type(e).__name__}: {e}")

    if img is None:
        try:
            data = np.fromfile(str(p), dtype=np.uint8)
            if data.size == 0:
                raise RuntimeError("file read returned 0 bytes")
            img = cv2.imdecode(data, cv2.IMREAD_UNCHANGED)
        except Exception as e:
            read_errors.append(
                f"np.fromfile/cv2.imdecode: {type(e).__name__}: {e}"
            )
            img = None

    if img is None:
        detail = " | ".join(read_errors) if read_errors else "OpenCV returned None"
        raise RuntimeError(f"Cannot read X-ray: {p} | {detail}")

    gray = _ensure_gray_2d(img, p)

    if gray.dtype != np.uint8:
        arr = gray.astype(np.float32, copy=False)

        finite = np.isfinite(arr)
        if not finite.all():
            bad_n = int((~finite).sum())
            raise RuntimeError(
                f"Non-finite X-ray pixels detected: {p} | count={bad_n}"
            )

        lo, hi = np.percentile(arr, [0.5, 99.5])
        if not np.isfinite(lo) or not np.isfinite(hi):
            raise RuntimeError(f"Invalid intensity percentiles: {p}")

        if hi <= lo:
            lo, hi = float(arr.min()), float(arr.max())

        if hi <= lo:
            raise RuntimeError(
                f"Degenerate/constant X-ray intensity range: {p} | value={lo}"
            )

        arr = np.clip((arr - lo) / (hi - lo) * 255.0, 0, 255)
        gray = arr.astype(np.uint8)

    if gray.shape[0] < 32 or gray.shape[1] < 32:
        raise RuntimeError(
            f"X-ray is implausibly small: {p} | shape={gray.shape}"
        )

    if int(gray.min()) == int(gray.max()):
        raise RuntimeError(
            f"Degenerate/constant uint8 X-ray: {p} | value={int(gray.min())}"
        )

    return np.ascontiguousarray(gray)


def hand_foreground_mask(gray: np.ndarray) -> np.ndarray:
    if gray.ndim != 2 or gray.dtype != np.uint8:
        raise ValueError(
            f"hand_foreground_mask expects 2-D uint8; "
            f"got shape={gray.shape}, dtype={gray.dtype}"
        )

    g = cv2.GaussianBlur(gray, (5, 5), 0)
    _, otsu = cv2.threshold(
        g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    k = max(5, int(round(min(gray.shape[:2]) * 0.015)))
    if k % 2 == 0:
        k += 1

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    mask = cv2.morphologyEx(otsu, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.dilate(mask, kernel, iterations=1)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n <= 1:
        return mask > 0

    idx = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    main = (labels == idx).astype(np.uint8) * 255
    main = cv2.dilate(main, kernel, iterations=2)
    result = main > 0

    area_frac = float(result.mean())
    if area_frac <= 0.001 or area_frac >= 0.999:
        warnings.warn(
            f"Foreground-mask heuristic has extreme area fraction={area_frac:.4f}. "
            "It is advisory only and must not be treated as anatomical truth."
        )

    return result


def clip_box(xyxy, w: int, h: int):
    x1, y1, x2, y2 = map(float, xyxy)
    x1 = max(0.0, min(x1, w - 1))
    x2 = max(0.0, min(x2, w))
    y1 = max(0.0, min(y1, h - 1))
    y2 = max(0.0, min(y2, h))
    if x2 <= x1 or y2 <= y1:
        return None
    return [x1, y1, x2, y2]


def normalized_box(xyxy, w: int, h: int) -> dict:
    x1, y1, x2, y2 = xyxy
    return {
        "xc": ((x1 + x2) / 2) / w,
        "yc": ((y1 + y2) / 2) / h,
        "w": (x2 - x1) / w,
        "h": (y2 - y1) / h,
    }


def geometry_z(roi: str, xyxy, w: int, h: int) -> float:
    b = normalized_box(xyxy, w, h)
    p = GEOMETRY_PRIORS.loc[roi]
    z = []
    for c in ["xc", "yc", "w", "h"]:
        z.append(
            abs(b[c] - float(p[f"{c}_median"]))
            / float(p[f"{c}_scale"])
        )
    return float(max(z))


def box_hand_overlap(mask: np.ndarray, xyxy) -> float:
    h, w = mask.shape[:2]
    c = clip_box(xyxy, w, h)
    if c is None:
        return 0.0

    x1, y1, x2, y2 = map(lambda v: int(round(v)), c)
    crop = mask[y1:y2, x1:x2]
    return float(crop.mean()) if crop.size else 0.0


def technical_crop_metrics(gray: np.ndarray, xyxy) -> dict:
    h, w = gray.shape
    c = clip_box(xyxy, w, h)

    if c is None:
        return {"valid": False}

    x1, y1, x2, y2 = map(lambda v: int(round(v)), c)
    crop = gray[y1:y2, x1:x2]

    if crop.size == 0:
        return {"valid": False}

    return {
        "valid": True,
        "height": int(crop.shape[0]),
        "width": int(crop.shape[1]),
        "std": float(crop.std()),
        "p01": float(np.percentile(crop, 1)),
        "p99": float(np.percentile(crop, 99)),
        "bright_frac": float((crop >= 245).mean()),
        "dark_frac": float((crop <= 10).mean()),
    }


def qc_proposal(
    roi: str,
    conf: float,
    xyxy,
    gray: np.ndarray,
    mask: np.ndarray,
) -> dict:
    h, w = gray.shape
    c = clip_box(xyxy, w, h)

    if c is None:
        return {
            "status": "REJECT",
            "reason": "OUT_OF_BOUNDS",
            "geometry_z": np.inf,
            "hand_overlap": 0.0,
            "heuristic_only": False,
        }

    gz = geometry_z(roi, c, w, h)
    overlap = box_hand_overlap(mask, c)
    tech = technical_crop_metrics(gray, c)

    reasons = []
    status = "ACCEPT"

    # Automatic REJECT is reserved for objective technical invalidity.
    if (
        not tech.get("valid", False)
        or min(tech.get("height", 0), tech.get("width", 0)) < 20
    ):
        status = "REJECT"
        reasons.append("INVALID_OR_TINY_CROP")

    # Confidence is a detector signal, not anatomical proof.
    if status != "REJECT" and conf < CFG.auto_accept_conf:
        status = "NEEDS_REVIEW"
        reasons.append("LOW_DETECTOR_CONF")

    # Foreground mask is known to have false negatives on valid finger/wrist crops.
    # Therefore it can ONLY escalate to human review.
    if status != "REJECT" and overlap < CFG.accept_hand_overlap_min:
        status = "NEEDS_REVIEW"
        if overlap < CFG.reject_hand_overlap_below:
            reasons.append("VERY_LOW_FOREGROUND_OVERLAP_HEURISTIC")
        else:
            reasons.append("LOW_FOREGROUND_OVERLAP_HEURISTIC")

    # Geometry priors were learned from detector-training framing.
    # Deployment radiographs can have different rotation/canvas/background,
    # so even severe geometry deviation is a REVIEW signal, not auto-reject.
    if status != "REJECT" and gz > CFG.accept_geometry_z_max:
        status = "NEEDS_REVIEW"
        if gz >= CFG.reject_geometry_z_min:
            reasons.append("SEVERE_GEOMETRY_SHIFT")
        else:
            reasons.append("GEOMETRY_SUSPECT")

    # Keep this as a weak risk flag only.
    if (
        status != "REJECT"
        and tech.get("bright_frac", 0) > 0.35
        and overlap < 0.35
    ):
        status = "NEEDS_REVIEW"
        reasons.append("BRIGHT_MARKER_TEXT_RISK_HEURISTIC")

    return {
        "status": status,
        "reason": "|".join(sorted(set(reasons))) if reasons else "PASS",
        "geometry_z": gz,
        "hand_overlap": overlap,
        "heuristic_only": True,
        **tech,
    }


# Regression tests for channel handling.
_test_2d = _ensure_gray_2d(
    np.zeros((32, 32), dtype=np.uint8),
    Path("<selftest_2d>")
)
_test_1ch = _ensure_gray_2d(
    np.zeros((32, 32, 1), dtype=np.uint8),
    Path("<selftest_1ch>")
)
_test_3ch = _ensure_gray_2d(
    np.zeros((32, 32, 3), dtype=np.uint8),
    Path("<selftest_3ch>")
)
_test_4ch = _ensure_gray_2d(
    np.zeros((32, 32, 4), dtype=np.uint8),
    Path("<selftest_4ch>")
)
assert _test_2d.shape == (32, 32)
assert _test_1ch.shape == (32, 32)
assert _test_3ch.shape == (32, 32)
assert _test_4ch.shape == (32, 32)

stage_result(
    "STAGE_12_ROI_QC_UTILS",
    "PASS",
    (
        "QC utilities loaded. Foreground-overlap and geometry are advisory review "
        "signals only; they can no longer auto-reject plausible anatomy."
    )
)


[STAGE_12_ROI_QC_UTILS] PASS: QC utilities loaded. Foreground-overlap and geometry are advisory review signals only; they can no longer auto-reject plausible anatomy.


In [18]:

# Cell 18 — Frozen YOLO proposal with R3 deployment-domain rescue
#
# Key R3 fixes:
# 1) rescue continues for MISSING OR low-confidence ROI, not merely "any proposal exists"
# 2) rescue floor may be 0.05, but low-confidence detections remain NEEDS_REVIEW
# 3) candidate selection no longer lets heuristic QC status outrank detector confidence
# 4) cross-view consensus is recorded as evidence but does not fabricate confidence
# 5) transformed boxes are always mapped back to original X-ray coordinates

ROI_PROPOSAL_CODE_VERSION = "20260817_roi_proposal_rootcause_r3"

detector = YOLO(str(FROZEN_DETECTOR))

model_names = {
    int(k): norm_roi_class(v)
    for k, v in detector.names.items()
}

if set(model_names.values()) != set(ROI_ORDER):
    stage_result(
        "STAGE_13_ROI_PROPOSAL",
        "FAIL",
        f"Frozen detector class map mismatch: {detector.names}"
    )

PROPOSAL_PASSES = [
    {
        "name": "RAW640_BASE",
        "kind": "raw",
        "enabled": True,
        "imgsz": int(CFG.primary_inference_imgsz),
        "conf": float(CFG.auto_conf_floor),
        "run_when": "always",
    },
    {
        "name": "RAW960_RESCUE",
        "kind": "raw",
        "enabled": True,
        "imgsz": int(CFG.rescue_inference_imgsz),
        "conf": float(CFG.rescue_conf_floor),
        "run_when": "missing_or_low_confidence_roi",
    },
    {
        "name": "RAW1280_RESCUE",
        "kind": "raw",
        "enabled": bool(CFG.enable_1280_rescue),
        "imgsz": 1280,
        "conf": float(CFG.rescue_conf_floor),
        "run_when": "missing_or_low_confidence_roi",
    },
]

if CFG.enable_rotation_rescue:
    PROPOSAL_PASSES.extend([
        {
            "name": f"ROT{float(angle):+g}_960_RESCUE",
            "kind": "rotation_rescue",
            "enabled": True,
            "angle_deg": float(angle),
            "imgsz": int(CFG.rescue_inference_imgsz),
            "conf": float(CFG.rescue_conf_floor),
            "run_when": "missing_or_low_confidence_roi",
            "box_mapping": "inverse_affine_to_original",
        }
        for angle in CFG.rotation_rescue_degrees
    ])

PROPOSAL_POLICY_HASH = sha256_text(
    json.dumps(PROPOSAL_PASSES, sort_keys=True, default=str)
)

print("Proposal passes      :", len(PROPOSAL_PASSES))
print("Proposal policy hash :", PROPOSAL_POLICY_HASH)


def rotate_bound_with_inverse(gray: np.ndarray, angle_deg: float):
    h, w = gray.shape[:2]
    center = (w / 2.0, h / 2.0)

    M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    cos = abs(M[0, 0])
    sin = abs(M[0, 1])

    new_w = int(np.ceil(h * sin + w * cos))
    new_h = int(np.ceil(h * cos + w * sin))

    M[0, 2] += new_w / 2.0 - center[0]
    M[1, 2] += new_h / 2.0 - center[1]

    rotated = cv2.warpAffine(
        gray,
        M,
        (new_w, new_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=int(np.median(gray[: max(1, h // 20), :])),
    )

    return rotated, cv2.invertAffineTransform(M)


def map_box_affine_to_original(xyxy, inverse_affine, orig_w: int, orig_h: int):
    x1, y1, x2, y2 = map(float, xyxy)
    pts = np.array(
        [[[x1, y1], [x2, y1], [x2, y2], [x1, y2]]],
        dtype=np.float32,
    )
    mapped = cv2.transform(pts, inverse_affine)[0]
    out = [
        float(mapped[:, 0].min()),
        float(mapped[:, 1].min()),
        float(mapped[:, 0].max()),
        float(mapped[:, 1].max()),
    ]
    return clip_box(out, orig_w, orig_h)


def box_iou_xyxy(a, b) -> float:
    ax1, ay1, ax2, ay2 = map(float, a)
    bx1, by1, bx2, by2 = map(float, b)

    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter

    return float(inter / union) if union > 0 else 0.0


def run_detector_view(
    gray_view: np.ndarray,
    *,
    view_name: str,
    inverse_affine,
    imgsz: int,
    conf: float,
    orig_gray: np.ndarray,
    orig_mask: np.ndarray,
):
    bgr = cv2.cvtColor(gray_view, cv2.COLOR_GRAY2BGR)

    result = detector.predict(
        source=bgr,
        imgsz=int(imgsz),
        conf=float(conf),
        iou=0.50,
        max_det=100,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False,
    )[0]

    candidates = []
    if result.boxes is None:
        return candidates

    orig_h, orig_w = orig_gray.shape

    for box in result.boxes:
        cid = int(box.cls.item())
        roi = norm_roi_class(result.names[cid])

        if roi not in ROI_ORDER:
            continue

        det_conf = float(box.conf.item())
        view_xyxy = box.xyxy[0].detach().cpu().numpy().tolist()

        if inverse_affine is None:
            orig_xyxy = clip_box(view_xyxy, orig_w, orig_h)
        else:
            orig_xyxy = map_box_affine_to_original(
                view_xyxy,
                inverse_affine,
                orig_w,
                orig_h,
            )

        if orig_xyxy is None:
            continue

        qc = qc_proposal(
            roi,
            det_conf,
            orig_xyxy,
            orig_gray,
            orig_mask,
        )

        candidates.append({
            "roi": roi,
            "conf": det_conf,
            "xyxy": orig_xyxy,
            "qc": qc,
            "view_name": view_name,
            "imgsz": int(imgsz),
        })

    return candidates


def unresolved_rois(candidates: list[dict]) -> list[str]:
    """
    Rescue if ROI is absent OR every proposal remains below automatic confidence.
    Geometry/foreground heuristics alone do not force more expensive inference.
    """
    by_roi = defaultdict(list)
    for item in candidates:
        by_roi[item["roi"]].append(item)

    unresolved = []
    for roi in ROI_ORDER:
        items = by_roi.get(roi, [])
        if not items:
            unresolved.append(roi)
            continue
        if max(float(x["conf"]) for x in items) < CFG.auto_accept_conf:
            unresolved.append(roi)

    return unresolved


def annotate_consensus(items: list[dict]) -> list[dict]:
    out = []
    for item in items:
        ious = []
        support_views = set()

        for other in items:
            if other is item:
                continue
            if other["view_name"] == item["view_name"]:
                continue

            iou = box_iou_xyxy(item["xyxy"], other["xyxy"])
            if iou >= CFG.candidate_consensus_iou:
                ious.append(iou)
                support_views.add(other["view_name"])

        z = dict(item)
        z["cross_view_support"] = int(len(support_views))
        z["consensus_mean_iou"] = float(np.mean(ious)) if ious else 0.0
        out.append(z)

    return out


def candidate_sort_key(item: dict):
    """
    Detector evidence comes first.
    QC heuristics are not allowed to cause a lower-confidence, geometry-normal
    false positive to outrank a stronger detector proposal.
    """
    raw_preference = int(str(item["view_name"]).startswith("RAW"))
    return (
        float(item["conf"]),
        int(item.get("cross_view_support", 0)),
        float(item.get("consensus_mean_iou", 0.0)),
        raw_preference,
    )


@torch.inference_mode()
def propose_case(patient_id: str, image_path: Path) -> dict:
    image_path = Path(image_path)

    if not image_path.is_file():
        raise FileNotFoundError(
            f"ROI proposal source image missing: patient={patient_id} path={image_path}"
        )

    stat = image_path.stat()
    gray = read_xray_u8(image_path)
    mask = hand_foreground_mask(gray)

    if gray.ndim != 2:
        raise RuntimeError(
            f"Internal invariant failed: grayscale image is not 2-D: "
            f"patient={patient_id}, path={image_path}, shape={gray.shape}"
        )

    h, w = gray.shape
    all_candidates = []

    # 1) Normal view.
    all_candidates.extend(
        run_detector_view(
            gray,
            view_name="RAW640_BASE",
            inverse_affine=None,
            imgsz=CFG.primary_inference_imgsz,
            conf=CFG.auto_conf_floor,
            orig_gray=gray,
            orig_mask=mask,
        )
    )

    # 2) Rescue low-confidence OR missing ROI.
    if unresolved_rois(all_candidates):
        all_candidates.extend(
            run_detector_view(
                gray,
                view_name="RAW960_RESCUE",
                inverse_affine=None,
                imgsz=CFG.rescue_inference_imgsz,
                conf=CFG.rescue_conf_floor,
                orig_gray=gray,
                orig_mask=mask,
            )
        )

    if CFG.enable_1280_rescue and unresolved_rois(all_candidates):
        all_candidates.extend(
            run_detector_view(
                gray,
                view_name="RAW1280_RESCUE",
                inverse_affine=None,
                imgsz=1280,
                conf=CFG.rescue_conf_floor,
                orig_gray=gray,
                orig_mask=mask,
            )
        )

    rotation_views_run = []
    if CFG.enable_rotation_rescue and unresolved_rois(all_candidates):
        for angle in CFG.rotation_rescue_degrees:
            rotated, invM = rotate_bound_with_inverse(gray, float(angle))
            rotation_views_run.append(float(angle))

            all_candidates.extend(
                run_detector_view(
                    rotated,
                    view_name=f"ROT{angle:+g}_960_RESCUE",
                    inverse_affine=invM,
                    imgsz=CFG.rescue_inference_imgsz,
                    conf=CFG.rescue_conf_floor,
                    orig_gray=gray,
                    orig_mask=mask,
                )
            )

            if not unresolved_rois(all_candidates):
                break

    by_roi = defaultdict(list)
    for item in all_candidates:
        by_roi[item["roi"]].append(item)

    best = {}

    for roi in ROI_ORDER:
        items = by_roi.get(roi, [])
        if not items:
            continue

        items = annotate_consensus(items)
        items = sorted(items, key=candidate_sort_key, reverse=True)
        chosen = items[0]

        best[roi] = {
            "xyxy": chosen["xyxy"],
            "conf": float(chosen["conf"]),
            "pass_name": chosen["view_name"],
            "view_name": chosen["view_name"],
            "imgsz": int(chosen["imgsz"]),
            "qc": chosen["qc"],
            "candidate_count": int(len(items)),
            "cross_view_support": int(chosen["cross_view_support"]),
            "consensus_mean_iou": float(chosen["consensus_mean_iou"]),
        }

    missing = [r for r in ROI_ORDER if r not in best]
    needs_review = [
        r for r, d in best.items()
        if d["qc"]["status"] != "ACCEPT"
    ]

    return {
        "patient_id": str(patient_id),
        "image_path": str(image_path.resolve()),
        "image_bytes": int(stat.st_size),
        "image_mtime_ns": int(stat.st_mtime_ns),
        "reader_version": XRAY_READER_VERSION,
        "roi_qc_code_version": ROI_QC_CODE_VERSION,
        "roi_proposal_code_version": ROI_PROPOSAL_CODE_VERSION,
        "geometry_priors_sha256": GEOMETRY_PRIORS_SHA256,
        "proposal_policy_hash": PROPOSAL_POLICY_HASH,
        "image_shape_hw": [int(h), int(w)],
        "hand_mask_fraction": float(mask.mean()),
        "detections": best,
        "missing_roi": missing,
        "needs_review_roi": needs_review,
        "auto_accept_9of9": (
            len(missing) == 0
            and len(needs_review) == 0
        ),
        "rotation_views_run": rotation_views_run,
        "unresolved_after_all_passes": unresolved_rois(all_candidates),
        "detector_sha256": FROZEN_HASH,
    }


stage_result(
    "STAGE_13_ROI_PROPOSAL",
    "PASS",
    (
        "Frozen detector proposal ready with R3 confidence-aware 640/960/1280/"
        "rotation rescue and cross-view consensus audit metadata. "
        "Low-confidence rescue never becomes automatic confidence."
    )
)


Proposal passes      : 8
Proposal policy hash : 4a53bb6a138c0eadc41c84ac3ab7610e6cae846cda069cd4bfe60e5410c02314
[STAGE_13_ROI_PROPOSAL] PASS: Frozen detector proposal ready with R3 confidence-aware 640/960/1280/rotation rescue and cross-view consensus audit metadata. Low-confidence rescue never becomes automatic confidence.


In [19]:
# Cell 19 — Age/sex-balanced DEV/VAL ROI pilot + complete visual audit + failure diagnostics
#
# PURPOSE:
# This is deliberately NOT the full 11,808-case DEV+VAL run.
# It samples CFG.roi_pilot_per_age cases from each bone-age bin 1..18.
# With the default 8/bin: 18 x 8 = 144 pilot cases.
# Each case expects 9 ROI => 144 x 9 = 1,296 ROI slots.
#
# SAFETY:
# - cache fingerprint includes detector, source image, QC code, geometry priors and thresholds
# - ALL non-ACCEPT ROI are prioritized
# - ALL 1,296 pilot ROI slots also receive a complete contact-sheet manifest
#   so false ACCEPT examples (e.g. MP5 actually being an L/R marker) can be found
# - this stage never auto-PASSes; Cell 20 requires explicit human approval

# Fail once with a complete dependency list instead of producing sequential NameErrors.
CELL19_REQUIRED_GLOBALS = [
    "GEOMETRY_PRIORS",
    "GEOMETRY_PRIOR_FILE",
    "GEOMETRY_PRIORS_SHA256",
    "FROZEN_DETECTOR",
    "FROZEN_HASH",
    "ROI_QC_CODE_VERSION",
    "ROI_PROPOSAL_CODE_VERSION",
    "PROPOSAL_PASSES",
    "PROPOSAL_POLICY_HASH",
    "propose_case",
]

cell19_missing_globals = [
    name for name in CELL19_REQUIRED_GLOBALS
    if name not in globals()
]

if cell19_missing_globals:
    stage_result(
        "STAGE_14_VISUAL_PILOT",
        "FAIL",
        (
            "Cell 19 dependencies are missing: "
            f"{cell19_missing_globals}. "
            "Rerun Cells 13, 16, 17 and 18 before Cell 19."
        ),
        {"missing_globals": cell19_missing_globals},
    )

if sha256_file(GEOMETRY_PRIOR_FILE) != GEOMETRY_PRIORS_SHA256:
    stage_result(
        "STAGE_14_VISUAL_PILOT",
        "FAIL",
        "Geometry-prior file changed after Cell 13. Rerun Cell 13 before ROI pilot."
    )

dev = get_v1_role("V1_DEV").assign(v1_role="V1_DEV")
val = get_v1_role("V1_VALIDATION").assign(v1_role="V1_VALIDATION")

def pilot_sample(frame: pd.DataFrame, per_age: int) -> pd.DataFrame:
    required = {
        "patient_id", "bone_age_year_bin", "male",
        "fullhand_image_path", "v1_role",
    }
    missing_cols = required - set(frame.columns)
    if missing_cols:
        raise RuntimeError(f"Pilot frame missing columns: {sorted(missing_cols)}")
    if per_age <= 0:
        raise ValueError("per_age must be > 0")
    if frame["patient_id"].astype(str).duplicated().any():
        raise RuntimeError("Pilot input contains duplicate patient IDs.")

    rows = []
    for age in range(1, 19):
        g = frame[frame["bone_age_year_bin"].eq(age)].copy()
        if len(g) == 0:
            continue

        g["_rank"] = g["patient_id"].map(
            lambda pid: stable_rank(str(pid), f"PILOT_A{age}")
        )
        target = min(per_age, len(g))
        base_quota = target // 2

        selected_parts = [
            g[g["male"].eq(sex)].sort_values("_rank").head(base_quota)
            for sex in [0, 1]
        ]
        selected = pd.concat(selected_parts, ignore_index=False)
        selected_ids = set(selected["patient_id"].astype(str))
        remaining_n = target - len(selected)

        if remaining_n > 0:
            fill = (
                g[~g["patient_id"].astype(str).isin(selected_ids)]
                .sort_values("_rank")
                .head(remaining_n)
            )
            selected = pd.concat([selected, fill], ignore_index=False)

        rows.append(selected.head(target))

    if not rows:
        raise RuntimeError("Pilot sampling produced zero rows.")

    out = (
        pd.concat(rows, ignore_index=True)
        .drop(columns=["_rank"], errors="ignore")
    )
    if out["patient_id"].astype(str).duplicated().any():
        raise RuntimeError("Pilot sampling produced duplicate patient IDs.")
    return out


pilot = pilot_sample(
    pd.concat([dev, val], ignore_index=True),
    per_age=CFG.roi_pilot_per_age,
)

PILOT_EXPECTED_CASES = sum(
    min(
        CFG.roi_pilot_per_age,
        int(pd.concat([dev, val], ignore_index=True)["bone_age_year_bin"].eq(age).sum())
    )
    for age in range(1, 19)
)
if len(pilot) != PILOT_EXPECTED_CASES:
    raise RuntimeError(
        f"Pilot cardinality mismatch: got={len(pilot)}, expected={PILOT_EXPECTED_CASES}"
    )

pilot_policy = {
    "reader_version": XRAY_READER_VERSION,
    "roi_qc_code_version": ROI_QC_CODE_VERSION,
    "roi_proposal_code_version": ROI_PROPOSAL_CODE_VERSION,
    "geometry_priors_sha256": GEOMETRY_PRIORS_SHA256,
    "detector_sha256": FROZEN_HASH,
    "proposal_policy_hash": PROPOSAL_POLICY_HASH,
    "proposal_passes": PROPOSAL_PASSES,
    "auto_conf_floor": CFG.auto_conf_floor,
    "auto_accept_conf": CFG.auto_accept_conf,
    "accept_geometry_z_max": CFG.accept_geometry_z_max,
    "reject_geometry_z_min": CFG.reject_geometry_z_min,
    "accept_hand_overlap_min": CFG.accept_hand_overlap_min,
    "reject_hand_overlap_below": CFG.reject_hand_overlap_below,
    "pilot_per_age": CFG.roi_pilot_per_age,
    "pilot_patient_ids": sorted(pilot["patient_id"].astype(str).tolist()),
}

PILOT_POLICY_HASH = sha256_text(
    json.dumps(pilot_policy, sort_keys=True, default=str)
)
PILOT_CACHE = (
    DIR["qc_reports"]
    / f"19_roi_pilot_{PILOT_POLICY_HASH[:10]}.jsonl"
)

def load_jsonl_by_patient(path: Path) -> dict:
    records = {}
    if not path.is_file():
        return records

    for line_no, line in enumerate(
        path.read_text(encoding="utf-8").splitlines(), start=1
    ):
        if not line.strip():
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError as e:
            raise RuntimeError(
                f"Corrupt JSONL cache: {path} line={line_no}: {e}"
            ) from e
        if "patient_id" not in rec:
            raise RuntimeError(
                f"Cache record missing patient_id: {path} line={line_no}"
            )
        records[str(rec["patient_id"])] = rec
    return records

def image_record_matches(
    rec: dict,
    image_path: Path,
    *,
    expected_detector_hash: str = FROZEN_HASH,
) -> bool:
    p = Path(image_path)
    if not p.is_file():
        return False

    st = p.stat()
    try:
        rec_path = str(Path(str(rec.get("image_path", ""))).resolve())
        current_path = str(p.resolve())
    except Exception as e:
        warnings.warn(
            f"Cache path resolution failed for {p}: {type(e).__name__}: {e}"
        )
        return False

    return (
        rec.get("detector_sha256") == expected_detector_hash
        and rec.get("reader_version") == XRAY_READER_VERSION
        and rec.get("roi_qc_code_version") == ROI_QC_CODE_VERSION
        and rec.get("roi_proposal_code_version") == ROI_PROPOSAL_CODE_VERSION
        and rec.get("geometry_priors_sha256") == GEOMETRY_PRIORS_SHA256
        and rec.get("proposal_policy_hash") == PROPOSAL_POLICY_HASH
        and rec_path == current_path
        and int(rec.get("image_bytes", -1)) == int(st.st_size)
        and int(rec.get("image_mtime_ns", -1)) == int(st.st_mtime_ns)
    )

existing = load_jsonl_by_patient(PILOT_CACHE)

for _, r in tqdm(
    pilot.iterrows(),
    total=len(pilot),
    desc="DEV/VAL ROI pilot"
):
    pid = str(r["patient_id"])
    image_path = Path(r["fullhand_image_path"])
    rec = existing.get(pid)

    cache_ok = (
        rec is not None
        and rec.get("pilot_policy_hash") == PILOT_POLICY_HASH
        and image_record_matches(rec, image_path)
    )
    if cache_ok:
        continue

    rec = propose_case(pid, image_path)
    rec["pilot_policy_hash"] = PILOT_POLICY_HASH
    rec["v1_role"] = str(r["v1_role"])
    append_jsonl(PILOT_CACHE, rec)
    existing[pid] = rec

pilot_records = [
    existing[str(pid)]
    for pid in pilot["patient_id"].astype(str)
]

pilot_meta = (
    pilot.assign(patient_id=pilot["patient_id"].astype(str))
    .set_index("patient_id")[["v1_role", "bone_age_year_bin", "male"]]
    .to_dict(orient="index")
)

pilot_summary_rows = []
for rec in pilot_records:
    pid = str(rec["patient_id"])
    meta = pilot_meta[pid]

    for roi in ROI_ORDER:
        d = rec["detections"].get(roi)
        if d is None:
            pilot_summary_rows.append({
                "patient_id": pid,
                "v1_role": meta["v1_role"],
                "bone_age_year_bin": int(meta["bone_age_year_bin"]),
                "male": int(meta["male"]),
                "roi": roi,
                "status": "MISSING",
                "reason": "NO_PROPOSAL",
                "conf": np.nan,
                "geometry_z": np.nan,
                "hand_overlap": np.nan,
                "pass_name": "",
                "view_name": "",
                "imgsz": np.nan,
                "cross_view_support": np.nan,
                "consensus_mean_iou": np.nan,
            })
        else:
            q = d["qc"]
            pilot_summary_rows.append({
                "patient_id": pid,
                "v1_role": meta["v1_role"],
                "bone_age_year_bin": int(meta["bone_age_year_bin"]),
                "male": int(meta["male"]),
                "roi": roi,
                "status": q["status"],
                "reason": q["reason"],
                "conf": d["conf"],
                "geometry_z": q["geometry_z"],
                "hand_overlap": q["hand_overlap"],
                "pass_name": d["pass_name"],
                "view_name": d.get("view_name", d["pass_name"]),
                "imgsz": d["imgsz"],
                "cross_view_support": d.get("cross_view_support", 0),
                "consensus_mean_iou": d.get("consensus_mean_iou", 0.0),
            })

pilot_qc = pd.DataFrame(pilot_summary_rows)

PILOT_EXPECTED_ROI_SLOTS = len(pilot) * len(ROI_ORDER)
if len(pilot_qc) != PILOT_EXPECTED_ROI_SLOTS:
    raise RuntimeError(
        f"Pilot ROI-slot mismatch: got={len(pilot_qc)}, "
        f"expected={len(pilot)}*{len(ROI_ORDER)}={PILOT_EXPECTED_ROI_SLOTS}"
    )
if pilot_qc.duplicated(["patient_id", "roi"]).any():
    raise RuntimeError("Duplicate patient_id/ROI rows in pilot QC.")
if not pilot_qc.groupby("patient_id")["roi"].nunique().eq(len(ROI_ORDER)).all():
    raise RuntimeError("At least one pilot patient does not have exactly nine ROI audit slots.")

allowed_status = {"ACCEPT", "MISSING", "NEEDS_REVIEW", "REJECT"}
unexpected_status = set(pilot_qc["status"].astype(str)) - allowed_status
if unexpected_status:
    raise RuntimeError(f"Unexpected pilot QC statuses: {sorted(unexpected_status)}")

PILOT_QC_CSV = (
    DIR["qc_reports"]
    / f"19_roi_pilot_qc_{PILOT_POLICY_HASH[:10]}.csv"
)
pilot_qc.to_csv(PILOT_QC_CSV, index=False, encoding="utf-8-sig")

status_counts = pilot_qc["status"].value_counts(dropna=False)
nonaccept_n = int((pilot_qc["status"] != "ACCEPT").sum())
nonaccept_rate = nonaccept_n / len(pilot_qc)

print("Pilot cases             :", len(pilot))
print("ROI classes/case        :", len(ROI_ORDER))
print("Expected ROI audit slots:", PILOT_EXPECTED_ROI_SLOTS)
print("Actual ROI audit rows   :", len(pilot_qc))
print(status_counts)
print(f"Pilot non-ACCEPT        : {nonaccept_n}/{len(pilot_qc)} = {nonaccept_rate:.2%}")
print(
    "NOTE: this pilot is deliberately age/sex-balanced, so this rate is NOT "
    "a prevalence estimate for the full DEV/VAL cohort."
)

display(
    pilot_qc.groupby(["roi", "status"])
    .size().unstack(fill_value=0)
    .reindex(ROI_ORDER)
)

print("\nPilot sex counts by bone-age bin:")
display(
    pilot.groupby(["bone_age_year_bin", "male"])
    .size().unstack(fill_value=0)
)

# Root-cause diagnostics: failures by age, ROI and rescue view.
pilot_qc["_is_missing"] = pilot_qc["status"].eq("MISSING")
pilot_qc["_is_nonaccept"] = pilot_qc["status"].ne("ACCEPT")

age_diag = (
    pilot_qc.groupby("bone_age_year_bin")
    .agg(
        roi_slots=("roi", "size"),
        missing_n=("_is_missing", "sum"),
        nonaccept_n=("_is_nonaccept", "sum"),
    )
)
age_diag["missing_rate"] = age_diag["missing_n"] / age_diag["roi_slots"]
age_diag["nonaccept_rate"] = age_diag["nonaccept_n"] / age_diag["roi_slots"]

roi_diag = (
    pilot_qc.groupby("roi")
    .agg(
        roi_slots=("patient_id", "size"),
        missing_n=("_is_missing", "sum"),
        nonaccept_n=("_is_nonaccept", "sum"),
    )
)
roi_diag["missing_rate"] = roi_diag["missing_n"] / roi_diag["roi_slots"]
roi_diag["nonaccept_rate"] = roi_diag["nonaccept_n"] / roi_diag["roi_slots"]

case_diag = (
    pilot_qc.groupby("patient_id")["status"]
    .apply(lambda s: pd.Series({
        "accept_n": int((s == "ACCEPT").sum()),
        "missing_n": int((s == "MISSING").sum()),
        "needs_review_n": int((s == "NEEDS_REVIEW").sum()),
        "reject_n": int((s == "REJECT").sum()),
    }))
    .unstack()
)

print("\nFailure diagnostics by bone-age bin:")
display(age_diag)

print("\nFailure diagnostics by ROI:")
display(roi_diag)

print("\nCase-level pilot summary:")
print("9/9 ACCEPT cases :", int((case_diag["accept_n"] == len(ROI_ORDER)).sum()))
print("9/9 MISSING cases:", int((case_diag["missing_n"] == len(ROI_ORDER)).sum()))
print("Cases with >=5 non-ACCEPT ROI:",
      int(((len(ROI_ORDER) - case_diag["accept_n"]) >= 5).sum()))

print("\nProposal view usage:")
display(
    pilot_qc[pilot_qc["status"].ne("MISSING")]
    ["view_name"]
    .fillna("UNKNOWN")
    .value_counts()
    .to_frame("roi_n")
)

# Rough operational projection to the actual DEV+VAL age/sex distribution.
# This is NOT a performance estimate: each pilot stratum contains only four
# patients per sex/age and therefore has wide uncertainty.
pilot_strata = (
    pilot_qc.assign(
        is_missing=pilot_qc["status"].eq("MISSING"),
        is_nonaccept=pilot_qc["status"].ne("ACCEPT"),
    )
    .groupby(["bone_age_year_bin", "male"])
    .agg(
        pilot_roi_slots=("roi", "size"),
        missing_rate=("is_missing", "mean"),
        nonaccept_rate=("is_nonaccept", "mean"),
    )
    .reset_index()
)

devval_distribution = (
    pd.concat([dev, val], ignore_index=True)
    .groupby(["bone_age_year_bin", "male"])
    .size()
    .rename("cases")
    .reset_index()
)

projection = devval_distribution.merge(
    pilot_strata,
    on=["bone_age_year_bin", "male"],
    how="left",
    validate="one_to_one",
)

if projection[["missing_rate", "nonaccept_rate"]].isna().any().any():
    warnings.warn(
        "Some DEV+VAL age/sex strata are absent from the pilot; weighted workload "
        "projection is incomplete."
    )
else:
    projection["roi_slots"] = projection["cases"] * len(ROI_ORDER)
    projection["projected_missing_roi"] = (
        projection["roi_slots"] * projection["missing_rate"]
    )
    projection["projected_nonaccept_roi"] = (
        projection["roi_slots"] * projection["nonaccept_rate"]
    )

    projected_slots = int(projection["roi_slots"].sum())
    projected_missing = float(projection["projected_missing_roi"].sum())
    projected_nonaccept = float(projection["projected_nonaccept_roi"].sum())

    PILOT_WEIGHTED_PROJECTION = {
        "devval_cases": int(projection["cases"].sum()),
        "devval_roi_slots": projected_slots,
        "projected_missing_roi": projected_missing,
        "projected_missing_rate": projected_missing / projected_slots,
        "projected_nonaccept_roi": projected_nonaccept,
        "projected_nonaccept_rate": projected_nonaccept / projected_slots,
        "warning": (
            "Rough workload projection only; pilot has 4 patients per sex/age stratum."
        ),
    }

    print("\nRough age/sex-weighted DEV+VAL workload projection:")
    print(
        f"Projected MISSING   ≈ {projected_missing:,.0f}/{projected_slots:,} "
        f"({projected_missing/projected_slots:.2%})"
    )
    print(
        f"Projected non-ACCEPT≈ {projected_nonaccept:,.0f}/{projected_slots:,} "
        f"({projected_nonaccept/projected_slots:.2%})"
    )
    print("WARNING: workload projection only; not a calibrated prevalence/performance estimate.")

# Do not let stale visual folders be mistaken for the current rule.
print(
    "\\nIMPORTANT: Only review contact-sheet folders whose suffix matches CURRENT "
    f"pilot hash {PILOT_POLICY_HASH[:10]}. Older hash folders are historical artifacts."
)

pilot_qc.drop(columns=["_is_missing", "_is_nonaccept"], inplace=True)

def crop_for_display(rec: dict, roi: str, pad: float = 0.10):
    p = Path(rec["image_path"])
    gray = read_xray_u8(p)
    d = rec["detections"].get(roi)

    if d is None:
        # Show the full hand rather than an empty black tile.
        return gray, "MISSING: NO_PROPOSAL"

    x1, y1, x2, y2 = d["xyxy"]
    bw, bh = x2 - x1, y2 - y1
    c = clip_box(
        [x1 - bw*pad, y1 - bh*pad, x2 + bw*pad, y2 + bh*pad],
        gray.shape[1], gray.shape[0]
    )
    if c is None:
        return gray, "INVALID_BOX"

    x1, y1, x2, y2 = map(lambda v: int(round(v)), c)
    crop = gray[y1:y2, x1:x2]
    if crop.size == 0:
        return gray, "EMPTY_CROP"

    return crop, (
        f"{d['qc']['status']} {d['conf']:.2f} {d['qc']['reason']}"
    )

def save_contact_sheets(
    pair_frame: pd.DataFrame,
    out_dir: Path,
    manifest_path: Path,
    *,
    batch_size: int = 20,
) -> pd.DataFrame:
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []

    pairs = list(
        pair_frame[["patient_id", "roi", "review_kind"]]
        .itertuples(index=False, name=None)
    )

    for batch_i in range(0, len(pairs), batch_size):
        batch = pairs[batch_i:batch_i + batch_size]
        fig, axes = plt.subplots(4, 5, figsize=(15, 10))
        axes = axes.ravel()
        for ax in axes:
            ax.axis("off")

        for ax, (pid, roi, review_kind) in zip(axes, batch):
            rec = existing[str(pid)]
            crop, label = crop_for_display(rec, roi)
            ax.imshow(crop, cmap="gray")
            ax.set_title(
                f"{pid} | {roi}\n{review_kind}\n{label}",
                fontsize=7
            )
            ax.axis("off")
            rows.append({
                "patient_id": str(pid),
                "roi": str(roi),
                "review_kind": str(review_kind),
                "label": str(label),
                "sheet_batch": int(batch_i // batch_size),
            })

        fig.tight_layout()
        out = out_dir / f"pilot_review_{batch_i//batch_size:03d}.png"
        fig.savefig(out, dpi=150)
        plt.close(fig)

    manifest = pd.DataFrame(rows)
    manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
    return manifest

# Priority sheets: every non-ACCEPT + deterministic ACCEPT sample.
nonaccept_pairs = (
    pilot_qc[pilot_qc["status"].ne("ACCEPT")]
    [["patient_id", "roi", "status"]]
    .drop_duplicates()
    .rename(columns={"status": "review_kind"})
)
nonaccept_pairs["review_kind"] = (
    "NON_ACCEPT_" + nonaccept_pairs["review_kind"].astype(str)
)

accepted_pool = (
    pilot_qc[pilot_qc["status"].eq("ACCEPT")]
    [["patient_id", "roi"]]
    .drop_duplicates()
    .copy()
)
if len(accepted_pool):
    accepted_pool["_audit_rank"] = [
        stable_rank(f"{pid}|{roi}", "PILOT_ACCEPT_AUDIT")
        for pid, roi in zip(
            accepted_pool["patient_id"].astype(str),
            accepted_pool["roi"].astype(str),
        )
    ]
    accepted_pairs = (
        accepted_pool.sort_values(["roi", "_audit_rank"])
        .groupby("roi", as_index=False)
        .head(CFG.roi_pilot_accept_audit_per_roi)
        [["patient_id", "roi"]]
        .copy()
    )
    accepted_pairs["review_kind"] = "ACCEPT_AUDIT"
else:
    accepted_pairs = pd.DataFrame(
        columns=["patient_id", "roi", "review_kind"]
    )

priority_pairs = (
    pd.concat([nonaccept_pairs, accepted_pairs], ignore_index=True)
    .drop_duplicates(["patient_id", "roi"], keep="first")
)

PILOT_PRIORITY_SHEET_DIR = (
    DIR["qc_review"]
    / f"pilot_contact_sheets_priority_{PILOT_POLICY_HASH[:10]}"
)
PILOT_PRIORITY_MANIFEST = (
    DIR["qc_reports"]
    / f"19_pilot_priority_review_manifest_{PILOT_POLICY_HASH[:10]}.csv"
)
priority_manifest = save_contact_sheets(
    priority_pairs,
    PILOT_PRIORITY_SHEET_DIR,
    PILOT_PRIORITY_MANIFEST,
)

# Complete pilot sheets: all 1,296 ROI slots (default settings).
all_pairs = pilot_qc[["patient_id", "roi", "status"]].copy()
all_pairs["review_kind"] = "ALL_" + all_pairs["status"].astype(str)
all_pairs = all_pairs[["patient_id", "roi", "review_kind"]]

PILOT_ALL_SHEET_DIR = (
    DIR["qc_review"]
    / f"pilot_contact_sheets_ALL_{PILOT_POLICY_HASH[:10]}"
)
PILOT_ALL_MANIFEST = (
    DIR["qc_reports"]
    / f"19_pilot_ALL_review_manifest_{PILOT_POLICY_HASH[:10]}.csv"
)
all_manifest = save_contact_sheets(
    all_pairs,
    PILOT_ALL_SHEET_DIR,
    PILOT_ALL_MANIFEST,
)

if len(all_manifest) != PILOT_EXPECTED_ROI_SLOTS:
    raise RuntimeError(
        f"Complete pilot visual manifest has {len(all_manifest)} rows; "
        f"expected {PILOT_EXPECTED_ROI_SLOTS}."
    )

register_artifact(
    "STAGE_14", PILOT_QC_CSV, "roi_pilot_qc",
    {"pilot_policy_hash": PILOT_POLICY_HASH}
)
register_artifact(
    "STAGE_14", PILOT_PRIORITY_MANIFEST, "roi_pilot_priority_visual_manifest",
    {"pilot_policy_hash": PILOT_POLICY_HASH}
)
register_artifact(
    "STAGE_14", PILOT_ALL_MANIFEST, "roi_pilot_complete_visual_manifest",
    {"pilot_policy_hash": PILOT_POLICY_HASH}
)

stage_result(
    "STAGE_14_VISUAL_PILOT",
    "WARNING",
    (
        "Age/sex-balanced DEV/Validation pilot generated. "
        "Before rule freeze, inspect the COMPLETE pilot contact sheets, not only counts. "
        "Pay special attention to false ACCEPT examples such as L/R/text being labeled as MP5. "
        "This stage cannot become PASS automatically."
    ),
    {
        "pilot_cases": int(len(pilot)),
        "roi_slots": int(PILOT_EXPECTED_ROI_SLOTS),
        "policy_hash": PILOT_POLICY_HASH,
        "accept_n": int((pilot_qc["status"] == "ACCEPT").sum()),
        "nonaccept_n": int(nonaccept_n),
        "nonaccept_rate_age_sex_balanced": float(nonaccept_rate),
        "priority_manifest_n": int(len(priority_manifest)),
        "complete_manifest_n": int(len(all_manifest)),
        "priority_sheets": str(PILOT_PRIORITY_SHEET_DIR),
        "complete_sheets": str(PILOT_ALL_SHEET_DIR),
    }
)

print("Priority review sheets :", PILOT_PRIORITY_SHEET_DIR)
print("Complete pilot sheets :", PILOT_ALL_SHEET_DIR)
print("Pilot QC CSV          :", PILOT_QC_CSV)
print("Priority manifest     :", PILOT_PRIORITY_MANIFEST)
print("Complete manifest     :", PILOT_ALL_MANIFEST)


DEV/VAL ROI pilot: 100%|██████████| 144/144 [00:41<00:00,  3.45it/s]

Pilot cases             : 144
ROI classes/case        : 9
Expected ROI audit slots: 1296
Actual ROI audit rows   : 1296
status
ACCEPT          1036
NEEDS_REVIEW     186
MISSING           74
Name: count, dtype: int64
Pilot non-ACCEPT        : 260/1296 = 20.06%
NOTE: this pilot is deliberately age/sex-balanced, so this rate is NOT a prevalence estimate for the full DEV/VAL cohort.


status,ACCEPT,MISSING,NEEDS_REVIEW
roi,,,
Carpal,128,6,10
DP3,103,11,30
MC1,95,24,25
MP3,115,2,27
MP5,115,7,22
PP1,109,10,25
PP3,130,0,14
Radius,124,8,12
Ulna,117,6,21



Pilot sex counts by bone-age bin:


male,0,1
bone_age_year_bin,,
1,4,4
2,4,4
3,4,4
4,4,4
5,4,4
6,4,4
7,4,4
8,4,4
9,4,4



Failure diagnostics by bone-age bin:


... (2 outputs left out)

,roi_slots,missing_n,nonaccept_n,missing_rate,nonaccept_rate
roi,,,,,
Carpal,144,6,16,0.041667,0.111111
DP3,144,11,41,0.076389,0.284722
MC1,144,24,49,0.166667,0.340278
MP3,144,2,29,0.013889,0.201389
MP5,144,7,29,0.048611,0.201389
PP1,144,10,35,0.069444,0.243056
PP3,144,0,14,0.000000,0.097222
Radius,144,8,20,0.055556,0.138889
Ulna,144,6,27,0.041667,0.187500



Case-level pilot summary:
9/9 ACCEPT cases : 79
9/9 MISSING cases: 0
Cases with >=5 non-ACCEPT ROI: 22

Proposal view usage:


,roi_n
view_name,
RAW640_BASE,742
RAW960_RESCUE,173
ROT+45_960_RESCUE,83
ROT-45_960_RESCUE,60
ROT-90_960_RESCUE,57
ROT+90_960_RESCUE,56
RAW1280_RESCUE,32
ROT+180_960_RESCUE,19



Rough age/sex-weighted DEV+VAL workload projection:
Projected MISSING   ≈ 3,835/106,272 (3.61%)
Projected non-ACCEPT≈ 18,962/106,272 (17.84%)
\nIMPORTANT: Only review contact-sheet folders whose suffix matches CURRENT pilot hash abb788a420. Older hash folders are historical artifacts.
[STAGE_14_VISUAL_PILOT] WARNING: Age/sex-balanced DEV/Validation pilot generated. Before rule freeze, inspect the COMPLETE pilot contact sheets, not only counts. Pay special attention to false ACCEPT examples such as L/R/text being labeled as MP5. This stage cannot become PASS automatically.
Priority review sheets : D:\workspace\Bone_age\BoneAge_LOCAL4050_V1\qc\needs_review\pilot_contact_sheets_priority_abb788a420
Complete pilot sheets : D:\workspace\Bone_age\BoneAge_LOCAL4050_V1\qc\needs_review\pilot_contact_sheets_ALL_abb788a420
Pilot QC CSV          : D:\workspace\Bone_age\BoneAge_LOCAL4050_V1\qc\reports\19_roi_pilot_qc_abb788a420.csv
Priority manifest     : D:\workspace\Bone_age\BoneAge_LOCAL4050_V1\

In [20]:

# Cell 20 — PRE-FREEZE quality gate + explicit human approval
#
# R3 behavior:
# - malformed/missing artifacts are true FAIL and raise
# - expected quality/procedure holds are BLOCKED (no Python traceback)
# - one small pilot age-bin does not use a brittle 30.00% vs 30.56% binary cliff
# - known warnings require a second explicit acknowledgement before freezing
# - no thresholds are changed merely to make this run pass

required_vars = [
    "PILOT_POLICY_HASH",
    "PILOT_QC_CSV",
    "PILOT_ALL_MANIFEST",
    "PILOT_ALL_SHEET_DIR",
    "pilot_qc",
    "pilot",
    "PROPOSAL_POLICY_HASH",
]
missing_vars = [name for name in required_vars if name not in globals()]

if missing_vars:
    stage_result(
        "STAGE_15_ROI_RULE_FREEZE",
        "FAIL",
        f"Cell 19 outputs missing from memory: {missing_vars}. Rerun Cell 19."
    )

for required_file in [PILOT_QC_CSV, PILOT_ALL_MANIFEST]:
    if not Path(required_file).is_file():
        stage_result(
            "STAGE_15_ROI_RULE_FREEZE",
            "FAIL",
            f"Required pilot artifact missing: {required_file}"
        )

if not Path(PILOT_ALL_SHEET_DIR).is_dir():
    stage_result(
        "STAGE_15_ROI_RULE_FREEZE",
        "FAIL",
        f"Complete pilot contact-sheet directory missing: {PILOT_ALL_SHEET_DIR}"
    )

expected_slots = len(pilot) * len(ROI_ORDER)

if len(pilot_qc) != expected_slots:
    stage_result(
        "STAGE_15_ROI_RULE_FREEZE",
        "FAIL",
        f"Pilot QC rows={len(pilot_qc)} but expected={expected_slots}."
    )

if pilot_qc.duplicated(["patient_id", "roi"]).any():
    stage_result(
        "STAGE_15_ROI_RULE_FREEZE",
        "FAIL",
        "Duplicate patient_id/ROI rows exist in pilot QC."
    )

per_patient_roi_n = pilot_qc.groupby("patient_id")["roi"].nunique()
if not per_patient_roi_n.eq(len(ROI_ORDER)).all():
    stage_result(
        "STAGE_15_ROI_RULE_FREEZE",
        "FAIL",
        "At least one pilot patient is missing an ROI audit slot."
    )

all_manifest = pd.read_csv(
    PILOT_ALL_MANIFEST,
    dtype={"patient_id": str},
)

if len(all_manifest) != expected_slots:
    stage_result(
        "STAGE_15_ROI_RULE_FREEZE",
        "FAIL",
        f"Complete visual manifest rows={len(all_manifest)} but expected={expected_slots}."
    )

manifest_pairs = set(zip(
    all_manifest["patient_id"].astype(str),
    all_manifest["roi"].astype(str),
))
qc_pairs = set(zip(
    pilot_qc["patient_id"].astype(str),
    pilot_qc["roi"].astype(str),
))

if manifest_pairs != qc_pairs:
    stage_result(
        "STAGE_15_ROI_RULE_FREEZE",
        "FAIL",
        "Complete visual manifest patient/ROI pairs do not exactly match pilot QC."
    )

sheet_files = sorted(Path(PILOT_ALL_SHEET_DIR).glob("pilot_review_*.png"))
expected_sheet_n = math.ceil(expected_slots / 20)

if len(sheet_files) != expected_sheet_n:
    stage_result(
        "STAGE_15_ROI_RULE_FREEZE",
        "FAIL",
        f"Complete pilot sheets={len(sheet_files)} but expected={expected_sheet_n}."
    )

pilot_status_counts = (
    pilot_qc["status"]
    .value_counts()
    .reindex(["ACCEPT", "MISSING", "NEEDS_REVIEW", "REJECT"], fill_value=0)
)

pilot_missing_rate = float((pilot_qc["status"] == "MISSING").mean())
pilot_nonaccept_rate = float((pilot_qc["status"] != "ACCEPT").mean())

age_missing = (
    pilot_qc.assign(is_missing=pilot_qc["status"].eq("MISSING"))
    .groupby("bone_age_year_bin")["is_missing"]
    .mean()
)

roi_missing = (
    pilot_qc.assign(is_missing=pilot_qc["status"].eq("MISSING"))
    .groupby("roi")["is_missing"]
    .mean()
)

case_missing = (
    pilot_qc.assign(is_missing=pilot_qc["status"].eq("MISSING"))
    .groupby("patient_id")["is_missing"]
    .sum()
)

heuristic_reject = pilot_qc[
    pilot_qc["status"].eq("REJECT")
    & pilot_qc["reason"].astype(str).str.contains(
        "FOREGROUND|HAND_OVERLAP|GEOMETRY|MARKER_RISK",
        regex=True,
        na=False,
    )
]

hard_blockers = []
quality_warnings = []

if pilot_missing_rate > CFG.pilot_max_missing_rate:
    hard_blockers.append(
        f"overall missing {pilot_missing_rate:.2%} > {CFG.pilot_max_missing_rate:.2%}"
    )

if len(roi_missing) and float(roi_missing.max()) > CFG.pilot_max_any_roi_missing_rate:
    hard_blockers.append(
        f"ROI {roi_missing.idxmax()} missing {float(roi_missing.max()):.2%} "
        f"> {CFG.pilot_max_any_roi_missing_rate:.2%}"
    )

if len(age_missing) and float(age_missing.max()) >= CFG.pilot_agebin_hard_missing_rate:
    hard_blockers.append(
        f"age-bin {int(age_missing.idxmax())} missing {float(age_missing.max()):.2%} "
        f">= hard limit {CFG.pilot_agebin_hard_missing_rate:.2%}"
    )

if int(case_missing.max()) == len(ROI_ORDER):
    hard_blockers.append("at least one pilot patient is 9/9 MISSING")

if len(heuristic_reject):
    hard_blockers.append(
        f"{len(heuristic_reject)} ROI auto-REJECTED by heuristic anatomy/marker logic"
    )

# Warning tier: these are meaningful but should not be converted into a fake runtime error.
warn_age = age_missing[age_missing >= CFG.pilot_agebin_warning_missing_rate]
if len(warn_age):
    quality_warnings.append(
        "young/age-domain instability: "
        + ", ".join(
            f"age {int(age)}={rate:.2%}"
            for age, rate in warn_age.sort_values(ascending=False).items()
        )
    )

if pilot_nonaccept_rate >= CFG.pilot_nonaccept_warning_rate:
    quality_warnings.append(
        f"pilot non-ACCEPT rate is {pilot_nonaccept_rate:.2%} "
        f"(warning threshold {CFG.pilot_nonaccept_warning_rate:.2%})"
    )

# Detector class weakness from strict VAL is a warning, not an automatic anatomy claim.
weak_detector_classes = []
if "per_class_df" in globals() and len(per_class_df):
    weak = per_class_df[
        per_class_df["val_map50_95"].astype(float) < 0.60
    ]
    weak_detector_classes = [
        {
            "roi": str(r["roi"]),
            "val_map50_95": float(r["val_map50_95"]),
        }
        for _, r in weak.iterrows()
    ]
    if weak_detector_classes:
        quality_warnings.append(
            "weak strict-VAL detector class(es): "
            + ", ".join(
                f"{x['roi']}={x['val_map50_95']:.3f}"
                for x in weak_detector_classes
            )
        )

weighted_projection = (
    PILOT_WEIGHTED_PROJECTION
    if "PILOT_WEIGHTED_PROJECTION" in globals()
    else None
)

prefreeze_report = {
    "pilot_policy_hash": PILOT_POLICY_HASH,
    "proposal_policy_hash": PROPOSAL_POLICY_HASH,
    "pilot_cases": int(len(pilot)),
    "pilot_roi_slots": int(expected_slots),
    "status_counts": {k: int(v) for k, v in pilot_status_counts.items()},
    "pilot_missing_rate": pilot_missing_rate,
    "pilot_nonaccept_rate": pilot_nonaccept_rate,
    "max_agebin_missing_rate": float(age_missing.max()) if len(age_missing) else None,
    "worst_agebin": int(age_missing.idxmax()) if len(age_missing) else None,
    "max_roi_missing_rate": float(roi_missing.max()) if len(roi_missing) else None,
    "worst_roi": str(roi_missing.idxmax()) if len(roi_missing) else None,
    "hard_blockers": hard_blockers,
    "quality_warnings": quality_warnings,
    "weak_detector_classes": weak_detector_classes,
    "weighted_devval_projection": weighted_projection,
    "manual_pilot_approval": bool(CFG.roi_rule_manual_approval),
    "known_warnings_acknowledged": bool(CFG.roi_rule_accept_known_warnings),
}

PREFREEZE_REPORT_FILE = (
    DIR["qc_reports"]
    / f"20_prefreeze_audit_{PILOT_POLICY_HASH[:10]}.json"
)
PREFREEZE_REPORT_FILE.write_text(
    json.dumps(prefreeze_report, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)

register_artifact(
    "STAGE_15",
    PREFREEZE_REPORT_FILE,
    "roi_rule_prefreeze_audit",
)

print("=" * 92)
print("CELL 20 — R3 PRE-FREEZE AUDIT")
print("=" * 92)
print("Pilot cases      :", len(pilot))
print("ROI slots        :", expected_slots)
print("ACCEPT           :", int(pilot_status_counts["ACCEPT"]))
print("MISSING          :", int(pilot_status_counts["MISSING"]))
print("NEEDS_REVIEW     :", int(pilot_status_counts["NEEDS_REVIEW"]))
print("REJECT           :", int(pilot_status_counts["REJECT"]))
print(f"Missing rate     : {pilot_missing_rate:.2%}")
print(f"Non-ACCEPT rate  : {pilot_nonaccept_rate:.2%}")
print("Worst age-bin    :", prefreeze_report["worst_agebin"],
      f"{prefreeze_report['max_agebin_missing_rate']:.2%}")
print("Worst ROI        :", prefreeze_report["worst_roi"],
      f"{prefreeze_report['max_roi_missing_rate']:.2%}")
print("Hard blockers    :", hard_blockers if hard_blockers else "NONE")
print("Quality warnings :", quality_warnings if quality_warnings else "NONE")
print("Complete sheets  :", PILOT_ALL_SHEET_DIR)

ROI_RULE_FROZEN = False

if hard_blockers:
    stage_result(
        "STAGE_15_ROI_RULE_FREEZE",
        "BLOCKED",
        (
            "ROI rule is structurally not ready to freeze. "
            "No exception is raised because this is an expected quality gate, not a code error. "
            + " | ".join(hard_blockers)
        ),
        prefreeze_report,
    )

elif not CFG.roi_rule_manual_approval:
    stage_result(
        "STAGE_15_ROI_RULE_FREEZE",
        "BLOCKED",
        (
            f"Inspect ALL {len(sheet_files)} CURRENT-HASH contact sheets in "
            f"{PILOT_ALL_SHEET_DIR}. Confirm each ROI identity and especially "
            "L/R/text/border/marker false accepts. Then set "
            "roi_rule_manual_approval=True."
        ),
        prefreeze_report,
    )

elif quality_warnings and not CFG.roi_rule_accept_known_warnings:
    stage_result(
        "STAGE_15_ROI_RULE_FREEZE",
        "BLOCKED",
        (
            "Manual visual approval is present, but known quality warnings remain. "
            "For a high-reliability workflow, improve/retrain the detector first. "
            "If you deliberately accept these warnings after review, set "
            "roi_rule_accept_known_warnings=True."
        ),
        prefreeze_report,
    )

else:
    rule_payload = {
        "version": CFG.version,
        "detector_path": str(FROZEN_DETECTOR.resolve()),
        "detector_sha256": FROZEN_HASH,
        "train_config_hash": TRAIN_CONFIG_HASH,
        "reader_version": XRAY_READER_VERSION,
        "roi_qc_code_version": ROI_QC_CODE_VERSION,
        "roi_proposal_code_version": ROI_PROPOSAL_CODE_VERSION,
        "geometry_priors_sha256": GEOMETRY_PRIORS_SHA256,
        "proposal_policy_hash": PROPOSAL_POLICY_HASH,
        "proposal_passes": PROPOSAL_PASSES,
        "auto_conf_floor": CFG.auto_conf_floor,
        "rescue_conf_floor": CFG.rescue_conf_floor,
        "auto_accept_conf": CFG.auto_accept_conf,
        "candidate_consensus_iou": CFG.candidate_consensus_iou,
        "accept_geometry_z_max": CFG.accept_geometry_z_max,
        "reject_geometry_z_min": CFG.reject_geometry_z_min,
        "accept_hand_overlap_min": CFG.accept_hand_overlap_min,
        "reject_hand_overlap_below": CFG.reject_hand_overlap_below,
        "enable_1280_rescue": CFG.enable_1280_rescue,
        "enable_rotation_rescue": CFG.enable_rotation_rescue,
        "rotation_rescue_degrees": list(CFG.rotation_rescue_degrees),
        "pilot_policy_hash": PILOT_POLICY_HASH,
        "pilot_qc_sha256": sha256_file(PILOT_QC_CSV),
        "pilot_complete_visual_manifest_sha256": sha256_file(PILOT_ALL_MANIFEST),
        "pilot_cases": int(len(pilot)),
        "pilot_roi_slots": int(expected_slots),
        "pilot_status_counts": {k: int(v) for k, v in pilot_status_counts.items()},
        "quality_warnings_accepted": quality_warnings,
        "manual_pilot_approval": True,
        "v1_final_test_used_for_rule_selection": False,
    }

    ROI_RULE_HASH = sha256_text(
        json.dumps(rule_payload, sort_keys=True, default=str)
    )
    ROI_RULE_FILE = (
        DIR["configs"]
        / f"roi_rule_frozen_{ROI_RULE_HASH[:10]}.json"
    )

    if ROI_RULE_FILE.is_file():
        old_rule = json.loads(ROI_RULE_FILE.read_text(encoding="utf-8"))
        if old_rule != rule_payload:
            stage_result(
                "STAGE_15_ROI_RULE_FREEZE",
                "FAIL",
                "Frozen ROI rule file exists but payload differs from current rule."
            )
    else:
        ROI_RULE_FILE.write_text(
            json.dumps(rule_payload, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )

    ROI_APPROVAL_FILE = (
        DIR["qc_reports"]
        / f"20_roi_rule_manual_approval_{ROI_RULE_HASH[:10]}.json"
    )
    approval_payload = {
        "roi_rule_hash": ROI_RULE_HASH,
        "approved": True,
        "approved_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "pilot_policy_hash": PILOT_POLICY_HASH,
        "known_warnings_acknowledged": bool(CFG.roi_rule_accept_known_warnings),
        "quality_warnings": quality_warnings,
        "complete_sheet_dir": str(PILOT_ALL_SHEET_DIR),
        "complete_sheet_count": int(len(sheet_files)),
    }
    ROI_APPROVAL_FILE.write_text(
        json.dumps(approval_payload, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    register_artifact("STAGE_15", ROI_RULE_FILE, "frozen_roi_rule")
    register_artifact(
        "STAGE_15",
        ROI_APPROVAL_FILE,
        "roi_rule_manual_approval_receipt"
    )

    ROI_RULE_FROZEN = True

    stage_result(
        "STAGE_15_ROI_RULE_FREEZE",
        "PASS",
        (
            "ROI proposal/QC rule frozen after structural audit, complete visual "
            "review attestation, and explicit acknowledgement of any remaining warnings."
        ),
        {
            **prefreeze_report,
            "roi_rule_hash": ROI_RULE_HASH,
        }
    )

print("ROI_RULE_FROZEN:", ROI_RULE_FROZEN)


CELL 20 — R3 PRE-FREEZE AUDIT
Pilot cases      : 144
ROI slots        : 1296
ACCEPT           : 1036
MISSING          : 74
NEEDS_REVIEW     : 186
REJECT           : 0
Missing rate     : 5.71%
Non-ACCEPT rate  : 20.06%
Worst age-bin    : 2 26.39%
Worst ROI        : MC1 16.67%
Hard blockers    : NONE
Quality warnings : ['young/age-domain instability: age 2=26.39%', 'pilot non-ACCEPT rate is 20.06% (warning threshold 15.00%)', 'weak strict-VAL detector class(es): MC1=0.434']
Complete sheets  : D:\workspace\Bone_age\BoneAge_LOCAL4050_V1\qc\needs_review\pilot_contact_sheets_ALL_abb788a420
[STAGE_15_ROI_RULE_FREEZE] BLOCKED: Inspect ALL 65 CURRENT-HASH contact sheets in D:\workspace\Bone_age\BoneAge_LOCAL4050_V1\qc\needs_review\pilot_contact_sheets_ALL_abb788a420. Confirm each ROI identity and especially L/R/text/border/marker false accepts. Then set roi_rule_manual_approval=True.
ROI_RULE_FROZEN: False


In [21]:
# Cell 21 — Apply ROOTCAUSE-R2 frozen rule to ALL V1_DEV + V1_VALIDATION; build review queue
#
# Default split sizes:
# V1_DEV=11,548 + V1_VALIDATION=260 => 11,808 cases.
# At 9 ROI/case, the full DEV+VAL inventory contains 106,272 ROI audit slots.
#
# This is the cell that scales from the 144-case pilot to the full development cohort.

if not globals().get("ROI_RULE_FROZEN", False) or "ROI_RULE_HASH" not in globals():
    stage_result(
        "STAGE_16_DEV_ROI_INVENTORY",
        "FAIL",
        (
            "ROI rule is not frozen. Cell 20 must report PASS; a stale ROI_RULE_HASH "
            "in memory is not sufficient."
        )
    )

DEVVAL_CACHE = (
    DIR["qc_reports"]
    / f"21_devval_roi_inventory_{ROI_RULE_HASH[:10]}.jsonl"
)

devval = pd.concat(
    [
        get_v1_role("V1_DEV").assign(v1_role="V1_DEV"),
        get_v1_role("V1_VALIDATION").assign(v1_role="V1_VALIDATION"),
    ],
    ignore_index=True
)

expected_devval_cases = len(V1_DEV_IDS) + len(V1_VAL_IDS)
if len(devval) != expected_devval_cases:
    stage_result(
        "STAGE_16_DEV_ROI_INVENTORY",
        "FAIL",
        f"DEV+VAL rows={len(devval)} but expected={expected_devval_cases}."
    )
if devval["patient_id"].astype(str).duplicated().any():
    stage_result(
        "STAGE_16_DEV_ROI_INVENTORY",
        "FAIL",
        "Duplicate patient IDs detected in combined DEV/Validation inventory."
    )

cache = load_jsonl_by_patient(DEVVAL_CACHE)

for _, r in tqdm(
    devval.iterrows(),
    total=len(devval),
    desc="V1 DEV+VAL frozen ROI rule"
):
    pid = str(r["patient_id"])
    image_path = Path(r["fullhand_image_path"])
    rec = cache.get(pid)

    cache_ok = (
        rec is not None
        and rec.get("roi_rule_hash") == ROI_RULE_HASH
        and image_record_matches(rec, image_path)
    )
    if cache_ok:
        continue

    rec = propose_case(pid, image_path)
    rec["roi_rule_hash"] = ROI_RULE_HASH
    rec["v1_role"] = str(r["v1_role"])
    append_jsonl(DEVVAL_CACHE, rec)
    cache[pid] = rec

queue_rows = []
case_rows = []
inventory_rows = []

for _, r in devval.iterrows():
    pid = str(r["patient_id"])
    rec = cache[pid]
    role = str(r["v1_role"])

    case_rows.append({
        "patient_id": pid,
        "v1_role": role,
        "bone_age_year_bin": int(r["bone_age_year_bin"]),
        "male": int(r["male"]),
        "auto_accept_9of9": bool(rec["auto_accept_9of9"]),
        "missing_n": len(rec["missing_roi"]),
        "needs_review_n": len(rec["needs_review_roi"]),
        "reader_version": rec.get("reader_version"),
        "detector_sha256": rec.get("detector_sha256"),
        "roi_rule_hash": ROI_RULE_HASH,
    })

    for roi in ROI_ORDER:
        d = rec["detections"].get(roi)
        if d is None:
            status = "MISSING"
            reason = "NO_PROPOSAL"
            conf = np.nan
            geometry_z_value = np.nan
            hand_overlap_value = np.nan
        else:
            status = str(d["qc"]["status"])
            reason = str(d["qc"]["reason"])
            conf = float(d["conf"])
            geometry_z_value = float(d["qc"]["geometry_z"])
            hand_overlap_value = float(d["qc"]["hand_overlap"])

        inventory_rows.append({
            "patient_id": pid,
            "v1_role": role,
            "bone_age_year_bin": int(r["bone_age_year_bin"]),
            "male": int(r["male"]),
            "roi": roi,
            "status": status,
            "reason": reason,
            "conf": conf,
            "geometry_z": geometry_z_value,
            "hand_overlap": hand_overlap_value,
            "fullhand_image_path": r["fullhand_image_path"],
        })

        if status != "ACCEPT":
            queue_rows.append({
                "patient_id": pid,
                "v1_role": role,
                "bone_age_year_bin": int(r["bone_age_year_bin"]),
                "male": int(r["male"]),
                "roi": roi,
                "queue_kind": "NON_ACCEPT",
                "original_status": status,
                "reason": reason,
                "conf": conf,
                "geometry_z": geometry_z_value,
                "hand_overlap": hand_overlap_value,
                "fullhand_image_path": r["fullhand_image_path"],
            })

DEVVAL_CASES = pd.DataFrame(case_rows)
DEVVAL_ROI_INVENTORY = pd.DataFrame(inventory_rows)

expected_roi_slots = len(devval) * len(ROI_ORDER)
if len(DEVVAL_ROI_INVENTORY) != expected_roi_slots:
    stage_result(
        "STAGE_16_DEV_ROI_INVENTORY",
        "FAIL",
        f"DEV+VAL ROI rows={len(DEVVAL_ROI_INVENTORY)} but expected={expected_roi_slots}."
    )
if DEVVAL_ROI_INVENTORY.duplicated(["patient_id", "roi"]).any():
    stage_result(
        "STAGE_16_DEV_ROI_INVENTORY",
        "FAIL",
        "Duplicate patient_id/ROI rows detected in full DEV+VAL inventory."
    )

# Deterministic accepted-ROI audit across ROI x age-bin x sex.
# These are not errors; they are sampled auto-ACCEPTs that a human must verify.
accepted = DEVVAL_ROI_INVENTORY[
    DEVVAL_ROI_INVENTORY["status"].eq("ACCEPT")
].copy()

accepted["_audit_rank"] = [
    stable_rank(
        f"{pid}|{roi}|{age}|{male}",
        "DEVVAL_ACCEPT_AUDIT"
    )
    for pid, roi, age, male in zip(
        accepted["patient_id"].astype(str),
        accepted["roi"].astype(str),
        accepted["bone_age_year_bin"].astype(int),
        accepted["male"].astype(int),
    )
]

accepted_audit = (
    accepted.sort_values("_audit_rank")
    .groupby(["roi", "bone_age_year_bin", "male"], as_index=False)
    .head(CFG.devval_accept_audit_per_stratum)
    .copy()
)

for _, r in accepted_audit.iterrows():
    queue_rows.append({
        "patient_id": str(r["patient_id"]),
        "v1_role": str(r["v1_role"]),
        "bone_age_year_bin": int(r["bone_age_year_bin"]),
        "male": int(r["male"]),
        "roi": str(r["roi"]),
        "queue_kind": "ACCEPT_AUDIT",
        "original_status": "ACCEPT",
        "reason": "STRATIFIED_FALSE_ACCEPT_AUDIT",
        "conf": float(r["conf"]),
        "geometry_z": float(r["geometry_z"]),
        "hand_overlap": float(r["hand_overlap"]),
        "fullhand_image_path": r["fullhand_image_path"],
    })

REVIEW_QUEUE = (
    pd.DataFrame(queue_rows)
    .drop_duplicates(["patient_id", "roi"], keep="first")
    .reset_index(drop=True)
)

DEVVAL_CASE_FILE = (
    DIR["qc_reports"]
    / f"21_devval_case_completeness_{ROI_RULE_HASH[:10]}.csv"
)
DEVVAL_ROI_FILE = (
    DIR["qc_reports"]
    / f"21_devval_roi_inventory_{ROI_RULE_HASH[:10]}.csv"
)
REVIEW_QUEUE_FILE = (
    DIR["qc_review"]
    / f"21_human_review_queue_{ROI_RULE_HASH[:10]}.csv"
)

DEVVAL_CASES.to_csv(DEVVAL_CASE_FILE, index=False, encoding="utf-8-sig")
DEVVAL_ROI_INVENTORY.to_csv(DEVVAL_ROI_FILE, index=False, encoding="utf-8-sig")
REVIEW_QUEUE.to_csv(REVIEW_QUEUE_FILE, index=False, encoding="utf-8-sig")

full_status_counts = (
    DEVVAL_ROI_INVENTORY["status"]
    .value_counts()
    .reindex(["ACCEPT", "MISSING", "NEEDS_REVIEW", "REJECT"], fill_value=0)
)
nonaccept_roi_n = int((DEVVAL_ROI_INVENTORY["status"] != "ACCEPT").sum())
accepted_audit_n = int((REVIEW_QUEUE["queue_kind"] == "ACCEPT_AUDIT").sum())

print("=" * 90)
print("FULL DEV+VAL ROI INVENTORY")
print("=" * 90)
print("DEV+VAL cases          :", len(DEVVAL_CASES))
print("ROI classes/case       :", len(ROI_ORDER))
print("Expected ROI slots     :", expected_roi_slots)
print("Actual ROI rows        :", len(DEVVAL_ROI_INVENTORY))
print("Auto accepted 9/9 cases:", int(DEVVAL_CASES["auto_accept_9of9"].sum()))
print("Non-ACCEPT ROI         :", nonaccept_roi_n)
print("Accepted audit ROI     :", accepted_audit_n)
print("Total human queue      :", len(REVIEW_QUEUE))
print("Affected patients      :", REVIEW_QUEUE["patient_id"].nunique() if len(REVIEW_QUEUE) else 0)
print(full_status_counts)

if len(REVIEW_QUEUE):
    display(
        REVIEW_QUEUE.groupby(["queue_kind", "roi"])
        .size().unstack(fill_value=0)
        .reindex(columns=ROI_ORDER, fill_value=0)
    )

register_artifact("STAGE_16", DEVVAL_CASE_FILE, "devval_case_completeness")
register_artifact("STAGE_16", DEVVAL_ROI_FILE, "devval_roi_inventory")
register_artifact("STAGE_16", REVIEW_QUEUE_FILE, "human_review_queue")

if nonaccept_roi_n > 0 or accepted_audit_n > 0:
    stage_result(
        "STAGE_16_DEV_ROI_INVENTORY",
        "WARNING",
        (
            f"Full DEV+VAL inventory completed: {len(devval)} cases / {expected_roi_slots} ROI slots. "
            f"{nonaccept_roi_n} non-ACCEPT ROI require resolution and "
            f"{accepted_audit_n} auto-ACCEPT ROI require stratified false-accept audit. "
            "No fake ROI will be generated."
        ),
        {
            "devval_cases": int(len(devval)),
            "roi_slots": int(expected_roi_slots),
            "status_counts": {k: int(v) for k, v in full_status_counts.items()},
            "nonaccept_roi": int(nonaccept_roi_n),
            "accepted_audit_roi": int(accepted_audit_n),
            "review_queue": int(len(REVIEW_QUEUE)),
        }
    )
else:
    stage_result(
        "STAGE_16_DEV_ROI_INVENTORY",
        "PASS",
        "All DEV/Validation ROI accepted and no accepted-audit samples were requested."
    )


[STAGE_16_DEV_ROI_INVENTORY] FAIL: ROI rule is not frozen. Cell 20 must report PASS; a stale ROI_RULE_HASH in memory is not sufficient.


RuntimeError: STAGE_16_DEV_ROI_INVENTORY FAIL — STOP PIPELINE: ROI rule is not frozen. Cell 20 must report PASS; a stale ROI_RULE_HASH in memory is not sufficient.

In [ ]:
# Cell 22 — Resumable LOCAL Windows human ROI review/correction
#
# Queue kinds:
# - NON_ACCEPT   : detector/QC already flagged the ROI
# - ACCEPT_AUDIT : system auto-accepted it, but we sample it to detect false ACCEPT
#
# Decisions:
# A = ACCEPT current detector ROI as anatomically correct
# C = CORRECT by manually drawing a replacement ROI
# R = REJECT the ROI (do not use it as a valid training ROI)
# S = SKIP for now (remains pending)
#
# Every decision is saved immediately and tied to ROI_RULE_HASH.

MANUAL_LOG = (
    DIR["manual"]
    / f"manual_review_log_{ROI_RULE_HASH[:10]}.csv"
)

MANUAL_COLUMNS = [
    "patient_id", "v1_role", "bone_age_year_bin", "male",
    "roi", "queue_kind", "original_status", "status",
    "x1", "y1", "x2", "y2",
    "source_image", "override_path",
    "roi_rule_hash", "detector_sha256",
    "proposal_reason", "time",
]

def load_manual_log() -> pd.DataFrame:
    if MANUAL_LOG.is_file():
        df = pd.read_csv(MANUAL_LOG, dtype={"patient_id": str})
        for col in MANUAL_COLUMNS:
            if col not in df.columns:
                df[col] = np.nan
        return df[MANUAL_COLUMNS].copy()
    return pd.DataFrame(columns=MANUAL_COLUMNS)

def safe_imwrite(path: Path, image: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    ext = path.suffix if path.suffix else ".png"
    ok, buf = cv2.imencode(ext, image)
    if not ok:
        raise RuntimeError(f"cv2.imencode failed: {path}")
    buf.tofile(str(path))

def latest_manual_decisions(df: pd.DataFrame) -> pd.DataFrame:
    if len(df) == 0:
        return df.copy()
    work = df.copy()
    if "time" in work.columns:
        work = work.sort_values("time", kind="stable")
    return (
        work.drop_duplicates(["patient_id", "roi"], keep="last")
        .reset_index(drop=True)
    )

if len(REVIEW_QUEUE) == 0:
    print("No DEV/VAL manual review required.")

elif not IS_WINDOWS:
    print(
        "Manual review/correction is LOCAL-Windows-only because correction uses cv2.selectROI.\n"
        "Run Cells 1-21 on the Windows project, then run this cell."
    )

else:
    log_df = load_manual_log()
    latest = latest_manual_decisions(log_df)

    # Never silently reuse decisions from another frozen rule.
    if len(latest):
        wrong_rule = latest[
            latest["roi_rule_hash"].astype(str).ne(str(ROI_RULE_HASH))
        ]
        if len(wrong_rule):
            raise RuntimeError(
                f"Manual log contains {len(wrong_rule)} decisions from another ROI rule."
            )

    resolved_status = {"ACCEPT", "CORRECTED", "REJECT"}
    resolved = set(
        zip(
            latest.loc[latest["status"].isin(resolved_status), "patient_id"].astype(str),
            latest.loc[latest["status"].isin(resolved_status), "roi"].astype(str),
        )
    )

    for i, r in REVIEW_QUEUE.iterrows():
        pid = str(r["patient_id"])
        roi = str(r["roi"])
        key = (pid, roi)

        if key in resolved:
            continue

        img_path = Path(r["fullhand_image_path"])
        gray = read_xray_u8(img_path)

        rec = cache.get(pid)
        if rec is None:
            raise RuntimeError(
                f"Missing ROI cache record during manual review: {pid}"
            )
        d = rec["detections"].get(roi)

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].imshow(gray, cmap="gray")
        axes[0].set_title(f"{pid} | {roi} | source")
        axes[0].axis("off")

        if d is not None:
            x1, y1, x2, y2 = map(float, d["xyxy"])
            rect_patch = plt.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                fill=False, linewidth=2
            )
            axes[0].add_patch(rect_patch)

            crop, crop_label = crop_for_display(rec, roi, pad=0.10)
            axes[1].imshow(crop, cmap="gray")
            axes[1].set_title(f"current proposal\n{crop_label}")
            axes[1].axis("off")
        else:
            axes[1].text(0.5, 0.5, "NO PROPOSAL", ha="center", va="center")
            axes[1].axis("off")

        fig.tight_layout()
        plt.show()
        plt.close(fig)

        proposal_reason = str(r.get("reason", ""))
        queue_kind = str(r.get("queue_kind", "NON_ACCEPT"))

        print(
            f"[{i+1}/{len(REVIEW_QUEUE)}] patient={pid} role={r['v1_role']} "
            f"ROI={roi} kind={queue_kind} original={r.get('original_status', '')} "
            f"reason={proposal_reason}"
        )

        decision = input(
            "Decision [A=accept / C=correct / R=reject / S=skip]: "
        ).strip().upper()

        if decision in {"", "S"}:
            print("Skipped; remains pending.")
            continue

        if decision == "A":
            if d is None:
                print(
                    "Cannot ACCEPT because there is no detector proposal. "
                    "Use C to draw the correct ROI or R to reject."
                )
                continue
            x1, y1, x2, y2 = map(float, d["xyxy"])
            status = "ACCEPT"
            override_path = ""

        elif decision == "R":
            if d is None:
                x1 = y1 = x2 = y2 = np.nan
            else:
                x1, y1, x2, y2 = map(float, d["xyxy"])
            status = "REJECT"
            override_path = ""

        elif decision == "C":
            display_img = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
            rect = cv2.selectROI(
                f"{pid} | {roi} | ENTER=save, ESC/0-size=cancel",
                display_img,
                showCrosshair=True,
                fromCenter=False
            )
            cv2.destroyAllWindows()

            x, y, w, h = map(int, rect)
            if w <= 0 or h <= 0:
                print("Correction cancelled; remains pending.")
                continue

            crop = gray[y:y+h, x:x+w]
            if crop.size == 0:
                raise RuntimeError(f"Empty manual crop: {pid} {roi}")

            out = (
                DIR["manual"]
                / ROI_RULE_HASH[:10]
                / pid
                / f"{roi}.png"
            )
            safe_imwrite(out, crop)

            x1, y1, x2, y2 = x, y, x+w, y+h
            status = "CORRECTED"
            override_path = str(out.resolve())
            print("Saved corrected ROI:", out)

        else:
            print(f"Unknown decision '{decision}'. Use A, C, R or S.")
            continue

        row = pd.DataFrame([{
            "patient_id": pid,
            "v1_role": r["v1_role"],
            "bone_age_year_bin": int(r["bone_age_year_bin"]),
            "male": int(r["male"]),
            "roi": roi,
            "queue_kind": queue_kind,
            "original_status": r.get("original_status", ""),
            "status": status,
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "source_image": str(img_path.resolve()),
            "override_path": override_path,
            "roi_rule_hash": ROI_RULE_HASH,
            "detector_sha256": FROZEN_HASH,
            "proposal_reason": proposal_reason,
            "time": pd.Timestamp.now(tz="UTC").isoformat(),
        }])

        log_df = pd.concat([log_df, row], ignore_index=True)
        log_df.to_csv(MANUAL_LOG, index=False, encoding="utf-8-sig")
        resolved.add(key)
        print(
            f"Saved decision: {status} | progress={len(resolved)}/{len(REVIEW_QUEUE)}"
        )

    print("Manual review log:", MANUAL_LOG)


In [ ]:
# Cell 23 — DEV/VAL anatomy gate + false-ACCEPT red-team check

manual_log = load_manual_log()
latest_manual = latest_manual_decisions(manual_log)

if len(latest_manual):
    bad_rule_rows = latest_manual[
        latest_manual["roi_rule_hash"].astype(str).ne(str(ROI_RULE_HASH))
    ]
    if len(bad_rule_rows):
        stage_result(
            "STAGE_17_DEVVAL_ANATOMY_GATE",
            "FAIL",
            f"{len(bad_rule_rows)} manual-review rows belong to another ROI rule hash."
        )

queue_lookup = {
    (str(r["patient_id"]), str(r["roi"])): {
        "queue_kind": str(r["queue_kind"]),
        "original_status": str(r["original_status"]),
    }
    for _, r in REVIEW_QUEUE.iterrows()
}
queue_keys = set(queue_lookup)

decision_map = {
    (str(r["patient_id"]), str(r["roi"])): str(r["status"])
    for _, r in latest_manual.iterrows()
    if (str(r["patient_id"]), str(r["roi"])) in queue_keys
}

resolved_statuses = {"ACCEPT", "CORRECTED", "REJECT"}
pending = sorted(
    key for key in queue_keys
    if decision_map.get(key) not in resolved_statuses
)

if pending:
    stage_result(
        "STAGE_17_DEVVAL_ANATOMY_GATE",
        "FAIL",
        (
            f"{len(pending)} DEV/Validation ROI remain unresolved. "
            "Do not proceed to final bone-age training."
        ),
        {"pending_examples": pending[:20]}
    )

# Critical red-team rule:
# if an item that the automatic rule ACCEPTED is later corrected/rejected by a human,
# we have direct evidence of a false auto-accept. Do NOT simply patch that sample and continue.
false_accept_hits = []
for key, meta in queue_lookup.items():
    if meta["queue_kind"] != "ACCEPT_AUDIT":
        continue
    decision = decision_map.get(key)
    if decision in {"CORRECTED", "REJECT"}:
        false_accept_hits.append({
            "patient_id": key[0],
            "roi": key[1],
            "decision": decision,
        })

accepted_keys = {k for k, v in decision_map.items() if v == "ACCEPT"}
corrected_keys = {k for k, v in decision_map.items() if v == "CORRECTED"}
rejected_keys = {k for k, v in decision_map.items() if v == "REJECT"}

ROI_AUDIT_REPORT = {
    "roi_rule_hash": ROI_RULE_HASH,
    "detector_sha256": FROZEN_HASH,
    "reader_version": XRAY_READER_VERSION,
    "roi_qc_code_version": ROI_QC_CODE_VERSION,
    "roi_proposal_code_version": ROI_PROPOSAL_CODE_VERSION,
    "devval_cases": int(len(DEVVAL_CASES)),
    "devval_roi_slots": int(len(DEVVAL_ROI_INVENTORY)),
    "auto_accepted_cases_9of9": int(DEVVAL_CASES["auto_accept_9of9"].sum()),
    "review_roi_total": int(len(REVIEW_QUEUE)),
    "manual_accept_roi": int(len(accepted_keys)),
    "manual_corrected_roi": int(len(corrected_keys)),
    "manual_rejected_roi": int(len(rejected_keys)),
    "pending_review_roi": int(len(pending)),
    "false_auto_accept_audit_failures": false_accept_hits,
    "final_test_used": False,
}

ROI_AUDIT_FILE = (
    DIR["qc_reports"]
    / f"23_roi_audit_status_{ROI_RULE_HASH[:10]}.json"
)
ROI_AUDIT_FILE.write_text(
    json.dumps(ROI_AUDIT_REPORT, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

register_artifact("STAGE_17", ROI_AUDIT_FILE, "devval_roi_anatomy_audit")

if false_accept_hits:
    stage_result(
        "STAGE_17_DEVVAL_ANATOMY_GATE",
        "FAIL",
        (
            f"FALSE AUTO-ACCEPT detected in {len(false_accept_hits)} stratified audit ROI. "
            "This is evidence that the frozen auto-ACCEPT rule is not safe enough. "
            "Do NOT merely correct these samples and continue: return to Cells 17-20, "
            "improve QC/marker/anatomy rejection, regenerate the pilot, and freeze a new rule."
        ),
        ROI_AUDIT_REPORT
    )

stage_result(
    "STAGE_17_DEVVAL_ANATOMY_GATE",
    "PASS",
    (
        "All DEV/Validation review-queue ROI have explicit human decisions and "
        "the stratified auto-ACCEPT audit found no corrected/rejected false accepts. "
        "REJECT items remain excluded; no fake ROI is created."
    ),
    ROI_AUDIT_REPORT
)


## Final test unlock — intentionally not automatic

At this point the detector, ROI proposal rule, QC thresholds, and manual-review policy are frozen.

**Do not unlock the V1 FinalTest260 merely to inspect detector completeness.**

The intended future sequence is:

1. Build the verified V1 training/validation dataset.
2. Train candidate bone-age models.
3. Select the winner using **V1 Validation260 only**.
4. Freeze model checkpoint + preprocessing + ROI rule hashes.
5. Set `UNLOCK_V1_FINAL_TEST = True` in the next cell.
6. Run the V1 FinalTest260 once for the final internal evaluation.
7. Never tune the model/rule from those final-test results.

In [ ]:
# Cell 24 — Final-test unlock guard (default OFF)
#
# The final holdout remains inaccessible until:
# - the frozen ROI rule exists
# - the DEV/VAL anatomy gate is PASS
# - the anatomy audit report exists and has no pending/false-auto-accept failures

UNLOCK_V1_FINAL_TEST = False

if UNLOCK_V1_FINAL_TEST:
    anatomy_gate_state = get_stage_state("STAGE_17_DEVVAL_ANATOMY_GATE") or {}
    if anatomy_gate_state.get("status") != "PASS":
        stage_result(
            "STAGE_18_FINAL_TEST_UNLOCK",
            "FAIL",
            "DEV/Validation anatomy gate has not passed in this runtime."
        )

    required_final_artifacts = [ROI_RULE_FILE, ROI_AUDIT_FILE]
    missing_final_artifacts = [
        str(p) for p in required_final_artifacts if not Path(p).is_file()
    ]
    if missing_final_artifacts:
        stage_result(
            "STAGE_18_FINAL_TEST_UNLOCK",
            "FAIL",
            f"Required frozen/audit artifacts missing: {missing_final_artifacts}"
        )

    audit_payload = json.loads(Path(ROI_AUDIT_FILE).read_text(encoding="utf-8"))
    if int(audit_payload.get("pending_review_roi", -1)) != 0:
        stage_result(
            "STAGE_18_FINAL_TEST_UNLOCK",
            "FAIL",
            "ROI anatomy audit still has pending review items."
        )
    if audit_payload.get("false_auto_accept_audit_failures"):
        stage_result(
            "STAGE_18_FINAL_TEST_UNLOCK",
            "FAIL",
            "ROI anatomy audit contains false auto-accept failures."
        )

    FINAL_TEST_UNLOCKED = True
    unlock_payload = {
        "unlocked": True,
        "time": pd.Timestamp.now(tz="UTC").isoformat(),
        "v1_final_test_hash": sha256_text("\n".join(sorted(V1_FINAL_TEST_IDS))),
        "detector_sha256": FROZEN_HASH,
        "roi_rule_hash": ROI_RULE_HASH,
        "roi_rule_file_sha256": sha256_file(ROI_RULE_FILE),
        "roi_audit_file_sha256": sha256_file(ROI_AUDIT_FILE),
        "warning": "This action exposes the V1 final holdout. Do not tune after this point.",
    }
    unlock_file = (
        DIR["qc_reports"]
        / f"24_FINAL_TEST_UNLOCKED_{ROI_RULE_HASH[:10]}.json"
    )
    unlock_file.write_text(
        json.dumps(unlock_payload, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    register_artifact("STAGE_18", unlock_file, "final_test_unlock_receipt")
    stage_result(
        "STAGE_18_FINAL_TEST_UNLOCK",
        "WARNING",
        "V1 FinalTest260 is now accessible. This event is permanently logged."
    )
else:
    FINAL_TEST_UNLOCKED = False
    print("V1 FinalTest260 remains LOCKED — correct state during development.")


In [ ]:
# Cell 25 — Optional project export / backup manifest
#
# Creates a compact machine-readable inventory of all important files.
# It does NOT delete or move source data.

records = []
for root_name, root in DIR.items():
    root = Path(root)
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        # Hash important, reasonably-sized metadata/checkpoints; large images are inventoried by size/path.
        rec = {
            "group": root_name,
            "relative_path": str(p.relative_to(PROJECT_ROOT)),
            "bytes": int(p.stat().st_size),
            "sha256": None,
            "hash_error": None,
        }
        if p.suffix.lower() in {".json",".jsonl",".csv",".yaml",".yml",".txt",".pt"}:
            try:
                rec["sha256"] = sha256_file(p)
            except Exception as e:
                rec["hash_error"] = f"{type(e).__name__}: {e}"
        records.append(rec)

EXPORT_MANIFEST = DIR["outputs"] / "project_export_manifest.csv"
pd.DataFrame(records).to_csv(EXPORT_MANIFEST, index=False)
register_artifact("STAGE_EXPORT", EXPORT_MANIFEST, "project_export_manifest")

print("Export manifest:", EXPORT_MANIFEST)
print("Project root    :", PROJECT_ROOT)
print()
print("For backup, archive/copy the entire BoneAge_LOCAL4050_V1 project folder.")


# Run order

1. Run Cells 1–18 normally.  
   - Cell 15 will **resume** a partial detector run instead of treating a partial `results.csv` as completed.
   - Cell 16 verifies saved training args and frozen-checkpoint SHA256.
2. Run Cell 19. With the default pilot setting it uses **144 cases**, not the full dataset:
   - 18 bone-age bins × 8 cases = 144 cases
   - 144 × 9 ROI = **1,296 pilot ROI audit slots**
   - inspect the **complete** hash-versioned pilot contact-sheet directory, not counts alone.
3. Keep `CFG.roi_rule_manual_approval=False` until the complete pilot visual audit is acceptable.
4. After visual inspection, set `CFG.roi_rule_manual_approval=True`, rerun from Cell 3 through Cell 20, and freeze the exact rule.
5. Run Cell 21. This is the **full DEV+VAL application**:
   - V1_DEV 11,548 + V1_VALIDATION 260 = **11,808 cases**
   - 11,808 × 9 = **106,272 ROI audit slots**
   - it also samples auto-ACCEPT ROI across ROI × age-bin × sex for false-accept auditing.
6. Run Cell 22 on Windows and explicitly record:
   - `A` = anatomically correct proposal
   - `C` = manually corrected ROI
   - `R` = invalid/reject
   - `S` = leave pending
7. Run Cell 23.
   - Any unresolved ROI => FAIL.
   - Any sampled `ACCEPT_AUDIT` that must be corrected/rejected => **FAIL and return to Cells 17–20**; do not hide a false auto-accept with a one-off patch.
8. Keep Cell 24 locked until all development/model/rule selection is frozen. Unlocking FinalTest is a one-way evaluation event.

## Important limitation

The current ROI QC combines detector confidence, geometry, hand-overlap and human audit.  
It is **not yet a learned anatomical-identity classifier or dedicated L/R text detector**. Therefore a PASS means the defined QC + audit gates passed; it must not be described as proof that every auto-accepted ROI in the full dataset is anatomically correct.


## ROOT-CAUSE R2 pilot rule

Do **not** set `roi_rule_manual_approval=True` using an old `ea22...` pilot folder.

After rerunning Cells 17–19, review only:

- `pilot_contact_sheets_priority_<CURRENT_HASH>/`
- `pilot_contact_sheets_ALL_<CURRENT_HASH>/`

The old `pilot_contact_sheets_ea22d24f0f/` directory is historical and may show black
tiles for MISSING ROI; it must not be used for the R2 freeze decision.

R2 intentionally treats foreground-mask overlap and geometry as **review signals**,
not proof that a crop is non-anatomical. Only objective invalid crops may be
automatically REJECTED.


## Detector retraining policy fix

`CFG.version` is **not** part of detector-training identity anymore.

Changing ROI/QC code, confidence thresholds, marker checks, contact sheets,
rotation rescue, or Cell 17–20 must not cause YOLO to retrain.

Cell 14 searches historical detector configurations and reuses an older completed
run when both of these match exactly:

1. strict detector dataset SHA256
2. complete `train_params`

If Cell 14 prints:

`[REUSE] Existing YOLO detector training is training-equivalent.`

then Cell 15 must reuse/recover that checkpoint and must not enter `[FRESH]`.

A genuinely new YOLO run is allowed only when detector training data or detector
training hyperparameters actually changed.


## Verified detector reuse (critical)

The original validated detector is reused by exact SHA256. Cell 15 contains no Ultralytics training call. If the trusted checkpoint cannot be found, the notebook FAILS instead of starting another 100-epoch run. Accidental newer runs such as `e4f6efb8ce` or `696f24eab2` are not selected unless their checkpoint SHA is independently trusted by the old validation reports.


## R3 audit changes

- Cell 11 rejects fractional YOLO class IDs instead of silently truncating them.
- Cell 16 uses strict detector **VAL** for routine development; detector TEST is locked by default.
- Cell 18 rescues both MISSING and low-confidence ROI down to `rescue_conf_floor=0.05`.
  Low-confidence proposals remain review items; confidence is not fabricated.
- Candidate selection is detector-confidence-first and records cross-view consensus.
- Cell 20 uses `BLOCKED` for expected quality/procedure holds instead of throwing a
  misleading runtime traceback.
- Age-bin missing rates use warning/hard-block tiers because each pilot age bin contains
  only eight patients (72 ROI slots); R3 does not pretend 30.00% and 30.56% are
  scientifically different.
- Cell 21 requires `ROI_RULE_FROZEN=True`, preventing stale in-memory hashes from
  accidentally launching the 11,808-case full inventory.
- The current detector's weak MC1 strict-VAL AP and young-age pilot instability are
  surfaced as explicit quality warnings. Do not suppress them merely to make a gate pass.
